# Experimental and Testing of SQLAlchemy Code for uploading and relating DeviantArt data 

In [ ]:
#Create a new virutal environment because Alchemy could have conflicting environments. 
!python -m venv alchemy_db_env  # Create a new virtual environment
!source alchemy_db_env/bin/activate  # Activate the environment

In [ ]:
#install necessary libraries
!pip install sqlalchemy==1.4.6 pandas==2.0.3

In [ ]:
#Uninstall if previous installation was corrupted
#!pip uninstall sqlalchemy -y

#### Pre-processing watching data in the CSV and saving the processed data to be uploaded in the  

In [ ]:
# Process the watching data

def process_watching_data(csv_path):
    """
    Reads scraping CSV, extracts watching relationships, and returns a list of dictionaries.

    Args:
        csv_path: Path to the scraping CSV file.

    Returns:
        A list of dictionaries, where each dictionary represents a single
        watching relationship: [{'username': 'ArtistA', 'watching_name': 'ArtistB'}, ...]
    """
    processed_watchings = []
    try:
        # Adjust chunksize as needed
        for chunk in pd.read_csv(csv_path, chunksize=1000, on_bad_lines='skip'):
            for index, row in chunk.iterrows():
                artist_name_who_is_watching = row['username'] # Column with the watching artist's username

                # Check if 'Watching' column is not null or empty
                if pd.notnull(row['Watching']) and str(row['Watching']).strip():
                    watching_string = str(row['Watching']).strip()

                    # Use regex to remove "Watching XXX Deviants" at the beginning
                    # This regex looks for "Watching", followed by one or more spaces,
                    # followed by one or more digits (\d+), followed by one or more spaces,
                    # followed by "Deviants" (case-insensitive), followed by optional spaces.
                    # Use regex to remove "Watching XXX Deviants" at the beginning
                    watching_string_cleaned = re.sub(r'^Watching\s+\d+\s+Deviants\s*', '', watching_string, flags=re.IGNORECASE)

                    watched_usernames = watching_string_cleaned.split()

                    for watching_name in watched_usernames: # This is where the error occurs
                        # Check if the watching_name is not empty after splitting
                        if watching_name.strip():
                            processed_watchings.append({
                                'username': artist_name_who_is_watching,
                                'watching_name': watching_name.strip()
                            })
                # Handle the case where 'Watching' column is null or empty after stripping
                elif pd.isnull(row['Watching']) or not str(row['Watching']).strip():
                    # You might want to log this or take other action if needed
                    print(f"Info: 'Watching' column is empty or null for artist {artist_name_who_is_watching}. Skipping.")


    except FileNotFoundError:
        print(f"Error: CSV file not found at {csv_path}")
    except Exception as e:
        print(f"An error occurred during watchings data processing: {e}")

    return processed_watchings

In [ ]:
watching = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_snwballScraped_fin01.csv.gz"
processed_watching_list = process_watching_data(watching)

#     # You can print a sample of processed_watching_list here to verify
#print("Sample of processed data:", processed_watching_list[:10])

In [ ]:
#append the watching data to a dataframe
df_watchng = pd.DataFrame(columns=["Deviant_who_isWatching", "Deviant_being_Watched"])
chunk_size = 1000
for i in range(0, len(processed_watching_list), chunk_size):
        chunk = processed_watching_list[i : i + chunk_size]
        for row_data in chunk:
            # Append data as a new row to the DataFrame
            # Using pd.concat is generally more efficient than append in a loop
            new_row = pd.DataFrame([{
                "Deviant_who_isWatching": row_data['username'],
                "Deviant_being_Watched": row_data['watching_name']
            }])
            df_watching = pd.concat([df_watchng, new_row], ignore_index=True)

In [ ]:
df_watchng.to_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_WatchingSnwball_fin15-07-2025.csv.gz")

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import ast
import logging
from datetime import datetime
from dateutil import parser
import pytz
import sqlalchemy

# --- Configure Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Database Setup (unchanged) ---
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, BigInteger, Boolean, func, MetaData, Table, UniqueConstraint, inspect, text, select
from sqlalchemy.dialects import sqlite

DATABASE_URL = 'sqlite:////mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_main05.db'
engine = create_engine(DATABASE_URL)
Base = declarative_base()
Session = sessionmaker(bind=engine)

# Your Artist, Watcher, Friend models (unchanged)
class Artist(Base):
    __tablename__ = 'artists'
    id = Column(Integer, primary_key=True)
    artist_name = Column(String, unique=True, index=True)
    profile_url = Column(String)
    country = Column(String)
    level = Column(String)
    registration_date = Column(Integer)
    no_of_deviations = Column(Integer)
    no_of_favourites = Column(Integer)
    no_of_user_comments = Column(Integer)
    no_of_pageviews = Column(Integer) # Corrected
    no_of_profile_comments = Column(Integer) # Corrected
    is_artist = Column(Boolean) # Corrected
    gender = Column(String)
    speciality = Column(String)
    no_of_images = Column(Integer)
    no_of_AI_images = Column(Integer)
    ai_adopter = Column(Boolean) # Corrected
    ai_adoption_first_time = Column(Integer)
    watchers = relationship("Watcher", back_populates="artist")
    friends = relationship("Friend", back_populates="artist")
    __table_args__ = (UniqueConstraint('artist_name', name='_artist_name_uc'),)

class Watcher(Base):
    __tablename__ = 'watchers_v1'
    id = Column(Integer, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'), nullable=False) # Corrected
    watcher_name = Column(String, nullable=False)
    watcher_type = Column(String)
    is_watching = Column(Boolean) # Corrected
    last_visit = Column(BigInteger)
    activity = Column(Boolean) # Corrected
    collections = Column(Boolean) # Corrected
    critiques = Column(Boolean) # Corrected
    deviations = Column(Boolean) # Corrected
    forum_threads = Column(Boolean) # Corrected
    friend = Column(Boolean) # Corrected
    journals = Column(Boolean) # Corrected
    scraps = Column(Boolean) # Corrected
    artist_name = Column(String)
    artist = relationship("Artist", back_populates="watchers_v1")
    __table_args__ = (UniqueConstraint('artist_id', 'watcher_name', name='_artist_watcher_uc'),)

class Friend(Base):
    __tablename__ = 'friends_v1'
    id = Column(Integer, primary_key=True) # Corrected
    artist_id = Column(Integer, ForeignKey('artists.id'), nullable=False) # Corrected
    friend_name = Column(String, nullable=False)
    friend_type = Column(String)
    is_watching = Column(Boolean) # Corrected
    last_visit = Column(BigInteger)
    friends = Column(Boolean) # Corrected
    deviations = Column(Boolean) # Corrected
    journals = Column(Boolean) # Corrected
    forum_threads = Column(Boolean) # Corrected
    critiques = Column(Boolean) # Corrected
    scraps = Column(Boolean) # Corrected
    activity = Column(Boolean) # Corrected
    collections = Column(Boolean) # Corrected
    artist_name = Column(String)
    watches_you = Column(Boolean) # Corrected
    artist = relationship("Artist", back_populates="friends_v1")
    __table_args__ = (UniqueConstraint('artist_id', 'friend_name', name='_artist_friend_uc'),)


Artist.watchers_v1 = relationship("Watcher", back_populates="artist")
Artist.friends_v1 = relationship("Friend", back_populates="artist")
#Create tables if they dont exist
Base.metadata.create_all(engine)

# --- Converters for pandas.read_csv (unchanged) ---
def bool_converter(x):
    if pd.isna(x) or not isinstance(x, str):
        return None
    return str(x).lower() == 'true'

def iso_date_to_ms(x):
    if pd.isna(x):
        return None
    if not isinstance(x, str) or not x.strip():
        return None
    try:
        dt_obj = parser.parse(x)
        if dt_obj.tzinfo is None:
             dt_obj = pytz.utc.localize(dt_obj)
        else:
            dt_obj = dt_obj.astimezone(pytz.utc)
        return int(dt_obj.timestamp() * 1000)
    except Exception as e:
        logger.warning(f"Could not convert date string '{x}' to timestamp. Error: {e}. Setting to None.")
        return None

 #   Column         Non-Null Count   Dtype Data columns (total 14 columns):
 # 0   Watchers name  888326 non-null  object
 # 1   user_icon      888326 non-null  object
  #2   type           888326 non-null  object
 # 3   is_watching    888326 non-null  bool
 # 4   last_visit     759781 non-null  object
 # 5   activity       888326 non-null  bool
 # 6   collections    888326 non-null  bool
 # 7   critiques      888326 non-null  bool
 # 8   deviations     888326 non-null  bool
 # 9   forum_threads  888326 non-null  bool
 # 10  friend         888326 non-null  bool
 # 11  journals       888326 non-null  bool
 # 12  scraps         888326 non-null  bool
 # 13  Deviant        888326 non-null  object

 # 0   Friends name   942998 non-null  object
 # 1   user_icon      942998 non-null  object
 # 2   type           942998 non-null  object
 # 3   is_watching    942998 non-null  bool
 # 4   watches_you    942998 non-null  bool
 # 5   last_visit     582454 non-null  object
 # 6   friends        942998 non-null  bool
 # 7   deviations     942998 non-null  bool
 # 8   journals       942998 non-null  bool
 # 9   forum_threads  942998 non-null  bool
 # 10  critiques      942998 non-null  bool
 # 11  scraps         942998 non-null  bool
 # 12  activity       942998 non-null  bool
 # 13  collections   942998 non-null  bool
 # 14  Deviant        942998 non-null  object

# --- Shared Dtype for CSV Reading ---
COMMON_COL_DTYPES = {
    'Watchers name': str, 'user_icon': str, 'type': str, 'Deviant': str,
    'Friends name': str,
}

COMMON_CONVERTERS = {
    'is_watching': bool_converter, 'activity': bool_converter, 'collections': bool_converter,
    'critiques': bool_converter, 'deviations': bool_converter, 'forum_threads': bool_converter,
    'friend': bool_converter, 'journals': bool_converter, 'scraps': bool_converter,
    'watches_you': bool_converter, 'last_visit': iso_date_to_ms
}

# --- Helper to get existing artist IDs once ---
def get_artist_name_to_id_map(session):
    logger.info("Fetching all existing artist names and IDs...")
    artist_map = {}
    for artist_id, artist_name in session.query(Artist.id, Artist.artist_name).all():
        artist_map[str(artist_name)] = artist_id
    logger.info(f"Loaded {len(artist_map)} existing artists.")
    return artist_map

# --- Optimized Incremental Load Function ---
def load_data_incrementally(csv_path, table_class, artist_name_col_csv, related_name_col_csv, chunksize=100000, artist_map_global=None, converters=None, temp_suffix='_temp'):
    if artist_map_global is None:
        raise ValueError("artist_map_global must be provided.")

    if not os.path.exists(csv_path):
        logger.error(f"CSV file not found: {csv_path}")
        return

    total_rows_processed = 0
    total_skipped_artist = 0
    total_skipped_duplicates = 0
    temp_table_name = table_class.__tablename__ + temp_suffix
    main_table = table_class.__table__

    # Define a mapping from CSV column names to database column names for the current table
    csv_to_db_col_map = {}
    if table_class == Watcher:
        csv_to_db_col_map = {
            'Watchers name': 'watcher_name',
            # 'user_icon': 'user_icon', # Excluded as per user request
            'type': 'watcher_type',
            'is_watching': 'is_watching',
            'last_visit': 'last_visit',
            'activity': 'activity',
            'collections': 'collections',
            'critiques': 'critiques',
            'deviations': 'deviations',
            'forum_threads': 'forum_threads',
            'friend': 'friend',
            'journals': 'journals',
            'scraps': 'scraps',
            'Deviant': 'artist_name', # Mapping Deviant from CSV to artist_name in DB
            # 'artist_id' is handled separately
        }
    elif table_class == Friend:
        csv_to_db_col_map = {
            'Friends name': 'friend_name',
            # 'user_icon': 'user_icon', # Excluded as per user request
            'type': 'friend_type',
            'is_watching': 'is_watching',
            'watches_you': 'watches_you',
            'last_visit': 'last_visit',
            'friends': 'friends',
            'deviations': 'deviations',
            'journals': 'journals',
            'forum_threads': 'forum_threads',
            'critiques': 'critiques',
            'scraps': 'scraps',
            'activity': 'activity',
            'collections': 'collections',
            'Deviant': 'artist_name', # Mapping Deviant from CSV to artist_name in DB
            # 'artist_id' is handled separately
        }
    else:
        logger.error(f"Unknown table class: {table_class.__name__}. Cannot define column mapping.")
        return


    logger.info(f"Starting incremental load for {table_class.__tablename__} from {csv_path}")

    try:
        for i, chunk in enumerate(pd.read_csv(
            csv_path,
            chunksize=chunksize,
            on_bad_lines='skip',
            dtype=COMMON_COL_DTYPES,
            converters=converters # Use the provided converters
        )):
            logger.info(f"Processing chunk {i + 1} for {table_class.__tablename__} (rows read: {len(chunk)})")

            # --- 1. Map artist_name to artist_id ---
            if artist_name_col_csv not in chunk.columns:
                logger.error(f"  Missing required column '{artist_name_col_csv}' in CSV chunk {i+1} for {table_class.__tablename__}. Skipping chunk.")
                continue

            chunk['artist_id'] = chunk[artist_name_col_csv].map(artist_map_global)

            unmapped_artists_df = chunk[chunk['artist_id'].isna()]
            if not unmapped_artists_df.empty:
                logger.warning(f"  Skipping {len(unmapped_artists_df)} rows in chunk {i+1} due to unmapped artist names. Examples: {unmapped_artists_df[artist_name_col_csv].unique()[:5].tolist()}")
                total_skipped_artist += len(unmapped_artists_df)


            chunk_mapped = chunk.dropna(subset=['artist_id']).copy() # Create a copy to avoid SettingWithCopyWarning
            if chunk_mapped.empty:
                logger.info(f"  Chunk {i+1} became empty after filtering unmapped artists.")
                continue

            chunk_mapped['artist_id'] = chunk_mapped['artist_id'].astype(int)
            logger.info(f"  Rows after artist mapping: {len(chunk_mapped)}")

            # --- 2. Create DataFrame for insertion, matching SQLA model attributes ---
            df_to_insert = pd.DataFrame()

            df_to_insert['artist_id'] = chunk_mapped['artist_id']

            # Use the defined mapping to select and rename columns
            for csv_col, db_col in csv_to_db_col_map.items():
                if csv_col in chunk_mapped.columns:
                    df_to_insert[db_col] = chunk_mapped[csv_col]
                else:
                    # Handle cases where a CSV column expected by the map is missing
                    # For boolean columns, default to False if not in CSV
                    if db_col in [c.name for c in main_table.columns if isinstance(c.type, Boolean)]:
                         df_to_insert[db_col] = False
                         logger.warning(f"  CSV column '{csv_col}' not found in chunk {i+1} for DB column '{db_col}', defaulting to False.")
                    else:
                         df_to_insert[db_col] = None
                         logger.warning(f"  CSV column '{csv_col}' not found in chunk {i+1} for DB column '{db_col}', defaulting to None.")


            # --- Add step to drop duplicates within the chunk DataFrame ---
            # Determine the columns for dropping duplicates based on the unique constraint
            unique_cols_main_names = []
            try:
                unique_constraint_cols = table_class.__table_args__[0].columns
                unique_cols_main_names = [col.name for col in unique_constraint_cols]
            except (AttributeError, IndexError):
                 logger.error(f"  Could not determine unique constraint columns for {table_class.__tablename__} to drop duplicates within chunk.")
                 # Continue without dropping duplicates if unique constraint cols can't be determined


            if unique_cols_main_names and all(col_name in df_to_insert.columns for col_name in unique_cols_main_names):
                initial_rows_in_df = len(df_to_insert)
                df_to_insert.drop_duplicates(subset=unique_cols_main_names, inplace=True)
                rows_dropped_in_df = initial_rows_in_df - len(df_to_insert)
                if rows_dropped_in_df > 0:
                    logger.warning(f"  Dropped {rows_dropped_in_df} duplicate rows within chunk {i+1} DataFrame based on {unique_cols_main_names}.")
                    total_skipped_duplicates += rows_dropped_in_df # Count these as skipped duplicates
            elif unique_cols_main_names:
                 logger.warning(f"  Could not find all unique constraint columns {unique_cols_main_names} in df_to_insert for dropping duplicates within chunk {i+1}.")


            # --- 3. Bulk insert to temporary table ---
            with engine.connect() as conn:
                with conn.begin():
                    try:
                        conn.execute(text(f"DROP TABLE IF EXISTS {temp_table_name}"))
                        logger.info(f"  Dropped temporary table {temp_table_name}")
                    except Exception as drop_e:
                        logger.warning(f"  Could not drop temporary table {temp_table_name}: {drop_e}")


                    sqla_dtype_map = {col.name: col.type for col in main_table.columns if col.name != 'id'}

                    try:
                        # Ensure column order matches the target table exactly before to_sql
                        # Get the list of column names from the main table (excluding 'id')
                        target_column_order = [c.name for c in main_table.columns if c.name != 'id']
                        # Reindex the DataFrame to match the target column order, filling missing columns with None
                        df_to_insert = df_to_insert.reindex(columns=target_column_order)

                        df_to_insert.to_sql(temp_table_name, conn, if_exists='replace', index=False, dtype=sqla_dtype_map)
                        rows_in_temp = len(df_to_insert)
                        logger.info(f"  Inserted {rows_in_temp} rows into temporary table {temp_table_name}")
                    except Exception as to_sql_e:
                        logger.error(f"  Error inserting into temporary table {temp_table_name}: {to_sql_e}", exc_info=True)
                        continue # Skip to next chunk if temp insert fails


                    # --- 4. Atomic insert from temp to main table with WHERE NOT EXISTS ---
                    cols_to_insert = [c.name for c in main_table.columns if c.name != main_table.primary_key.columns.keys()[0]]

                    # Important: Reflect the temporary table to get its Columns for the SELECT statement
                    # Need to ensure metadata is up-to-date for reflection
                    temp_metadata = MetaData()
                    # Use only= parameter to specify the temporary table name for reflection
                    temp_metadata.reflect(bind=conn, only=[temp_table_name])
                    temp_table = temp_metadata.tables[temp_table_name]


                    # Build select list where columns are from the temp_table.c (columns)
                    select_temp_cols = [temp_table.c[col_name] for col_name in cols_to_insert if col_name in temp_table.c]

                    # Build the WHERE NOT EXISTS clause based on the unique constraint columns
                    # The unique constraint is assumed to be the first one in __table_args__
                    try:
                        unique_cols_main_names = [col.name for col in table_class.__table_args__[0].columns]
                        # Ensure unique columns exist in both main and temporary tables before building the WHERE clause
                        if all(col_name in main_table.c and col_name in temp_table.c for col_name in unique_cols_main_names):
                            unique_cols_main = [main_table.c[col_name] for col_name in unique_cols_main_names]
                            unique_cols_temp = [temp_table.c[col_name] for col_name in unique_cols_main_names]

                            # Subquery to check for existence
                            exists_subquery = select(unique_cols_main[0]) \
                                             .where(unique_cols_main[0] == unique_cols_temp[0]) \
                                             .where(unique_cols_main[1] == unique_cols_temp[1]) \
                                             .exists()

                            # The final INSERT FROM SELECT statement
                            insert_stmt = sqlite.insert(main_table).from_select(
                                [c.name for c in select_temp_cols], # Ensure column names match for from_select
                                select(*select_temp_cols).where(~exists_subquery) # Only select rows that do NOT exist in main_table
                            )
                        else:
                            logger.error(f"  Unique constraint columns {unique_cols_main_names} not found in both main and temporary tables for {table_class.__tablename__}. Cannot build WHERE NOT EXISTS clause.")
                            # Fallback to a simple insert (will likely fail on duplicates unless tables were dropped)
                            insert_stmt = sqlite.insert(main_table).from_select(
                                [c.name for c in select_temp_cols],
                                select(*select_temp_cols)
                            )

                    except (AttributeError, IndexError):
                         logger.error(f"  Could not determine unique constraint columns for {table_class.__tablename__}. Skipping insert from temp.")
                         continue


                    logger.info(f"  Generated INSERT statement: {insert_stmt.compile(engine)}")

                    try:
                        result = conn.execute(insert_stmt)
                        rows_inserted = result.rowcount
                        logger.info(f"  Inserted {rows_inserted} new rows into {main_table.name} from temp table.")

                        rows_in_temp = len(df_to_insert)
                        rows_skipped_in_this_chunk = rows_in_temp - rows_inserted
                        # total_skipped_duplicates already includes duplicates dropped within the chunk DataFrame

                        if rows_skipped_in_this_chunk > 0:
                            logger.warning(f"  Skipped {rows_skipped_in_this_chunk} existing {table_class.__tablename__} records in chunk {i+1} during insert from temp (already in main table).")

                        total_rows_processed += rows_inserted
                    except Exception as insert_e:
                        logger.error(f"  Error inserting from temporary table to main table {main_table.name}: {insert_e}", exc_info=True)
                        # Reraise the exception to stop processing if an integrity error occurs
                        raise insert_e


    except Exception as e:
        logger.error(f"Error processing {table_class.__tablename__} chunk: {e}", exc_info=True)
        # If the error is an IntegrityError, it will be logged above and the process will stop due to the re-raise
        # For other errors, log and continue to the next file if applicable
        if not isinstance(e, sqlalchemy.exc.IntegrityError):
             logger.error(f"Continuing with the next file/step after error in {table_class.__tablename__}: {e}")


    logger.info(f"Finished loading {table_class.__tablename__}. Total processed: {total_rows_processed} rows. Total skipped unmapped artists: {total_skipped_artist} rows. Total skipped duplicates: {total_skipped_duplicates} rows.")

# --- Main Execution ---
if __name__ == "__main__":
    # Ensure tables exist
    Base.metadata.create_all(engine)
    with Session() as session:
        artist_name_to_id = get_artist_name_to_id_map(session)
        watchers_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_wtchrsSnwball_fin1.csv.gz"
        friends_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_friendsSnwball_fin.csv.gz"

        # --- Drop and Recreate Watchers Table ---
        #logger.info("\nDropping and recreating watchers_v1 table.")
        #try:
        #    Watcher.__table__.drop(engine)
        #    logger.info("watchers_v1 table dropped.")
        #except Exception as e:
        #    logger.warning(f"Could not drop watchers_v1 table (it might not exist): {e}")
        #Base.metadata.create_all(engine, tables=[Watcher.__table__])
        #logger.info("watchers_v1 table recreated.")


        # --- Drop and Recreate Friends Table ---
        #logger.info("\nDropping and recreating friends_v1 table.")
        #try:
        #    Friend.__table__.drop(engine)
        #    logger.info("friends_v1 table dropped.")
        #except Exception as e:
        #    logger.warning(f"Could not drop friends_v1 table (it might not exist): {e}")
        #Base.metadata.create_all(engine, tables=[Friend.__table__])
        #logger.info("friends_v1 table recreated.")


        logger.info("\nStarting Friends data load.")
        try:
            load_data_incrementally(
                csv_path=friends_csv,
                table_class=Friend,
                artist_name_col_csv='Deviant',
                related_name_col_csv='Friends name',
                artist_map_global=artist_name_to_id,
                converters=COMMON_CONVERTERS
            )
        except sqlalchemy.exc.IntegrityError as e:
             logger.error(f"IntegrityError occurred during Friends data load: {e}")
             # Decide whether to stop or continue based on your needs
             # For now, let's stop to investigate
             logger.info("Stopping data load due to IntegrityError in Friends data.")
             exit() # Stop execution


        logger.info("\nStarting Watchers data load.")
        try:
            load_data_incrementally(
                csv_path=watchers_csv,
                table_class=Watcher,
                artist_name_col_csv='Deviant',
                related_name_col_csv='Watchers name',
                artist_map_global=artist_name_to_id,
                converters=COMMON_CONVERTERS
            )
        except sqlalchemy.exc.IntegrityError as e:
             logger.error(f"IntegrityError occurred during Watchers data load: {e}")
             logger.info("Stopping data load due to IntegrityError in Watchers data.")
             # No need to exit here as it's the last step, but you could add a flag if needed

    logger.info("\nDatabase loading complete.")

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import ast
import logging
from datetime import datetime
from dateutil import parser
import pytz
import threading

# --- Configure Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Database Setup (unchanged) ---
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, BigInteger, Boolean, func, MetaData, Table, UniqueConstraint, inspect, text, select
from sqlalchemy.dialects import sqlite

DATABASE_URL = 'sqlite:////mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_main05.db'
engine = create_engine(DATABASE_URL)
Base = declarative_base()
Session = sessionmaker(bind=engine)

# Your Artist, Watcher, Friend models (unchanged)
class Artist(Base):
    __tablename__ = 'artists'
    id = Column(Integer, primary_key=True)
    artist_name = Column(String, unique=True, index=True)
    profile_url = Column(String)
    country = Column(String)
    level = Column(String)
    registration_date = Column(Integer)
    no_of_deviations = Column(Integer)
    no_of_favourites = Column(Integer)
    no_of_user_comments = Column(Integer)
    no_of_pageviews = Column(Integer)
    no_of_profile_comments = Column(Integer)
    is_artist = Column(Boolean)
    gender = Column(String)
    speciality = Column(String)
    no_of_images = Column(Integer)
    no_of_AI_images = Column(Integer)
    ai_adopter = Column(Boolean)
    ai_adoption_first_time = Column(Integer)
    watchers = relationship("Watcher", back_populates="artist")
    friends = relationship("Friend", back_populates="artist")
    __table_args__ = (UniqueConstraint('artist_name', name='_artist_name_uc'),)

class Watcher(Base):
    __tablename__ = 'watchers_v1'
    id = Column(Integer, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'), nullable=False)
    watcher_name = Column(String, nullable=False)
    watcher_type = Column(String)
    is_watching = Column(Boolean)
    last_visit = Column(BigInteger)
    activity = Column(Boolean)
    collections = Column(Boolean)
    critiques = Column(Boolean)
    deviations = Column(Boolean)
    forum_threads = Column(Boolean)
    friend = Column(Boolean)
    journals = Column(Boolean)
    scraps = Column(Boolean)
    artist_name = Column(String)
    artist = relationship("Artist", back_populates="watchers_v1")
    __table_args__ = (UniqueConstraint('artist_id', 'watcher_name', name='_artist_watcher_uc'),)

class Friend(Base):
    __tablename__ = 'friends_v1'
    id = Column(Integer, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'), nullable=False)
    friend_name = Column(String, nullable=False)
    friend_type = Column(String)
    is_watching = Column(Boolean)
    last_visit = Column(BigInteger)
    friends = Column(Boolean)
    deviations = Column(Boolean)
    journals = Column(Boolean)
    forum_threads = Column(Boolean)
    critiques = Column(Boolean)
    scraps = Column(Boolean)
    activity = Column(Boolean)
    collections = Column(Boolean)
    artist_name = Column(String)
    watches_you = Column(Boolean)
    artist = relationship("Artist", back_populates="friends_v1")
    __table_args__ = (UniqueConstraint('artist_id', 'friend_name', name='_artist_friend_uc'),)


Artist.watchers_v1 = relationship("Watcher", back_populates="artist")
Artist.friends_v1 = relationship("Friend", back_populates="artist")
#Create tables if they dont exist
Base.metadata.create_all(engine)

# --- Converters for pandas.read_csv (unchanged) ---
def bool_converter(x):
    if pd.isna(x) or not isinstance(x, str):
        return None
    return str(x).lower() == 'true'

def iso_date_to_ms(x):
    if pd.isna(x):
        return None
    if not isinstance(x, str) or not x.strip():
        return None
    try:
        dt_obj = parser.parse(x)
        if dt_obj.tzinfo is None:
             dt_obj = pytz.utc.localize(dt_obj)
        else:
            dt_obj = dt_obj.astimezone(pytz.utc)
        return int(dt_obj.timestamp() * 1000)
    except Exception as e:
        logger.warning(f"Could not convert date string '{x}' to timestamp. Error: {e}. Setting to None.")
        return None

 #   Column         Non-Null Count   Dtype Data columns (total 14 columns):
 # 0   Watchers name  888326 non-null  object
 # 1   user_icon      888326 non-null  object
  #2   type           888326 non-null  object
 # 3   is_watching    888326 non-null  bool
 # 4   last_visit     759781 non-null  object
 # 5   activity       888326 non-null  bool
 # 6   collections    888326 non-null  bool
 # 7   critiques      888326 non-null  bool
 # 8   deviations     888326 non-null  bool
 # 9   forum_threads  888326 non-null  bool
 # 10  friend         888326 non-null  bool
 # 11  journals       888326 non-null  bool
 # 12  scraps         888326 non-null  bool
 # 13  Deviant        888326 non-null  object

 # 0   Friends name   942998 non-null  object
 # 1   user_icon      942998 non-null  object
 # 2   type           942998 non-null  object
 # 3   is_watching    942998 non-null  bool
 # 4   watches_you    942998 non-null  bool
 # 5   last_visit     582454 non-null  object
 # 6   friends        942998 non-null  bool
 # 7   deviations     942998 non-null  bool
 # 8   journals       942998 non-null  bool
 # 9   forum_threads  942998 non-null  bool
 # 10  critiques      942998 non-null  bool
 # 11  scraps         942998 non-null  bool
 # 12  activity       942998 non-null  bool
 # 13  collections   942998 non-null  bool
 # 14  Deviant        942998 non-null  object

# --- Shared Dtype for CSV Reading ---
COMMON_COL_DTYPES = {
    'Watchers name': str, 'user_icon': str, 'type': str, 'Deviant': str,
    'Friends name': str,
}

COMMON_CONVERTERS = {
    'is_watching': bool_converter, 'activity': bool_converter, 'collections': bool_converter,
    'critiques': bool_converter, 'deviations': bool_converter, 'forum_threads': bool_converter,
    'friend': bool_converter, 'journals': bool_converter, 'scraps': bool_converter,
    'watches_you': bool_converter, 'last_visit': iso_date_to_ms
}

# --- Helper to get existing artist IDs once ---
def get_artist_name_to_id_map(session):
    logger.info("Fetching all existing artist names and IDs...")
    artist_map = {}
    for artist_id, artist_name in session.query(Artist.id, Artist.artist_name).all():
        artist_map[str(artist_name)] = artist_id
    logger.info(f"Loaded {len(artist_map)} existing artists.")
    return artist_map

# --- Tracking File Helpers ---
def get_last_processed_chunk(tracking_file_path):
    if not os.path.exists(tracking_file_path):
        return -1 # Start from the beginning if tracking file doesn't exist
    try:
        with open(tracking_file_path, 'r') as f:
            content = f.read().strip()
            if not content:
                return -1
            return int(content)
    except (ValueError, IOError) as e:
        logger.error(f"Error reading tracking file {tracking_file_path}: {e}")
        return -1 # Return -1 in case of error to be safe

def save_last_processed_chunk(tracking_file_path, chunk_index):
    try:
        with open(tracking_file_path, 'w') as f:
            f.write(str(chunk_index))
    except IOError as e:
        logger.error(f"Error writing to tracking file {tracking_file_path}: {e}")


# --- Optimized Incremental Load Function ---
def load_data_incrementally(csv_path, table_class, artist_name_col_csv, related_name_col_csv, chunksize=100000, artist_map_global=None, converters=None, temp_suffix='_temp', tracking_file_suffix='_last_chunk.txt'):
    if artist_map_global is None:
        raise ValueError("artist_map_global must be provided.")

    if not os.path.exists(csv_path):
        logger.error(f"CSV file not found: {csv_path}")
        return

    table_name = table_class.__tablename__
    tracking_file_path = f"{csv_path}{tracking_file_suffix}"
    last_processed_chunk_index = get_last_processed_chunk(tracking_file_path)
    logger.info(f"Starting incremental load for {table_name} from {csv_path}. Resuming from chunk {last_processed_chunk_index + 1}")


    total_rows_processed = 0
    total_skipped_artist = 0
    total_skipped_duplicates = 0
    temp_table_name = table_class.__tablename__ + temp_suffix
    main_table = table_class.__table__

    # Define a mapping from CSV column names to database column names for the current table
    csv_to_db_col_map = {}
    if table_class == Watcher:
        csv_to_db_col_map = {
            'Watchers name': 'watcher_name',
            # 'user_icon': 'user_icon', # Excluded as per user request
            'type': 'watcher_type',
            'is_watching': 'is_watching',
            'last_visit': 'last_visit',
            'activity': 'activity',
            'collections': 'collections',
            'critiques': 'critiques',
            'deviations': 'deviations',
            'forum_threads': 'forum_threads',
            'friend': 'friend',
            'journals': 'journals',
            'scraps': 'scraps',
            'Deviant': 'artist_name', # Mapping Deviant from CSV to artist_name in DB
            # 'artist_id' is handled separately
        }
    elif table_class == Friend:
        csv_to_db_col_map = {
            'Friends name': 'friend_name',
            # 'user_icon': 'user_icon', # Excluded as per user request
            'type': 'friend_type',
            'is_watching': 'is_watching',
            'watches_you': 'watches_you',
            'last_visit': 'last_visit',
            'friends': 'friends',
            'deviations': 'deviations',
            'journals': 'journals',
            'forum_threads': 'forum_threads',
            'critiques': 'critiques',
            'scraps': 'scraps',
            'activity': 'activity',
            'collections': 'collections',
            'Deviant': 'artist_name', # Mapping Deviant from CSV to artist_name in DB
            # 'artist_id' is handled separately
        }
    else:
        logger.error(f"Unknown table class: {table_class.__name__}. Cannot define column mapping.")
        return


    logger.info(f"Starting incremental load for {table_class.__tablename__} from {csv_path}")

    try:
        for i, chunk in enumerate(pd.read_csv(
            csv_path,
            chunksize=chunksize,
            on_bad_lines='skip',
            dtype=COMMON_COL_DTYPES,
            converters=converters # Use the provided converters
        )):
            # Skip chunks that have already been processed
            if i <= last_processed_chunk_index:
                logger.info(f"  Skipping already processed chunk {i + 1} for {table_name}")
                continue

            logger.info(f"Processing chunk {i + 1} for {table_class.__tablename__} (rows read: {len(chunk)})")

            # --- 1. Map artist_name to artist_id ---
            if artist_name_col_csv not in chunk.columns:
                logger.error(f"  Missing required column '{artist_name_col_csv}' in CSV chunk {i+1} for {table_class.__tablename__}. Skipping chunk.")
                continue

            chunk['artist_id'] = chunk[artist_name_col_csv].map(artist_map_global)

            unmapped_artists_df = chunk[chunk['artist_id'].isna()]
            if not unmapped_artists_df.empty:
                logger.warning(f"  Skipping {len(unmapped_artists_df)} rows in chunk {i+1} due to unmapped artist names. Examples: {unmapped_artists_df[artist_name_col_csv].unique()[:5].tolist()}")
                total_skipped_artist += len(unmapped_artists_df)


            chunk_mapped = chunk.dropna(subset=['artist_id']).copy() # Create a copy to avoid SettingWithCopyWarning
            if chunk_mapped.empty:
                logger.info(f"  Chunk {i+1} became empty after filtering unmapped artists.")
                save_last_processed_chunk(tracking_file_path, i) # Save progress even if chunk is empty after filtering
                continue

            chunk_mapped['artist_id'] = chunk_mapped['artist_id'].astype(int)
            logger.info(f"  Rows after artist mapping: {len(chunk_mapped)}")

            # --- 2. Create DataFrame for insertion, matching SQLA model attributes ---
            df_to_insert = pd.DataFrame()

            df_to_insert['artist_id'] = chunk_mapped['artist_id']

            # Use the defined mapping to select and rename columns
            for csv_col, db_col in csv_to_db_col_map.items():
                if csv_col in chunk_mapped.columns:
                    df_to_insert[db_col] = chunk_mapped[csv_col]
                else:
                    # Handle cases where a CSV column expected by the map is missing
                    # For boolean columns, default to False if not in CSV
                    if db_col in [c.name for c in main_table.columns if isinstance(c.type, Boolean)]:
                         df_to_insert[db_col] = False
                         logger.warning(f"  CSV column '{csv_col}' not found in chunk {i+1} for DB column '{db_col}', defaulting to False.")
                    else:
                         df_to_insert[db_col] = None
                         logger.warning(f"  CSV column '{csv_col}' not found in chunk {i+1} for DB column '{db_col}', defaulting to None.")


            # --- Add step to drop duplicates within the chunk DataFrame ---
            # Determine the columns for dropping duplicates based on the unique constraint
            unique_cols_main_names = []
            try:
                unique_constraint_cols = table_class.__table_args__[0].columns
                unique_cols_main_names = [col.name for col in unique_constraint_cols]
            except (AttributeError, IndexError):
                 logger.error(f"  Could not determine unique constraint columns for {table_class.__tablename__} to drop duplicates within chunk.")
                 # Continue without dropping duplicates if unique constraint cols can't be determined


            if unique_cols_main_names and all(col_name in df_to_insert.columns for col_name in unique_cols_main_names):
                initial_rows_in_df = len(df_to_insert)
                df_to_insert.drop_duplicates(subset=unique_cols_main_names, inplace=True)
                rows_dropped_in_df = initial_rows_in_df - len(df_to_insert)
                if rows_dropped_in_df > 0:
                    logger.warning(f"  Dropped {rows_dropped_in_df} duplicate rows within chunk {i+1} DataFrame based on {unique_cols_main_names}.")
                    total_skipped_duplicates += rows_dropped_in_df # Count these as skipped duplicates
            elif unique_cols_main_names:
                 logger.warning(f"  Could not find all unique constraint columns {unique_cols_main_names} in df_to_insert for dropping duplicates within chunk {i+1}.")


            # --- 3. Bulk insert to temporary table ---
            with engine.connect() as conn:
                with conn.begin():
                    try:
                        conn.execute(text(f"DROP TABLE IF EXISTS {temp_table_name}"))
                        logger.info(f"  Dropped temporary table {temp_table_name}")
                    except Exception as drop_e:
                        logger.warning(f"  Could not drop temporary table {temp_table_name}: {drop_e}")


                    sqla_dtype_map = {col.name: col.type for col in main_table.columns if col.name != 'id'}

                    try:
                        # Ensure column order matches the target table exactly before to_sql
                        # Get the list of column names from the main table (excluding 'id')
                        target_column_order = [c.name for c in main_table.columns if c.name != 'id']
                        # Reindex the DataFrame to match the target column order, filling missing columns with None
                        df_to_insert = df_to_insert.reindex(columns=target_column_order)

                        df_to_insert.to_sql(temp_table_name, conn, if_exists='replace', index=False, dtype=sqla_dtype_map)
                        rows_in_temp = len(df_to_insert)
                        logger.info(f"  Inserted {rows_in_temp} rows into temporary table {temp_table_name}")
                    except Exception as to_sql_e:
                        logger.error(f"  Error inserting into temporary table {temp_table_name}: {to_sql_e}", exc_info=True)
                        continue # Skip to next chunk if temp insert fails


                    # --- 4. Atomic insert from temp to main table with WHERE NOT EXISTS ---
                    cols_to_insert = [c.name for c in main_table.columns if c.name != main_table.primary_key.columns.keys()[0]]

                    # Important: Reflect the temporary table to get its Columns for the SELECT statement
                    # Need to ensure metadata is up-to-date for reflection
                    temp_metadata = MetaData()
                    # Use only= parameter to specify the temporary table name for reflection
                    temp_metadata.reflect(bind=conn, only=[temp_table_name])
                    temp_table = temp_metadata.tables[temp_table_name]


                    # Build select list where columns are from the temp_table.c (columns)
                    select_temp_cols = [temp_table.c[col_name] for col_name in cols_to_insert if col_name in temp_table.c]

                    # Build the WHERE NOT EXISTS clause based on the unique constraint columns
                    # The unique constraint is assumed to be the first one in __table_args__
                    try:
                        unique_cols_main_names = [col.name for col in table_class.__table_args__[0].columns]
                        # Ensure unique columns exist in both main and temporary tables before building the WHERE clause
                        if all(col_name in main_table.c and col_name in temp_table.c for col_name in unique_cols_main_names):
                            unique_cols_main = [main_table.c[col_name] for col_name in unique_cols_main_names]
                            unique_cols_temp = [temp_table.c[col_name] for col_name in unique_cols_main_names]

                            # Subquery to check for existence
                            exists_subquery = select(unique_cols_main[0]) \
                                             .where(unique_cols_main[0] == unique_cols_temp[0]) \
                                             .where(unique_cols_main[1] == unique_cols_temp[1]) \
                                             .exists()

                            # The final INSERT FROM SELECT statement
                            insert_stmt = sqlite.insert(main_table).from_select(
                                [c.name for c in select_temp_cols], # Ensure column names match for from_select
                                select(*select_temp_cols).where(~exists_subquery) # Only select rows that do NOT exist in main_table
                            )
                        else:
                            logger.error(f"  Unique constraint columns {unique_cols_main_names} not found in both main and temporary tables for {table_class.__tablename__}. Cannot build WHERE NOT EXISTS clause.")
                            # Fallback to a simple insert (will likely fail on duplicates unless tables were dropped)
                            insert_stmt = sqlite.insert(main_table).from_select(
                                [c.name for c in select_temp_cols],
                                select(*select_temp_cols)
                            )

                    except (AttributeError, IndexError):
                         logger.error(f"  Could not determine unique constraint columns for {table_class.__tablename__}. Skipping insert from temp.")
                         continue


                    logger.info(f"  Generated INSERT statement: {insert_stmt.compile(engine)}")

                    try:
                        result = conn.execute(insert_stmt)
                        rows_inserted = result.rowcount
                        logger.info(f"  Inserted {rows_inserted} new rows into {main_table.name} from temp table.")

                        rows_in_temp = len(df_to_insert)
                        rows_skipped_in_this_chunk = rows_in_temp - rows_inserted
                        # total_skipped_duplicates already includes duplicates dropped within the chunk DataFrame

                        if rows_skipped_in_this_chunk > 0:
                            logger.warning(f"  Skipped {rows_skipped_in_this_chunk} existing {table_class.__tablename__} records in chunk {i+1} during insert from temp (already in main table).")

                        total_rows_processed += rows_inserted

                        # --- Save progress after successful chunk processing ---
                        save_last_processed_chunk(tracking_file_path, i)

                    except Exception as insert_e:
                        logger.error(f"  Error inserting from temporary table to main table {main_table.name}: {insert_e}", exc_info=True)
                        # Reraise the exception to stop processing if an integrity error occurs
                        raise insert_e


    except Exception as e:
        logger.error(f"Error processing {table_class.__tablename__} chunk: {e}", exc_info=True)
        # If the error is an IntegrityError, it will be logged above and the process will stop due to the re-raise
        # For other errors, log and continue to the next file if applicable
        if not isinstance(e, sqlalchemy.exc.IntegrityError):
             logger.error(f"Continuing with the next file/step after error in {table_class.__tablename__}: {e}")


    logger.info(f"Finished loading {table_class.__tablename__}. Total processed: {total_rows_processed} rows. Total skipped unmapped artists: {total_skipped_artist} rows. Total skipped duplicates: {total_skipped_duplicates} rows.")

# --- Main Execution Function ---
def main():
    # Ensure tables exist
    Base.metadata.create_all(engine)
    with Session() as session:
        artist_name_to_id = get_artist_name_to_id_map(session)
        watchers_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_wtchrsSnwball_fin1.csv.gz"
        friends_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_friendsSnwball_fin.csv.gz"

        # --- Removed Drop and Recreate Table Logic ---

        logger.info("\nStarting Friends data load.")
        try:
            load_data_incrementally(
                csv_path=friends_csv,
                table_class=Friend,
                artist_name_col_csv='Deviant',
                related_name_col_csv='Friends name',
                artist_map_global=artist_name_to_id,
                converters=COMMON_CONVERTERS
            )
        except sqlalchemy.exc.IntegrityError as e:
             logger.error(f"IntegrityError occurred during Friends data load: {e}")
             # Decide whether to stop or continue based on your needs
             # For now, let's stop to investigate
             logger.info("Stopping data load due to IntegrityError in Friends data.")
             # No exit() here, allow the main thread to finish


        logger.info("\nStarting Watchers data load.")
        try:
            load_data_incrementally(
                csv_path=watchers_csv,
                table_class=Watcher,
                artist_name_col_csv='Deviant',
                related_name_col_csv='Watchers name',
                artist_map_global=artist_name_to_id,
                converters=COMMON_CONVERTERS
            )
        except sqlalchemy.exc.IntegrityError as e:
             logger.error(f"IntegrityError occurred during Watchers data load: {e}")
             logger.info("Stopping data load due to IntegrityError in Watchers data.")
             # No need to exit here as it's the last step, but you could add a flag if needed

    logger.info("\nDatabase loading complete.")

if __name__ == "__main__":
    # This part will be modified in the next step to use threading
    main()

In [ ]:
import pandas as pd
meta1 = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_metaDataSnwBall/ungDev_metaData_SnwBall_02-04-07-2025.csv.gz")

### Specific Installation and creating the necessary environment for SQL Alchemy database uploading 
#### Sometimes ubove mentioned generic installation fails then use the following commands to install the packages that match the requirements of SQLAlchemy. 
#### Import all the libraries
#### Configure the DB session and its logging

In [1]:
#Import all the necessary libraries
import os
import pandas as pd
import numpy as np
import re
import ast # For literal_eval
import logging
from datetime import datetime
import pytz # Make sure you have `pip install pytz` for timezone awareness
from dateutil import parser # Make sure you have `pip install python-dateutil` for robust date parsing

from sqlalchemy import create_engine, inspect, Column, Integer, String, ForeignKey, BigInteger, Boolean, func, MetaData, Table, UniqueConstraint, text, select
from sqlalchemy.orm import declarative_base, sessionmaker, relationship
from sqlalchemy.dialects import sqlite # Imported for specific SQLite dialect functions if needed

In [2]:
DATABASE_URL = 'sqlite:////mnt/hdd/maittewa/deviantArt_DeviantData/dbData/deviantArt_main05.db' #: This defines the connection string for your SQLite database. It tells SQLAlchemy where to find the database file.
engine = create_engine(DATABASE_URL) #: This creates the SQLAlchemy engine, which is the core object for interacting with the database. It manages connections and provides a way to execute SQL statements.
Base = declarative_base() #: This creates the base class for your declarative models. Your SQLAlchemy classes (like Artist, Watcher, Friend, etc.) will inherit from this Base. As discussed, you only need to call this once in your notebook.
Session = sessionmaker(bind=engine) #: This creates a Session class. A session is a workspace for holding and persisting objects (your model instances) that you want to save to the database.
session = Session() #: This creates an instance of the Session class. You will use this session object to add, query, and delete data from your database.

### Code for User, imgs_date, imgs_dscrpt, imgs_tags classes that define the schema and connections between these tables

In [ ]:
#Base class1. Images user base class that I called Artist
class Artist(Base):
    __tablename__ = 'artists'
    id = Column(Integer, primary_key=True)
    artist_name = Column(String)
    profile_url = Column(String)
    country = Column(String)
    level = Column(String)
    registration_date = Column(Integer)
    no_of_deviations = Column(Integer)  
    no_of_favourites = Column(Integer)
    no_of_user_comments = Column(Integer)
    no_of_pageviews = Column(Integer)
    no_of_profile_comments = Column(Integer)
    is_artist  = Column(Boolean)
    gender = Column(String)
    speciality = Column(String)
    no_of_images = Column(Integer)
    no_of_AI_images = Column(Integer)
    ai_adopter = Column(Boolean)
    ai_adoption_first_time = Column(Integer) 
    #Relationship description
    imgs_date = relationship("imgs_date", back_populates="artist")  # Establish relationship from Artist to images date
    imgs_dscrpt = relationship("imgs_dscrpt", back_populates="artist")  # Establish relationship from Artist to images description
    imgs_tags = relationship("imgs_tags", back_populates="artist")  # Establish relationship from Artist to images tags
    watchers = relationship("Watcher", back_populates="artist") # Establish relationship from Artist to watchers
    friends = relationship("Friend", back_populates="artist") # Establish relationship from Artist to friends
    watchings = relationship("Watching", back_populates="artist") # Establish relationship from Artist to watching
    interactions = relationship("ArtistInteraction", back_populates="artist_to") # Establish relationship from Artist to interactions

#Base class2. Images date class
class imgs_date(Base):  
    __tablename__ = 'imgs_date'
    id = Column(String, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'))  # Link to Artist table
    artist_name = Column(String)
    date = Column(BigInteger)

    artist = relationship("Artist", order_by=artist_id, back_populates="imgs_date")  # Define the relationship

    # Add UniqueConstraint for the 'id' column
    __table_args__ = (
        UniqueConstraint('id', name='unique_imgs_date_id'),
    )

#Base class3. Images description class
class imgs_dscrpt(Base):  
    __tablename__ = 'imgs_dscrpt'
    id = Column(String, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'))  # Link to Artist table
    artist_name = Column(String)
    description = Column(String)

    artist = relationship("Artist", order_by=artist_id, back_populates="imgs_dscrpt")  # Define the relationship

#Base class4. Images tags class
class imgs_tags(Base):  
    __tablename__ = 'imgs_tags'
    tag_id = Column(Integer, primary_key=True, autoincrement=True)
    image_id = Column(String, ForeignKey('imgs_dscrpt.id'))  # Foreign key to imgs_dscrpt
    artist_id = Column(Integer, ForeignKey('artists.id')) 
    artist_name = Column(String)
    tags = Column(String)

    # Relationships (if needed)
    image_description = relationship("imgs_dscrpt", back_populates="tags")  # Relationship to imgs_dscrpt
    artist = relationship("Artist", back_populates="imgs_tags")

imgs_dscrpt.tags = relationship("imgs_tags", order_by=imgs_tags.tag_id, back_populates="image_description")  # Back-reference in imgs_dscrpt


#Only use it for creating a table for the first time
#Base.metadata.create_all(engine)  

### Use these codes to include new tables of friends, watchers, watching, and interaction type after the initial definition of the DB above.
#### I would recommend this code for the four tables (friends, watchers, watching, and interaction type) refined to include datatype conversion of the columns. This conversion allows uploading of the data faster and more consistency in tables.

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import ast
import logging
from datetime import datetime
from dateutil import parser 
import pytz

# --- Configure Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Database Setup (unchanged) ---
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, BigInteger, Boolean, func, MetaData, Table, UniqueConstraint, inspect, text, select
from sqlalchemy.dialects import sqlite

DATABASE_URL = 'sqlite:////mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_main05.db'
engine = create_engine(DATABASE_URL)
Base = declarative_base()
Session = sessionmaker(bind=engine)

# Your Artist, Watcher, Friend models (unchanged)
class Artist(Base):
    __tablename__ = 'artists'
    id = Column(Integer, primary_key=True)
    artist_name = Column(String, unique=True, index=True)
    profile_url = Column(String)
    country = Column(String)
    level = Column(String)
    registration_date = Column(Integer)
    no_of_deviations = Column(Integer)
    no_of_favourites = Column(Integer)
    no_of_user_comments = Column(Integer)
    no_of_pageviews = Column(Integer)
    no_of_profile_comments = Column(Integer)
    is_artist = Column(Boolean)
    gender = Column(String)
    speciality = Column(String)
    no_of_images = Column(Integer)
    no_of_AI_images = Column(Integer)
    ai_adopter = Column(Boolean)
    ai_adoption_first_time = Column(Integer)
    watchers = relationship("Watcher", back_populates="artist")
    friends = relationship("Friend", back_populates="artist")
    __table_args__ = (UniqueConstraint('artist_name', name='_artist_name_uc'),)

class Watcher(Base):
    __tablename__ = 'watchers'
    id = Column(Integer, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'), nullable=False)
    watcher_name = Column(String, nullable=False)
    user_icon = Column(String)
    watcher_type = Column(String)
    is_watching = Column(Boolean)
    last_visit = Column(BigInteger)
    activity = Column(Boolean)
    collections = Column(Boolean)
    critiques = Column(Boolean)
    deviations = Column(Boolean)
    forum_threads = Column(Boolean)
    friend = Column(Boolean)
    journals = Column(Boolean)
    scraps = Column(Boolean)
    artist_name = Column(String) 
    artist = relationship("Artist", back_populates="watchers")
    __table_args__ = (UniqueConstraint('artist_id', 'watcher_name', name='_artist_watcher_uc'),)

class Friend(Base):
    __tablename__ = 'friends'
    id = Column(Integer, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'), nullable=False)
    friend_name = Column(String, nullable=False)
    user_icon = Column(String)
    friend_type = Column(String)
    is_watching = Column(Boolean)
    last_visit = Column(BigInteger) 
    friends = Column(Boolean)
    deviations = Column(Boolean)
    journals = Column(Boolean)
    forum_threads = Column(Boolean)
    critiques = Column(Boolean)
    scraps = Column(Boolean)
    activity = Column(Boolean)
    collections = Column(Boolean)
    artist_name = Column(String) 
    watches_you = Column(Boolean)
    artist = relationship("Artist", back_populates="friends")
    __table_args__ = (UniqueConstraint('artist_id', 'friend_name', name='_artist_friend_uc'),)

Base.metadata.create_all(engine)

# --- Converters for pandas.read_csv (unchanged) ---
def bool_converter(x):
    if pd.isna(x) or not isinstance(x, str):
        return None
    return str(x).lower() == 'true'

def iso_date_to_ms(x):
    if pd.isna(x):
        return None
    if not isinstance(x, str) or not x.strip():
        return None
    try:
        dt_obj = parser.parse(x)
        if dt_obj.tzinfo is None: 
             dt_obj = pytz.utc.localize(dt_obj)
        else:
            dt_obj = dt_obj.astimezone(pytz.utc)
        return int(dt_obj.timestamp() * 1000)
    except Exception as e:
        logger.warning(f"Could not convert date string '{x}' to timestamp. Error: {e}. Setting to None.")
        return None

# --- Shared Dtype for CSV Reading ---
COMMON_COL_DTYPES = {
    'user_icon': str, 'watcher_type': str, 'artist_name': str,
    'Friend Name': str, 'Deviant': str, 'Watchers name': str
}

COMMON_CONVERTERS = {
    'is_watching': bool_converter, 'activity': bool_converter, 'collections': bool_converter,
    'critiques': bool_converter, 'deviations': bool_converter, 'forum_threads': bool_converter,
    'friend': bool_converter, 'journals': bool_converter, 'scraps': bool_converter,
    'watches_you': bool_converter,
    'last_visit': iso_date_to_ms
}

# --- Helper to get existing artist IDs once ---
def get_artist_name_to_id_map(session):
    logger.info("Fetching all existing artist names and IDs...")
    artist_map = {}
    for artist_id, artist_name in session.query(Artist.id, Artist.artist_name).all():
        artist_map[str(artist_name)] = artist_id
    logger.info(f"Loaded {len(artist_map)} existing artists.")
    return artist_map

# --- Optimized Incremental Load Function ---
def load_data_incrementally(csv_path, table_class, artist_name_col_csv, related_name_col_csv, chunksize=100000, artist_map_global=None, converters=None, temp_suffix='_temp'):
    if artist_map_global is None:
        raise ValueError("artist_map_global must be provided.")

    total_rows_processed = 0
    total_skipped_artist = 0
    total_skipped_duplicates = 0
    temp_table_name = table_class.__tablename__ + temp_suffix
    main_table = table_class.__table__ 

    logger.info(f"Starting incremental load for {table_class.__tablename__} from {csv_path}")
    
    try:
        for i, chunk in enumerate(pd.read_csv(
            csv_path,
            chunksize=chunksize,
            on_bad_lines='skip',
            dtype=COMMON_COL_DTYPES, 
            converters=COMMON_CONVERTERS
        )):
            logger.info(f"Processing chunk {i + 1} for {table_class.__tablename__} (rows {total_rows_processed + 1} to {total_rows_processed + len(chunk)})")
            
            # --- 1. Map artist_name to artist_id ---
            if artist_name_col_csv not in chunk.columns:
                logger.error(f"  Missing required column '{artist_name_col_csv}' in CSV chunk {i+1} for {table_class.__tablename__}. Skipping chunk.")
                continue

            chunk['artist_id'] = chunk[artist_name_col_csv].map(artist_map_global)
            
            unmapped_artists_df = chunk[chunk['artist_id'].isna()]
            if not unmapped_artists_df.empty:
                logger.warning(f"  Skipping {len(unmapped_artists_df)} rows in chunk {i+1} due to unmapped artist names. Examples: {unmapped_artists_df[artist_name_col_csv].unique()[:5].tolist()}")
                total_skipped_artist += len(unmapped_artists_df)
                chunk = chunk.dropna(subset=['artist_id']) 

            if chunk.empty:
                logger.info(f"  Chunk {i+1} became empty after filtering unmapped artists.")
                continue

            chunk['artist_id'] = chunk['artist_id'].astype(int)

            # --- 2. Create DataFrame for insertion, matching SQLA model attributes ---
            df_to_insert = pd.DataFrame()
            
            df_to_insert['artist_id'] = chunk['artist_id']
            
            related_name_col_sqla = table_class.__table_args__[0].columns[1].name
            
            if related_name_col_csv not in chunk.columns:
                logger.error(f"  Missing required column '{related_name_col_csv}' in CSV chunk {i+1} for {table_class.__tablename__}. Skipping chunk.")
                continue
            
            df_to_insert[related_name_col_sqla] = chunk[related_name_col_csv]
            
            # Dynamically add other columns from the CSV that match SQLA model attributes
            for col_name in [c.name for c in main_table.columns]:
                if col_name in ['id', 'artist_id', related_name_col_sqla]:
                    continue
                
                if col_name in chunk.columns:
                    df_to_insert[col_name] = chunk[col_name]
                else:
                    df_to_insert[col_name] = None 
            
            # --- 3. Bulk insert to temporary table ---
            with engine.connect() as conn:
                with conn.begin():
                    conn.execute(text(f"DROP TABLE IF EXISTS {temp_table_name}"))
                    
                    sqla_dtype_map = {col.name: col.type for col in main_table.columns if col.name != 'id'}
                    
                    df_to_insert.to_sql(temp_table_name, conn, if_exists='replace', index=False, dtype=sqla_dtype_map)
                    
                    # --- 4. Atomic insert from temp to main table with WHERE NOT EXISTS ---
                    cols_to_insert = [c.name for c in main_table.columns if c.name != main_table.primary_key.columns.keys()[0]]

                    # Important: Reflect the temporary table to get its Columns for the SELECT statement
                    temp_table = Table(temp_table_name, Base.metadata, autoload_with=conn)
                    
                    # Build select list where columns are from the temp_table.c (columns)
                    select_temp_cols = [temp_table.c[col_name] for col_name in cols_to_insert]
                    
                    # Build the WHERE NOT EXISTS clause based on the unique constraint columns
                    # The unique constraint is assumed to be the first one in __table_args__
                    unique_cols_main = [main_table.c[col.name] for col in table_class.__table_args__[0].columns]
                    unique_cols_temp = [temp_table.c[col.name] for col in table_class.__table_args__[0].columns]
                    
                    # Subquery to check for existence
                    # This creates a SELECT statement from the main table, effectively checking if the (artist_id, watcher_name)
                    # or (artist_id, friend_name) already exist.
                    exists_subquery = select(unique_cols_main[0]) \
                                     .where(unique_cols_main[0] == unique_cols_temp[0]) \
                                     .where(unique_cols_main[1] == unique_cols_temp[1]) \
                                     .exists()
                    
                    # The final INSERT FROM SELECT statement
                    insert_stmt = sqlite.insert(main_table).from_select(
                        cols_to_insert, 
                        select(*select_temp_cols).where(~exists_subquery) # Only select rows that do NOT exist in main_table
                    )
                    
                    result = conn.execute(insert_stmt)
                    rows_inserted = result.rowcount

                    rows_in_temp = len(df_to_insert)
                    rows_skipped_in_this_chunk = rows_in_temp - rows_inserted
                    total_skipped_duplicates += rows_skipped_in_this_chunk

                    if rows_skipped_in_this_chunk > 0:
                        logger.warning(f"  Skipped {rows_skipped_in_this_chunk} existing {table_class.__tablename__} records in chunk {i+1} due to duplicate keys.")
                    
                    total_rows_processed += rows_inserted

    except Exception as e:
        logger.error(f"Error processing {table_class.__tablename__} chunk: {e}", exc_info=True)
    
    logger.info(f"Finished loading {table_class.__tablename__}. Total processed: {total_rows_processed} rows. Total skipped unmapped artists: {total_skipped_artist} rows. Total skipped duplicates: {total_skipped_duplicates} rows.")

# --- Main Execution ---
if __name__ == "__main__":
    # Ensure tables exist
    Base.metadata.create_all(engine)

    with Session() as session:
        artist_name_to_id = get_artist_name_to_id_map(session)

        watchers_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_wtchrsSnwball_fin1.csv.gz"
        friends_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_friendsSnwball_fin.csv.gz"

        logger.info("Starting Watchers data load.")
        load_data_incrementally(
            csv_path=watchers_csv,
            table_class=Watcher,
            artist_name_col_csv='Deviant',
            related_name_col_csv='Watchers name',
            artist_map_global=artist_name_to_id,
            converters=COMMON_CONVERTERS
        )

        logger.info("\nStarting Friends data load.")
        load_data_incrementally(
            csv_path=friends_csv,
            table_class=Friend,
            artist_name_col_csv='Deviant',
            related_name_col_csv='Friend Name',
            artist_map_global=artist_name_to_id,
            converters=COMMON_CONVERTERS
        )

    logger.info("\nDatabase loading complete.")

### To summarize a given database you can use the following codes.

In [3]:
import os
import pandas as pd
import numpy as np
import re
import ast # For literal_eval
import logging
from datetime import datetime
import pytz # Make sure you have `pip install pytz` for timezone awareness
from dateutil import parser # Make sure you have `pip install python-dateutil` for robust date parsing

from sqlalchemy import create_engine, inspect, Column, Integer, String, ForeignKey, BigInteger, Boolean, func, MetaData, Table, UniqueConstraint, text, select
from sqlalchemy.orm import declarative_base, sessionmaker, relationship
from sqlalchemy.dialects import sqlite # Imported for specific SQLite dialect functions if needed

# --- Configure Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Database Connection for the BACKUP DB ---
# IMPORTANT: Point this to your backup file
#DATABASE_URL_BACKUP = 'sqlite:////mnt/hdd/maittewa/deviantArt_DeviantData/dbData/deviantArt_main05_backup_20250603.db' # Adjust path
#engine_backup = create_engine(DATABASE_URL_BACKUP)
inspector = inspect(engine) # Used for schema introspection

# Define Base for database modeling (even if not creating, allows reflection)
Base = declarative_base() # This Base should ideally be the same you used for creating tables

In [6]:
# --- Define your models (necessary for relationships and type mapping) ---
# It's crucial to define your SQLAlchemy models as they were originally defined
# when the database schema was created, for consistent type handling and reflection.
class Artist(Base):
    __tablename__ = 'artists'
    id = Column(Integer, primary_key=True)
    artist_name = Column(String, unique=True, index=True)
    profile_url = Column(String)
    country = Column(String)
    level = Column(String)
    registration_date = Column(Integer)
    no_of_deviations = Column(Integer)
    no_of_favourites = Column(Integer)
    no_of_user_comments = Column(Integer)
    no_of_pageviews = Column(Integer)
    no_of_profile_comments = Column(Integer)
    is_artist = Column(Boolean)
    gender = Column(String)
    speciality = Column(String)
    no_of_images = Column(Integer)
    no_of_AI_images = Column(Integer) # Integer as per your original code
    ai_adopter = Column(Boolean)
    ai_adoption_first_time = Column(Integer)
    # Relationships for ArtistInteraction (if you used ArtistInteraction.interactions)
    interactions = relationship("ArtistInteraction", back_populates="artist_to")
    # Add other relationships if Artist is related to other tables you have (e.g., Watching)
    # watching = relationship("Watching", back_populates="artist") # If you have a Watching table


class Watcher(Base):
    __tablename__ = 'watchers'
    id = Column(Integer, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'), nullable=False)
    watcher_name = Column(String, nullable=False)
    user_icon = Column(String)
    watcher_type = Column(String)
    is_watching = Column(Boolean)
    last_visit = Column(BigInteger)
    activity = Column(Boolean)
    collections = Column(Boolean)
    critiques = Column(Boolean)
    deviations = Column(Boolean)
    forum_threads = Column(Boolean)
    friend = Column(Boolean)
    journals = Column(Boolean)
    scraps = Column(Boolean)
    artist_name = Column(String) 
    artist = relationship("Artist", back_populates="watchers")
    __table_args__ = (UniqueConstraint('artist_id', 'watcher_name', name='_artist_watcher_uc'),)

class Friend(Base):
    __tablename__ = 'friends'
    id = Column(Integer, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'), nullable=False)
    friend_name = Column(String, nullable=False)
    user_icon = Column(String)
    friend_type = Column(String)
    is_watching = Column(Boolean)
    last_visit = Column(BigInteger) 
    friends = Column(Boolean)
    deviations = Column(Boolean)
    journals = Column(Boolean)
    forum_threads = Column(Boolean)
    critiques = Column(Boolean)
    scraps = Column(Boolean)
    activity = Column(Boolean)
    collections = Column(Boolean)
    artist_name = Column(String) 
    watches_you = Column(Boolean)
    artist = relationship("Artist", back_populates="friends")
    __table_args__ = (UniqueConstraint('artist_id', 'friend_name', name='_artist_friend_uc'),)

# 3. Define Watching table
class Watching(Base):
    __tablename__ = 'watchings'
    id = Column(Integer, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id')) # Foreign key to Artist table
    watching_name = Column(String)
    artist = relationship("Artist", back_populates="watchings")
    
#Images date class
class imgs_date(Base):  
    __tablename__ = 'imgs_date'
    id = Column(String, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'))  # Link to Artist table
    artist_name = Column(String)
    date = Column(BigInteger)

    artist = relationship("Artist", order_by=artist_id, back_populates="imgs_date")  # Define the relationship

    # Add UniqueConstraint for the 'id' column
    __table_args__ = (
        UniqueConstraint('id', name='unique_imgs_date_id'),
    )

#Images description class
class imgs_dscrpt(Base):  
    __tablename__ = 'imgs_dscrpt'
    id = Column(String, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'))  # Link to Artist table
    artist_name = Column(String)
    description = Column(String)

    artist = relationship("Artist", order_by=artist_id, back_populates="imgs_dscrpt")  # Define the relationship

class imgs_tags(Base):  
    __tablename__ = 'imgs_tags'
    tag_id = Column(Integer, primary_key=True, autoincrement=True)
    image_id = Column(String, ForeignKey('imgs_dscrpt.id'))  # Foreign key to imgs_dscrpt
    artist_id = Column(Integer, ForeignKey('artists.id')) 
    artist_name = Column(String)
    tags = Column(String)

    # Relationships (if needed)
    image_description = relationship("imgs_dscrpt", back_populates="tags")  # Relationship to imgs_dscrpt
    artist = relationship("Artist", back_populates="imgs_tags")

class ArtistInteraction(Base):
    __tablename__ = 'artist_interactions'
    id = Column(Integer, primary_key=True)
    artist_id_to = Column(Integer, ForeignKey('artists.id'), nullable=False)
    artist_name_to = Column(String, nullable=False)
    artist_id_from = Column(BigInteger, nullable=False) # BigInteger for global interactor IDs
    artist_name_from = Column(String, nullable=False)
    interaction_type = Column(String, nullable=False) # 'watcher', 'watching', 'friend' (tag)
    date = Column(BigInteger, nullable=True) # Unix timestamp in milliseconds

    artist_to = relationship("Artist", back_populates="interactions")

    # Add other relationships back to Artist if needed (if defining them on ArtistInteraction side)
    # For a simple edge list, back_populates from Artist is usually enough.

    __table_args__ = (
        UniqueConstraint('artist_id_to', 'artist_id_from', 'interaction_type', name='_interaction_edge_uc'),
    )

In [5]:
def summarize_database(engine, inspector):
    """
    Connects to a database and provides a summary of all its tables,
    including schema details, row counts, and basic column statistics.
    """
    logger.info(f"--- Summarizing Database: {engine.url.database} ---")

    table_names = inspector.get_table_names()

    if not table_names:
        logger.warning("No tables found in the database.")
        return

    logger.info(f"Found {len(table_names)} tables: {', '.join(table_names)}")
    logger.info("=" * 60)

    for table_name in table_names:
        logger.info(f"\n--- TABLE: {table_name} ---")

        # 1. Table Schema
        columns = inspector.get_columns(table_name)
        print("  Columns:")
        for col in columns:
            pk_str = ' [PK]' if col['primary_key'] else ''
            nullable_str = ' [NULL]' if col['nullable'] else ''
            # Check for foreign keys
            fk_str = ' [FK]' if col.get('foreign_keys') and len(col['foreign_keys']) > 0 else ''
            print(f"    - {col['name']} ({col['type']}){pk_str}{fk_str}{nullable_str}")

        pk_constraint = inspector.get_pk_constraint(table_name)
        if pk_constraint and pk_constraint['constrained_columns']:
            print(f"  Primary Key: {', '.join(pk_constraint['constrained_columns'])}")

        foreign_keys = inspector.get_foreign_keys(table_name)
        if foreign_keys:
            print("  Foreign Keys:")
            for fk in foreign_keys:
                print(f"    - {', '.join(fk['constrained_columns'])} -> {fk['referred_table']}.{', '.join(fk['referred_columns'])}")
        
        unique_constraints = inspector.get_unique_constraints(table_name)
        if unique_constraints:
            print("  Unique Constraints:")
            for uc in unique_constraints:
                print(f"    - Name: {uc['name']}, Columns: {', '.join(uc['constrained_columns'])}")


        # 2. Total Rows
        # Reflect the table for querying
        metadata_obj = MetaData()
        current_table = Table(table_name, metadata_obj, autoload_with=engine)
        
        with engine.connect() as connection:
            row_count = connection.scalar(select(func.count()).select_from(current_table))
            print(f"\n  Total Rows: {row_count}")

            # 3. Basic Column Statistics (expensive for large tables, consider optimizing if performance is an issue)
            print("  Basic Statistics:")
            for col in columns:
                col_name = col['name']
                col_type = col['type']
                sqla_col = current_table.c[col_name] # Get the Column object from the reflected table

                try:
                    if isinstance(col_type, (Integer, BigInteger, Boolean)): # Include Boolean for min/max (0/1)
                        min_val = connection.scalar(select(func.min(sqla_col)).select_from(current_table))
                        max_val = connection.scalar(select(func.max(sqla_col)).select_from(current_table))
                        
                        # Handle average carefully, only if numeric
                        avg_val = None
                        if isinstance(col_type, (Integer, BigInteger)):
                            avg_val = connection.scalar(select(func.avg(sqla_col)).select_from(current_table))
                            print(f"    - {col_name} (Numeric): Min={min_val}, Max={max_val}, Avg={avg_val:.2f}" if avg_val is not None else f"    - {col_name} (Numeric): Min={min_val}, Max={max_val}, Avg=N/A")
                        else: # For Boolean, min/max are 0/1
                            print(f"    - {col_name} (Boolean): Min={min_val}, Max={max_val}")
                        
                        # Special handling for date columns stored as BigInteger
                        if col_name == 'last_visit' and (min_val is not None or max_val is not None):
                            try:
                                min_date = datetime.fromtimestamp(min_val / 1000, tz=pytz.utc).strftime('%Y-%m-%d %H:%M:%S %Z') if min_val is not None else "N/A"
                                max_date = datetime.fromtimestamp(max_val / 1000, tz=pytz.utc).strftime('%Y-%m-%d %H:%M:%S %Z') if max_val is not None else "N/A"
                                print(f"      (Date Range: {min_date} to {max_date})")
                            except (TypeError, ValueError, OSError): # Handle invalid timestamps (e.g., if column has non-timestamp integers)
                                print(f"      (Date Conversion Error for {col_name})")

                    elif isinstance(col_type, String):
                        distinct_count = connection.scalar(select(func.count(sqla_col.distinct())).select_from(current_table))
                        # Fetch some sample values if there are few distinct ones (optional)
                        # if distinct_count < 10 and distinct_count is not None:
                        #     sample_values = connection.scalars(select(sqla_col.distinct()).limit(5)).all()
                        #     print(f"      Sample: {sample_values}")
                        print(f"    - {col_name} (String): Distinct Count={distinct_count}")
                    else:
                        print(f"    - {col_name} (Other Type): Stats not implemented.")
                except Exception as e:
                    logger.warning(f"  Error getting stats for column {table_name}.{col_name}: {e}")
                    print(f"    - {col_name}: Error retrieving stats.")
        logger.info("\n" + "="*60 + "\n")

# --- Main Execution ---
if __name__ == "__main__":
    # You don't need to call Base.metadata.create_all(engine_backup) for a backup DB,
    # as you are only inspecting it. The models are defined for introspection.

    summarize_database(engine, inspector)
    logger.info("Database summary complete.")

2026-01-22 11:39:42,857 - INFO - --- Summarizing Database: /mnt/hdd/maittewa/deviantArt_DeviantData/dbData/deviantArt_main05.db ---
2026-01-22 11:39:42,883 - INFO - Found 18 tables: artist_interactions, artist_interactions_temp, artists, friends, friends_temp, friends_v1, friends_v1_temp, gallery, imgs_date, imgs_dscrpt, imgs_tags, meta, metaData, watchers, watchers_temp, watchers_v1, watchers_v1_temp, watchings
2026-01-22 11:39:42,884 - INFO - ============================================================
2026-01-22 11:39:42,884 - INFO - 
--- TABLE: artist_interactions ---


  Columns:
    - id (INTEGER) [PK]
    - artist_id_to (INTEGER) [NULL]
    - artist_name_to (VARCHAR) [NULL]
    - artist_id_from (INTEGER) [NULL]
    - artist_name_from (VARCHAR) [NULL]
    - interaction_type (VARCHAR) [NULL]
    - date (VARCHAR) [NULL]
  Primary Key: id
  Foreign Keys:
    - artist_id_to -> artists.id

  Total Rows: 9134178
  Basic Statistics:
    - id (Numeric): Min=1, Max=9134178, Avg=4567089.50
    - artist_id_to (Numeric): Min=1, Max=126789, Avg=16062.06
    - artist_name_to (String): Distinct Count=40522
    - artist_id_from (Numeric): Min=1, Max=759394, Avg=181514.19
    - artist_name_from (String): Distinct Count=843980
    - interaction_type (String): Distinct Count=3


2026-01-22 11:39:57,517 - INFO - 

2026-01-22 11:39:57,517 - INFO - 
--- TABLE: artist_interactions_temp ---
2026-01-22 11:39:57,524 - INFO - 

2026-01-22 11:39:57,524 - INFO - 
--- TABLE: artists ---


    - date (String): Distinct Count=781128
  Columns:
    - artist_id_to (INTEGER) [NULL]
    - artist_name_to (VARCHAR) [NULL]
    - artist_id_from (BIGINT) [NULL]
    - artist_name_from (VARCHAR) [NULL]
    - interaction_type (VARCHAR) [NULL]
    - date (BIGINT) [NULL]

  Total Rows: 0
  Basic Statistics:
    - artist_id_to (Numeric): Min=None, Max=None, Avg=N/A
    - artist_name_to (String): Distinct Count=0
    - artist_id_from (Numeric): Min=None, Max=None, Avg=N/A
    - artist_name_from (String): Distinct Count=0
    - interaction_type (String): Distinct Count=0
    - date (Numeric): Min=None, Max=None, Avg=N/A
  Columns:
    - id (INTEGER) [PK]
    - artist_name (VARCHAR) [NULL]
    - profile_url (VARCHAR) [NULL]
    - country (VARCHAR) [NULL]
    - level (VARCHAR) [NULL]
    - registration_date (INTEGER) [NULL]
    - no_of_deviations (INTEGER) [NULL]
    - no_of_favourites (INTEGER) [NULL]
    - no_of_user_comments (INTEGER) [NULL]
    - no_of_pageviews (INTEGER) [NULL]
    - n

2026-01-22 11:39:58,015 - INFO - 

2026-01-22 11:39:58,016 - INFO - 
--- TABLE: friends ---
2026-01-22 11:39:58,097 - INFO - 

2026-01-22 11:39:58,097 - INFO - 
--- TABLE: friends_temp ---
2026-01-22 11:39:58,118 - INFO - 

2026-01-22 11:39:58,118 - INFO - 
--- TABLE: friends_v1 ---


    - ai_adopter (Boolean): Min=None, Max=None
    - ai_adoption_first_time (Numeric): Min=None, Max=None, Avg=N/A
  Columns:
    - id (INTEGER) [PK]
    - artist_id (INTEGER) [NULL]
    - friend_name (VARCHAR) [NULL]
    - user_icon (VARCHAR) [NULL]
    - friend_type (VARCHAR) [NULL]
    - is_watching (BOOLEAN) [NULL]
    - last_visit (VARCHAR) [NULL]
    - friends (BOOLEAN) [NULL]
    - deviations (BOOLEAN) [NULL]
    - journals (BOOLEAN) [NULL]
    - forum_threads (BOOLEAN) [NULL]
    - critiques (BOOLEAN) [NULL]
    - scraps (BOOLEAN) [NULL]
    - activity (BOOLEAN) [NULL]
    - collections (BOOLEAN) [NULL]
    - artist_name (VARCHAR) [NULL]
    - watches_you (BOOLEAN) [NULL]
  Primary Key: id
  Foreign Keys:
    - artist_id -> artists.id

  Total Rows: 30027
  Basic Statistics:
    - id (Numeric): Min=1, Max=30027, Avg=15014.00
    - artist_id (Numeric): Min=78, Max=21962, Avg=11300.87
    - friend_name (String): Distinct Count=18976
    - user_icon (String): Distinct Count=0
    

KeyError: 'constrained_columns'

#### beautifying the summary of the data

In [9]:
!pip install tabulate

  Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)


In [ ]:
from sqlalchemy import create_engine, MetaData, text
import pandas as pd

# 1. Database Connection
engine = create_engine('sqlite:////mnt/hdd/maittewa/deviantArt_DeviantData/dbData/deviantArt_main05.db')
metadata = MetaData()

def generate_database_report():
    # 2. Reflect the database to get all table names
    metadata.reflect(bind=engine)
    tables = metadata.tables.keys()
    
    print(f"Generating report for {len(tables)} tables...")
    
    for table_name in tables:
        print(f"\n\n{'#'*60}")
        print(f"## REPORT: {table_name.upper()}")
        print(f"{'#'*60}")
        
        # A. Fetch Table Stats (Row Count)
        with engine.connect() as conn:
            count_query = text(f"SELECT COUNT(*) FROM {table_name}")
            row_count = conn.execute(count_query).scalar()
            
            # B. Fetch Sample Data
            sample_query = text(f"SELECT * FROM {table_name} LIMIT 5")
            df = pd.read_sql(sample_query, conn)
        
        print(f"\n**Total Rows:** {row_count}")
        
        # C. Print Column Summary (Types)
        print("\n**Column Definitions:**")
        cols_info = []
        for col in metadata.tables[table_name].columns:
            cols_info.append({
                "Column": col.name,
                "Type": col.type,
                "Primary Key": col.primary_key,
                "Nullable": col.nullable
            })
        print(pd.DataFrame(cols_info).to_markdown(index=False))
        
        # D. Print Data Sample
        print("\n**Data Preview (First 5 Rows):**")
        if not df.empty:
            print(df.to_markdown(index=False))
        else:
            print("_Table is currently empty._")

# Run the report
generate_database_report()

Generating report for 18 tables...


############################################################
## REPORT: ARTIST_INTERACTIONS
############################################################

**Total Rows:** 9134178

**Column Definitions:**
| Column           | Type    | Primary Key   | Nullable   |
|:-----------------|:--------|:--------------|:-----------|
| id               | INTEGER | True          | False      |
| artist_id_to     | INTEGER | False         | True       |
| artist_name_to   | VARCHAR | False         | True       |
| artist_id_from   | INTEGER | False         | True       |
| artist_name_from | VARCHAR | False         | True       |
| interaction_type | VARCHAR | False         | True       |
| date             | VARCHAR | False         | True       |

**Data Preview (First 5 Rows):**
|   id |   artist_id_to | artist_name_to   |   artist_id_from | artist_name_from    | interaction_type   |          date |
|-----:|---------------:|:-----------------|-----------------:|

In [1]:
from sqlalchemy import create_engine, MetaData, text
import pandas as pd

# Connection setup
engine = create_engine('sqlite:////mnt/hdd/maittewa/deviantArt_DeviantData/dbData/deviantArt_main05.db')
metadata = MetaData()
metadata.reflect(bind=engine)

def export_searchable_report(filename="db_dashboard.html"):
    active_sections = []
    toc_links = []
    skipped_tables = []
    
    # Advanced CSS & JS for the Search Feature
    style_and_script = """
    <style>
        body { font-family: 'Inter', -apple-system, sans-serif; margin: 0; padding: 40px; background-color: #f3f4f6; color: #1f2937; }
        .container { max-width: 1100px; margin: auto; }
        h1 { font-size: 2.5em; margin-bottom: 30px; color: #111827; }
        
        /* Search & TOC */
        .toc-card { background: white; padding: 25px; border-radius: 12px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); margin-bottom: 40px; }
        #tableSearch { width: 100%; padding: 12px; margin-bottom: 20px; border: 1px solid #d1d5db; border-radius: 8px; font-size: 1em; outline: none; }
        #tableSearch:focus { border-color: #3b82f6; box-shadow: 0 0 0 3px rgba(59, 130, 246, 0.1); }
        .toc-list { list-style: none; padding: 0; max-height: 400px; overflow-y: auto; }
        .toc-item { display: flex; justify-content: space-between; padding: 10px; border-bottom: 1px solid #f3f4f6; text-decoration: none; color: #4b5563; transition: 0.2s; }
        .toc-item:hover { background-color: #eff6ff; color: #2563eb; }
        
        /* Table Sections */
        .table-card { background: white; padding: 30px; border-radius: 12px; box-shadow: 0 10px 15px -3px rgba(0,0,0,0.1); margin-bottom: 60px; }
        .table-card h2 { color: #2563eb; margin-top: 0; display: flex; align-items: center; justify-content: space-between; }
        
        /* Data Tables */
        table { border-collapse: collapse; width: 100%; margin: 20px 0; border: 1px solid #e5e7eb; }
        th { background-color: #f9fafb; color: #374151; text-align: left; padding: 12px; font-weight: 600; border-bottom: 2px solid #e5e7eb; }
        td { padding: 10px; border-bottom: 1px solid #e5e7eb; font-size: 0.9em; }
        .badge { background: #dcfce7; color: #166534; padding: 4px 12px; border-radius: 9999px; font-size: 0.75em; }
        .skipped-box { background: #fffbeb; border: 1px solid #fef3c7; color: #92400e; padding: 20px; border-radius: 8px; margin-top: 40px; }
    </style>

    <script>
        function filterTables() {
            let input = document.getElementById('tableSearch').value.toLowerCase();
            let items = document.getElementsByClassName('toc-item');
            for (let i = 0; i < items.size; i++) {
                let text = items[i].textContent.toLowerCase();
                items[i].style.display = text.includes(input) ? "flex" : "none";
            }
        }
    </script>
    """

    for table_name in metadata.tables.keys():
        with engine.connect() as conn:
            row_count = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
            if row_count == 0:
                skipped_tables.append(table_name)
                continue
            
            df_sample = pd.read_sql(text(f"SELECT * FROM {table_name} LIMIT 10"), conn)
            
            # Schema logic
            cols = []
            for col in metadata.tables[table_name].columns:
                cols.append({
                    "Column": col.name, "Type": str(col.type), 
                    "PK": "🔑" if col.primary_key else "", "Nullable": "✔" if col.nullable else "✘"
                })
            df_schema = pd.DataFrame(cols)

        # Build TOC Links
        toc_links.append(f'<a href="#{table_name}" class="toc-item"><span>{table_name}</span> <span class="badge">{row_count} rows</span></a>')
        
        # Build Table Sections
        section = f"""
        <div class="table-card" id="{table_name}">
            <h2>{table_name.upper()} <span class="badge">ACTIVE</span></h2>
            <div style="margin-bottom: 25px;">
                <h4 style="color:#6b7280; margin-bottom:10px;">Structure</h4>
                {df_schema.to_html(index=False, border=0)}
            </div>
            <h4 style="color:#6b7280; margin-bottom:10px;">Data Preview (Top 10)</h4>
            <div style="overflow-x: auto;">{df_sample.to_html(index=False, border=0)}</div>
        </div>
        """
        active_sections.append(section)

    # Skipped Tables Footer
    skipped_html = ""
    if skipped_tables:
        skipped_list = ", ".join([f"<code>{t}</code>" for t in skipped_tables])
        skipped_html = f'<div class="skipped-box"><strong>Empty Tables Excluded:</strong><br>{skipped_list}</div>'

    # Combine everything
    full_html = f"""
    <!DOCTYPE html>
    <html>
        <head>
            <title>DB Data Explorer</title>
            {style_and_script}
        </head>
        <body>
            <div class="container">
                <h1>Data Insights Explorer</h1>
                
                <div class="toc-card">
                    <h3>Table Inventory</h3>
                    <input type="text" id="tableSearch" onkeyup="filterTables()" placeholder="Search tables by name...">
                    <div class="toc-list">
                        {''.join(toc_links)}
                    </div>
                </div>

                {''.join(active_sections)}
                {skipped_html}
            </div>
        </body>
    </html>
    """
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(full_html)
    print(f"Interactive Dashboard created: {filename}")

export_searchable_report()

Interactive Dashboard created: db_dashboard.html


In [ ]:
import os
import pandas as pd
import numpy as np
import re
import ast # For literal_eval
import logging
from datetime import datetime
import pytz # Make sure you have `pip install pytz` for timezone awareness
from dateutil import parser # Make sure you have `pip install python-dateutil` for robust date parsing

from sqlalchemy import create_engine, inspect, Column, Integer, String, ForeignKey, BigInteger, Boolean, func, MetaData, Table, UniqueConstraint, text, select
from sqlalchemy.orm import declarative_base, sessionmaker, relationship
from sqlalchemy.dialects import sqlite # Imported for specific SQLite dialect functions if needed

# --- Configure Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Database Connection for the BACKUP DB ---
# IMPORTANT: Point this to your backup file
DATABASE_URL_BACKUP = 'sqlite:////mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_main05_backup_20250603.db' # Adjust path
engine_backup = create_engine(DATABASE_URL_BACKUP)
inspector = inspect(engine_backup) # Used for schema introspection

# Define Base for database modeling (even if not creating, allows reflection)
Base = declarative_base() # This Base should ideally be the same you used for creating tables
# --- Define your models (necessary for relationships and type mapping) ---
# It's crucial to define your SQLAlchemy models as they were originally defined
# when the database schema was created, for consistent type handling and reflection.
class Artist(Base):
    __tablename__ = 'artists'
    id = Column(Integer, primary_key=True)
    artist_name = Column(String, unique=True, index=True)
    profile_url = Column(String)
    country = Column(String)
    level = Column(String)
    registration_date = Column(Integer)
    no_of_deviations = Column(Integer)
    no_of_favourites = Column(Integer)
    no_of_user_comments = Column(Integer)
    no_of_pageviews = Column(Integer)
    no_of_profile_comments = Column(Integer)
    is_artist = Column(Boolean)
    gender = Column(String)
    speciality = Column(String)
    no_of_images = Column(Integer)
    no_of_AI_images = Column(Integer) # Integer as per your original code
    ai_adopter = Column(Boolean)
    ai_adoption_first_time = Column(Integer)
    # Relationships for ArtistInteraction (if you used ArtistInteraction.interactions)
    interactions = relationship("ArtistInteraction", back_populates="artist_to")
    # Add other relationships if Artist is related to other tables you have (e.g., Watching)
    # watching = relationship("Watching", back_populates="artist") # If you have a Watching table


class ArtistInteraction(Base):
    __tablename__ = 'artist_interactions'
    id = Column(Integer, primary_key=True)
    artist_id_to = Column(Integer, ForeignKey('artists.id'), nullable=False)
    artist_name_to = Column(String, nullable=False)
    artist_id_from = Column(BigInteger, nullable=False) # BigInteger for global interactor IDs
    artist_name_from = Column(String, nullable=False)
    interaction_type = Column(String, nullable=False) # 'watcher', 'watching', 'friend' (tag)
    date = Column(BigInteger, nullable=True) # Unix timestamp in milliseconds

    artist_to = relationship("Artist", back_populates="interactions")

    # Add other relationships back to Artist if needed (if defining them on ArtistInteraction side)
    # For a simple edge list, back_populates from Artist is usually enough.

    __table_args__ = (
        UniqueConstraint('artist_id_to', 'artist_id_from', 'interaction_type', name='_interaction_edge_uc'),
    )

# --- Main Function to Summarize Database ---
def summarize_database(engine, inspector):
    """
    Connects to a database and provides a summary of all its tables,
    including schema details, row counts, and basic column statistics.
    """
    logger.info(f"--- Summarizing Database: {engine.url.database} ---")

    table_names = inspector.get_table_names()

    if not table_names:
        logger.warning("No tables found in the database.")
        return

    logger.info(f"Found {len(table_names)} tables: {', '.join(table_names)}")
    logger.info("=" * 60)

    for table_name in table_names:
        logger.info(f"\n--- TABLE: {table_name} ---")

        # 1. Table Schema
        columns = inspector.get_columns(table_name)
        print("  Columns:")
        for col in columns:
            pk_str = ' [PK]' if col['primary_key'] else ''
            nullable_str = ' [NULL]' if col['nullable'] else ''
            # Check for foreign keys
            fk_str = ' [FK]' if col.get('foreign_keys') and len(col['foreign_keys']) > 0 else ''
            print(f"    - {col['name']} ({col['type']}){pk_str}{fk_str}{nullable_str}")

        pk_constraint = inspector.get_pk_constraint(table_name)
        if pk_constraint and pk_constraint['constrained_columns']:
            print(f"  Primary Key: {', '.join(pk_constraint['constrained_columns'])}")

        foreign_keys = inspector.get_foreign_keys(table_name)
        if foreign_keys:
            print("  Foreign Keys:")
            for fk in foreign_keys:
                print(f"    - {', '.join(fk['constrained_columns'])} -> {fk['referred_table']}.{', '.join(fk['referred_columns'])}")
        
        unique_constraints = inspector.get_unique_constraints(table_name)
        if unique_constraints:
            print("  Unique Constraints:")
            for uc in unique_constraints:
                print(f"    - Name: {uc['name']}, Columns: {', '.join(uc['constrained_columns'])}")


        # 2. Total Rows
        # Reflect the table for querying
        metadata_obj = MetaData()
        current_table = Table(table_name, metadata_obj, autoload_with=engine)
        
        with engine.connect() as connection:
            row_count = connection.scalar(select(func.count()).select_from(current_table))
            print(f"\n  Total Rows: {row_count}")

            # 3. Basic Column Statistics (expensive for large tables, consider optimizing if performance is an issue)
            print(" Basic Statistics:")
            for col in columns:
                col_name = col['name']
                col_type = col['type']
                sqla_col = current_table.c[col_name] # Get the Column object from the reflected table

                try:
                    if isinstance(col_type, (Integer, BigInteger, Boolean)): # Include Boolean for min/max (0/1)
                        min_val = connection.scalar(select(func.min(sqla_col)).select_from(current_table))
                        max_val = connection.scalar(select(func.max(sqla_col)).select_from(current_table))
                        
                        # Handle average carefully, only if numeric
                        avg_val = None
                        if isinstance(col_type, (Integer, BigInteger)):
                            avg_val = connection.scalar(select(func.avg(sqla_col)).select_from(current_table))
                            print(f"    - {col_name} (Numeric): Min={min_val}, Max={max_val}, Avg={avg_val:.2f}" if avg_val is not None else f"    - {col_name} (Numeric): Min={min_val}, Max={max_val}, Avg=N/A")
                        else: # For Boolean, min/max are 0/1
                            print(f"    - {col_name} (Boolean): Min={min_val}, Max={max_val}")
                        
                        # Special handling for date columns stored as BigInteger
                        if col_name == 'last_visit' and (min_val is not None or max_val is not None):
                            try:
                                min_date = datetime.fromtimestamp(min_val / 1000, tz=pytz.utc).strftime('%Y-%m-%d %H:%M:%S %Z') if min_val is not None else "N/A"
                                max_date = datetime.fromtimestamp(max_val / 1000, tz=pytz.utc).strftime('%Y-%m-%d %H:%M:%S %Z') if max_val is not None else "N/A"
                                print(f"      (Date Range: {min_date} to {max_date})")
                            except (TypeError, ValueError, OSError): # Handle invalid timestamps (e.g., if column has non-timestamp integers)
                                print(f"      (Date Conversion Error for {col_name})")

                    elif isinstance(col_type, String):
                        distinct_count = connection.scalar(select(func.count(sqla_col.distinct())).select_from(current_table))
                        # Fetch some sample values if there are few distinct ones (optional)
                        # if distinct_count < 10 and distinct_count is not None:
                        #     sample_values = connection.scalars(select(sqla_col.distinct()).limit(5)).all()
                        #     print(f"      Sample: {sample_values}")
                        print(f"    - {col_name} (String): Distinct Count={distinct_count}")
                    else:
                        print(f"    - {col_name} (Other Type): Stats not implemented.")
                except Exception as e:
                    logger.warning(f"  Error getting stats for column {table_name}.{col_name}: {e}")
                    print(f"    - {col_name}: Error retrieving stats.")
        logger.info("\n" + "="*60 + "\n")

# --- Main Execution ---
if __name__ == "__main__":
    # You don't need to call Base.metadata.create_all(engine_backup) for a backup DB,
    # as you are only inspecting it. The models are defined for introspection.

    summarize_database(engine_backup, inspector)
    logger.info("Database summary complete.")

### Code to create a backup of the database

In [ ]:
import sqlite3
import time

source_db = '/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_main05.db'
backup_db = '/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_main05_backup_20250603.db'

print(f"Attempting robust Python backup of {source_db} to {backup_db}...")

try:
    # 1. Connect to the source database
    source_conn = sqlite3.connect(source_db)
    
    # 2. Connect to the (non-existent or empty) destination database
    dest_conn = sqlite3.connect(backup_db)
    
    # 3. Perform the backup
    # pages=1 for faster feedback, progress is printed per page
    # progress callback can be used for larger databases
    with source_conn: # Use context manager for source connection
        source_conn.backup(dest_conn, pages=1) # backup (destination, pages_to_copy_at_once)
    
    # 4. Close connections
    dest_conn.close()
    source_conn.close()
    
    print(f"Successfully created robust Python backup: {backup_db}")

except Exception as e:
    print(f"Error during robust Python backup: {e}")
    # Ensure connections are closed even if an error occurs
    if 'source_conn' in locals() and source_conn:
        source_conn.close()
    if 'dest_conn' in locals() and dest_conn:
        dest_conn.close()

### Loading watchers and friends incrementally to the database

In [ ]:
def load_watchers_incrementally(session, csv_path):
    """Loads watcher data incrementally from a CSV."""
    try:
        for chunk in pd.read_csv(csv_path, chunksize=1000, on_bad_lines='skip'):
            for index, row in chunk.iterrows():
                # Adjust column name to match your CSV: 'Deviant' for the artist being watched
                artist = session.query(Artist).filter_by(artist_name=row['Deviant']).first()
                if artist:
                    # Adjust column name to match your CSV if needed: 'Watchers name' for the watcher's name
                    watcher_name_from_csv = row['Watchers name']

                    # Check if record already exists (based on artist_id and watcher_name)
                    existing_record = session.query(Watcher).filter_by(
                        artist_id=artist.id,
                        watcher_name=watcher_name_from_csv,
                    ).first()

                    if not existing_record:
                        new_watcher = Watcher(
                            artist_id=artist.id,
                            watcher_name=watcher_name_from_csv,
                        )
                        session.add(new_watcher)
            session.commit() # Commit after each chunk
            print(f"Processed a chunk of watchers from {csv_path}")
    except Exception as e:
        session.rollback()
        print(f"Error loading watchers: {e}")

In [ ]:
def load_friends_incrementally(session, csv_path):
    """Loads friend data incrementally from a CSV."""
    try:
        for chunk in pd.read_csv(csv_path, chunksize=1000, on_bad_lines='skip'):
            for index, row in chunk.iterrows():
                # Adjust column name to match your CSV: 'Deviant' for one side of the friendship
                artist = session.query(Artist).filter_by(artist_name=row['Deviant']).first()
                if artist:
                    # Adjust column name to match your CSV if needed: 'Friend Name' for the friend's name
                    friend_name_from_csv = row['Friend Name'] # <-- Adjust this if the column name is different

                    # Check if record already exists (based on artist_id and friend_name)
                    existing_record = session.query(Friend).filter_by(
                        artist_id=artist.id,
                        friend_name=friend_name_from_csv,
                    ).first()

                    if not existing_record:
                        new_friend = Friend(
                            artist_id=artist.id,
                            friend_name=friend_name_from_csv,
                        )
                        session.add(new_friend)
            session.commit() # Commit after each chunk
            print(f"Processed a chunk of friends from {csv_path}")
    except Exception as e:
        session.rollback()
        print(f"Error loading friends: {e}")


### Pre-processing watching data and loading it incrementally to the DB

#### 1. Function for pre-processing watching data

In [ ]:
import pandas as pd
import re # Import regex module
import ast # Keep ast for potential fallback or other parsing needs, though not used directly for splitting here

# Assume Artist and Watching classes are defined elsewhere

def process_watching_data(csv_path):
    """
    Reads scraping CSV, extracts watching relationships, and returns a list of dictionaries.

    Args:
        csv_path: Path to the scraping CSV file.

    Returns:
        A list of dictionaries, where each dictionary represents a single
        watching relationship: [{'username': 'ArtistA', 'watching_name': 'ArtistB'}, ...]
    """
    processed_watchings = []
    try:
        # Adjust chunksize as needed
        for chunk in pd.read_csv(csv_path, chunksize=1000, on_bad_lines='skip'):
            for index, row in chunk.iterrows():
                artist_name_who_is_watching = row['username'] # Column with the watching artist's username

                # Check if 'Watching' column is not null or empty
                if pd.notnull(row['Watching']) and str(row['Watching']).strip():
                    watching_string = str(row['Watching']).strip()

                    # Use regex to remove "Watching XXX Deviants" at the beginning
                    # This regex looks for "Watching", followed by one or more spaces,
                    # followed by one or more digits (\d+), followed by one or more spaces,
                    # followed by "Deviants" (case-insensitive), followed by optional spaces.
                    # Use regex to remove "Watching XXX Deviants" at the beginning
                    watching_string_cleaned = re.sub(r'^Watching\s+\d+\s+Deviants\s*', '', watching_string, flags=re.IGNORECASE)

                    watched_usernames = watching_string_cleaned.split()

                    for watching_name in watched_usernames: # This is where the error occurs
                        # Check if the watching_name is not empty after splitting
                        if watching_name.strip():
                            processed_watchings.append({
                                'username': artist_name_who_is_watching,
                                'watching_name': watching_name.strip()
                            })
                # Handle the case where 'Watching' column is null or empty after stripping
                elif pd.isnull(row['Watching']) or not str(row['Watching']).strip():
                    # You might want to log this or take other action if needed
                    print(f"Info: 'Watching' column is empty or null for artist {artist_name_who_is_watching}. Skipping.")


    except FileNotFoundError:
        print(f"Error: CSV file not found at {csv_path}")
    except Exception as e:
        print(f"An error occurred during watchings data processing: {e}")

    return processed_watchings

#### 2. Load the watching data from the csv and use the above function to pre-process it

In [ ]:
scraping_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_snwballScraped_fin01.csv.gz" # Your scraping CSV with 'Watching' column

#     # Step 1: Process the watching data and print (optional for verification)
#     print("Processing watching data from CSV...")
processed_watching_list = process_watching_data(scraping_csv)

#     # You can print a sample of processed_watching_list here to verify
print("Sample of processed data:", processed_watching_list[:10])


#### 3. Loading the watching data incrementally to the DB

In [ ]:
def load_watchings_incrementally(session, csv_path):
    """Loads watching data incrementally from a CSV, processing the watching list."""
    try:
        for chunk in pd.read_csv(csv_path, chunksize=1000, on_bad_lines='skip'):
            for index, row in chunk.iterrows():
                artist = session.query(Artist).filter_by(artist_name=row['username']).first() # Adjust column name if needed
                if artist and not pd.isnull(row['Watching']): # Check if artist exists and 'Watching' column is not null
                    try:
                        watching_list = ast.literal_eval(row['Watching']) # Safely evaluate the string representation of the list
                        for watching_name in watching_list:
                            # Find the ID of the watching artist (assuming watching_name corresponds to an artist_name in the Artist table)
                            watching_artist = session.query(Artist).filter_by(artist_name=watching_name).first()
                            watching_id = watching_artist.id if watching_artist else None

                            # Check if record already exists
                            existing_record = session.query(Watching).filter_by(
                                artist_id=artist.id,
                                watching_name=watching_name,
                                watching_id=watching_id
                            ).first()

                            if not existing_record:
                                new_watching = Watching(
                                    artist_id=artist.id,
                                    watching_name=watching_name,
                                    watching_id=watching_id # Store the ID if found
                                )
                                session.add(new_watching)
                    except (SyntaxError, ValueError) as e:
                        print(f"Error parsing 'Watching' list for artist {row['username']}: {e}")
            session.commit() # Commit after each chunk
            print(f"Processed a chunk of watchings from {csv_path}")
    except Exception as e:
        session.rollback()
        print(f"Error loading watchings: {e}")

In [ ]:
# --- Main Execution ---
if __name__ == "__main__":
    session = Session()

    # Replace with your actual CSV file paths
    watchers_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_wtchrsSnwball_fin1.csv.gz" # watchers CSV
    friends_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_friendsSnwball_fin.csv.gz"   # Your friends CSV
    #scraping_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_friendsSnwball_fin.csv.gz" # Your scraping CSV with 'Watching' column

    # Load data into the new tables
    print("Loading watchers data...")
    load_watchers_incrementally(session, watchers_csv)

    print("\nLoading friends data...")
    load_friends_incrementally(session, friends_csv)

    #print("\nLoading watchings data...")
    #load_watchings_incrementally(session, scraping_csv)

    #print("\nPopulating artist interactions table...")
    #populate_artist_interactions(session)

    session.close()
    print("\nDatabase loading complete.")

### Loading interaction data incrementally

In [ ]:
def populate_artist_interactions(session):
    """Populates the artist_interactions table from the other tables."""
    try:
        # Process Watchers
        watchers = session.query(Watcher).all()
        for watcher in watchers:
             # Check if record already exists
            existing_interaction = session.query(ArtistInteraction).filter_by(
                artist_id_to=watcher.artist_id,
                artist_id_from=watcher.watcher_id,
                interaction_type='watcher'
            ).first()
            if not existing_interaction:
                # Find the name of the artist being watched
                artist_to = session.query(Artist).get(watcher.artist_id)
                artist_name_to = artist_to.artist_name if artist_to else None

                new_interaction = ArtistInteraction(
                    artist_id_to=watcher.artist_id,
                    artist_name_to=artist_name_to,
                    artist_id_from=watcher.watcher_id,
                    artist_name_from=watcher.watcher_name,
                    interaction_type='watcher',
                    date=None # Leave date empty for now
                )
                session.add(new_interaction)
        session.commit()
        print("Populated interactions from Watchers.")

        # Process Friends
        friends = session.query(Friend).all()
        for friend in friends:
            # Check if record already exists (consider both directions if friendship is mutual)
            existing_interaction = session.query(ArtistInteraction).filter(
                 ((ArtistInteraction.artist_id_to == friend.artist_id) & (ArtistInteraction.artist_id_from == friend.friend_id)) |
                 ((ArtistInteraction.artist_id_to == friend.friend_id) & (ArtistInteraction.artist_id_from == friend.artist_id))
             ).filter_by(interaction_type='friend').first()

            if not existing_interaction:
                # Find the names
                artist_to = session.query(Artist).get(friend.artist_id)
                artist_name_to = artist_to.artist_name if artist_to else None
                artist_from = session.query(Artist).get(friend.friend_id) # Assuming friend_id is an artist ID
                artist_name_from = artist_from.artist_name if artist_from else None


                new_interaction = ArtistInteraction(
                    artist_id_to=friend.artist_id,
                    artist_name_to=artist_name_to,
                    artist_id_from=friend.friend_id,
                    artist_name_from=artist_name_from,
                    interaction_type='friend',
                    date=None # Leave date empty for now
                )
                session.add(new_interaction)
        session.commit()
        print("Populated interactions from Friends.")

        # Process Watchings
        watchings = session.query(Watching).all()
        for watching in watchings:
            # Check if record already exists
            existing_interaction = session.query(ArtistInteraction).filter_by(
                artist_id_to=watching.artist_id,
                artist_id_from=watching.watching_id,
                interaction_type='watching'
            ).first()
            if not existing_interaction:
                 # Find the names
                artist_to = session.query(Artist).get(watching.artist_id)
                artist_name_to = artist_to.artist_name if artist_to else None
                artist_from = session.query(Artist).get(watching.watching_id) # Assuming watching_id is an artist ID
                artist_name_from = artist_from.artist_name if artist_from else None

                new_interaction = ArtistInteraction(
                    artist_id_to=watching.artist_id,
                    artist_name_to=artist_name_to,
                    artist_id_from=watching.watching_id,
                    artist_name_from=artist_name_from,
                    interaction_type='watching',
                    date=None # Leave date empty for now
                )
                session.add(new_interaction)
        session.commit()
        print("Populated interactions from Watchings.")

    except Exception as e:
        session.rollback()
        print(f"Error populating artist interactions: {e}")


In [ ]:
# Database connection - done before

# Create tables (if they don't exist)
# This will create only the new tables defined above if Base.metadata.create_all(engine) was already called
Base.metadata.create_all(engine)

# Create a session done before

def load_watchers_incrementally(session, csv_path):
    """Loads watcher data incrementally from a CSV."""
    try:
        for chunk in pd.read_csv(csv_path, chunksize=1000, on_bad_lines='skip'):
            for index, row in chunk.iterrows():
                artist = session.query(Artist).filter_by(artist_name=row['Author_Name']).first() # Adjust column name if needed
                if artist:
                    # Check if record already exists (based on artist_id and watcher_name/id)
                    existing_record = session.query(Watcher).filter_by(
                        artist_id=artist.id,
                        watcher_name=row['Watcher_Name'], # Adjust column name if needed
                        watcher_id=row['Watcher_Id'] # Adjust column name if needed
                    ).first()

                    if not existing_record:
                        new_watcher = Watcher(
                            artist_id=artist.id,
                            watcher_name=row['Watcher_Name'], # Adjust column name if needed
                            watcher_id=row['Watcher_Id'] # Adjust column name if needed
                        )
                        session.add(new_watcher)
            session.commit() # Commit after each chunk
            print(f"Processed a chunk of watchers from {csv_path}")
    except Exception as e:
        session.rollback()
        print(f"Error loading watchers: {e}")

def load_friends_incrementally(session, csv_path):
    """Loads friend data incrementally from a CSV."""
    try:
        for chunk in pd.read_csv(csv_path, chunksize=1000, on_bad_lines='skip'):
            for index, row in chunk.iterrows():
                artist = session.query(Artist).filter_by(artist_name=row['Author_Name']).first() # Adjust column name if needed
                if artist:
                    # Check if record already exists
                    existing_record = session.query(Friend).filter_by(
                        artist_id=artist.id,
                        friend_name=row['Friend_Name'], # Adjust column name if needed
                        friend_id=row['Friend_Id'] # Adjust column name if needed
                    ).first()

                    if not existing_record:
                        new_friend = Friend(
                            artist_id=artist.id,
                            friend_name=row['Friend_Name'], # Adjust column name if needed
                            friend_id=row['Friend_Id'] # Adjust column name if needed
                        )
                        session.add(new_friend)
            session.commit() # Commit after each chunk
            print(f"Processed a chunk of friends from {csv_path}")
    except Exception as e:
        session.rollback()
        print(f"Error loading friends: {e}")

def load_watchings_incrementally(session, csv_path):
    """Loads watching data incrementally from a CSV, processing the watching list."""
    try:
        for chunk in pd.read_csv(csv_path, chunksize=1000, on_bad_lines='skip'):
            for index, row in chunk.iterrows():
                artist = session.query(Artist).filter_by(artist_name=row['username']).first() # Adjust column name if needed
                if artist and not pd.isnull(row['Watching']): # Check if artist exists and 'Watching' column is not null
                    try:
                        watching_list = ast.literal_eval(row['Watching']) # Safely evaluate the string representation of the list
                        for watching_name in watching_list:
                            # Find the ID of the watching artist (assuming watching_name corresponds to an artist_name in the Artist table)
                            watching_artist = session.query(Artist).filter_by(artist_name=watching_name).first()
                            watching_id = watching_artist.id if watching_artist else None

                            # Check if record already exists
                            existing_record = session.query(Watching).filter_by(
                                artist_id=artist.id,
                                watching_name=watching_name,
                                watching_id=watching_id
                            ).first()

                            if not existing_record:
                                new_watching = Watching(
                                    artist_id=artist.id,
                                    watching_name=watching_name,
                                    watching_id=watching_id # Store the ID if found
                                )
                                session.add(new_watching)
                    except (SyntaxError, ValueError) as e:
                        print(f"Error parsing 'Watching' list for artist {row['username']}: {e}")
            session.commit() # Commit after each chunk
            print(f"Processed a chunk of watchings from {csv_path}")
    except Exception as e:
        session.rollback()
        print(f"Error loading watchings: {e}")

def populate_artist_interactions(session):
    """Populates the artist_interactions table from the other tables."""
    try:
        # Process Watchers
        watchers = session.query(Watcher).all()
        for watcher in watchers:
             # Check if record already exists
            existing_interaction = session.query(ArtistInteraction).filter_by(
                artist_id_to=watcher.artist_id,
                artist_id_from=watcher.watcher_id,
                interaction_type='watcher'
            ).first()
            if not existing_interaction:
                # Find the name of the artist being watched
                artist_to = session.query(Artist).get(watcher.artist_id)
                artist_name_to = artist_to.artist_name if artist_to else None

                new_interaction = ArtistInteraction(
                    artist_id_to=watcher.artist_id,
                    artist_name_to=artist_name_to,
                    artist_id_from=watcher.watcher_id,
                    artist_name_from=watcher.watcher_name,
                    interaction_type='watcher',
                    date=None # Leave date empty for now
                )
                session.add(new_interaction)
        session.commit()
        print("Populated interactions from Watchers.")

        # Process Friends
        friends = session.query(Friend).all()
        for friend in friends:
            # Check if record already exists (consider both directions if friendship is mutual)
            existing_interaction = session.query(ArtistInteraction).filter(
                 ((ArtistInteraction.artist_id_to == friend.artist_id) & (ArtistInteraction.artist_id_from == friend.friend_id)) |
                 ((ArtistInteraction.artist_id_to == friend.friend_id) & (ArtistInteraction.artist_id_from == friend.artist_id))
             ).filter_by(interaction_type='friend').first()

            if not existing_interaction:
                # Find the names
                artist_to = session.query(Artist).get(friend.artist_id)
                artist_name_to = artist_to.artist_name if artist_to else None
                artist_from = session.query(Artist).get(friend.friend_id) # Assuming friend_id is an artist ID
                artist_name_from = artist_from.artist_name if artist_from else None


                new_interaction = ArtistInteraction(
                    artist_id_to=friend.artist_id,
                    artist_name_to=artist_name_to,
                    artist_id_from=friend.friend_id,
                    artist_name_from=artist_name_from,
                    interaction_type='friend',
                    date=None # Leave date empty for now
                )
                session.add(new_interaction)
        session.commit()
        print("Populated interactions from Friends.")

        # Process Watchings
        watchings = session.query(Watching).all()
        for watching in watchings:
            # Check if record already exists
            existing_interaction = session.query(ArtistInteraction).filter_by(
                artist_id_to=watching.artist_id,
                artist_id_from=watching.watching_id,
                interaction_type='watching'
            ).first()
            if not existing_interaction:
                 # Find the names
                artist_to = session.query(Artist).get(watching.artist_id)
                artist_name_to = artist_to.artist_name if artist_to else None
                artist_from = session.query(Artist).get(watching.watching_id) # Assuming watching_id is an artist ID
                artist_name_from = artist_from.artist_name if artist_from else None

                new_interaction = ArtistInteraction(
                    artist_id_to=watching.artist_id,
                    artist_name_to=artist_name_to,
                    artist_id_from=watching.watching_id,
                    artist_name_from=artist_name_from,
                    interaction_type='watching',
                    date=None # Leave date empty for now
                )
                session.add(new_interaction)
        session.commit()
        print("Populated interactions from Watchings.")

    except Exception as e:
        session.rollback()
        print(f"Error populating artist interactions: {e}")

# --- Main Execution ---
if __name__ == "__main__":
    session = Session()

    # Replace with your actual CSV file paths
    watchers_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_wtchrsSnwball_fin1.csv.gz" # watchers CSV
    friends_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_friendsSnwball_fin.csv.gz"   # Your friends CSV
    #scraping_csv = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_friendsSnwball_fin.csv.gz" # Your scraping CSV with 'Watching' column

    # Load data into the new tables
    print("Loading watchers data...")
    load_watchers_incrementally(session, watchers_csv)

    print("\nLoading friends data...")
    load_friends_incrementally(session, friends_csv)

    print("\nLoading watchings data...")
    load_watchings_incrementally(session, scraping_csv)

    print("\nPopulating artist interactions table...")
    populate_artist_interactions(session)

    session.close()
    print("\nDatabase loading complete.")

# Tomorrow start here

In [ ]:
session.close()

In [ ]:
from sqlalchemy import create_engine, MetaData, Table, select

engine = create_engine(DATABASE_URL)
metadata = MetaData()

table_name = "imgs_tags"  # Replace with the actual table name
table = Table(table_name, metadata, autoload_with=engine)

# Create a session
#Session = sessionmaker(bind=engine)
#session = Session()

# Execute the query and fetch all data
results = session.query(table).limit(10).all()
total_rows = session.query(func.count(imgs_dscrpt.id)).scalar()
if total_rows == 0:
    print("imgs_dscrpt table is empty.")

# Print the data
for row in results:
    print(row)  # Prints each row as a tuple

# Alternatively, to print specific columns:
for row in results:
    print(row.id, row.artist_id, row.artist_name, row.description) # Replace column1, column2 with actual column names

In [ ]:
#function to clean the description in metadata for description table
def clean_description(description):
    if pd.isnull(description) or description is None:  # Check for NaN or None
        return ""  # Return empty string for NaN values
    elif isinstance(description, str):  # Check if it's a string
        description = re.sub(r"<br\s*/?>", "\n", description)
        description = re.sub(r"<[^>]+>", "", description)
        description = re.sub(r"[^a-zA-Z0-9 ]", "", description)
        return description
    else:
        #print(f"Unexpected data type for description: {type(description)}")  # Optional: print for debugging
        try: 
            return str(description) # Attempt to convert to string, but still remove HTML tags
        except Exception as e:
            print(f"Error converting description to string: {e}. Returning an empty string instead.")
            return "" 

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
from sqlalchemy.exc import SQLAlchemyError


def save_imgs_dscrpt_incrementally(session, image_dscrpt_data_path):
    """Saves imgs_dscrpt data incrementally, row by row."""
    engine = session.get_bind()  # Get the engine from the session
    total_rows_saved = 0

    try:
        for chunk in pd.read_csv(image_dscrpt_data_path, chunksize=1000, on_bad_lines='skip'):  # Adjust chunksize as needed
            for index, row in chunk.iterrows():
                artist = session.query(Artist).filter_by(artist_name=row['Author_Name']).first()
                if artist:
                    artist_id = artist.id
                    artist_name = artist.artist_name
                    cleaned_description = clean_description(row['Devtn_Descp'])

                    # Check if record already exists 
                    existing_record = session.query(imgs_dscrpt).filter_by(id=row['Devtn_Id']).first()
                    if existing_record:
                        print(f"Skipping duplicate imgs_dscrpt record with id: {row['Devtn_Id']}")
                        continue 

                    new_image_dscrpt = imgs_dscrpt(id=row['Devtn_Id'], artist_id=artist_id,
                                                   artist_name=artist_name, description=cleaned_description)
                    session.add(new_image_dscrpt)

                    try:
                        session.commit()  # Commit after each row
                        total_rows_saved += 1
                        print(f"Saved imgs_dscrpt record: {new_image_dscrpt.id}, Total saved: {total_rows_saved}")
                    except SQLAlchemyError as e:
                        session.rollback()
                        print(f"Error saving record: {e}")
                        # Handle the error (e.g., log, retry, skip)

                else:
                    print(f"Artist not found for image with ID: {row['Devtn_Id']}")

    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        session.close()

# ... (In your main function or script) ...

image_dscrpt_data_path = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_metaDataSnwBall/uniqueDev_metaData_SnwBall_02.csv.gz"
# Assuming 'session' is your SQLAlchemy session object
save_imgs_dscrpt_incrementally(session, image_dscrpt_data_path)

In [ ]:
#testing imgs description table
def check_imgs_dscrpt_data(session):
    """Checks the first, middle, and last ten rows of the imgs_dscrpt table."""

    total_rows = session.query(func.count(imgs_dscrpt.id)).scalar()

    if total_rows == 0:
        print("imgs_dscrpt table is empty.")
        return

    print("First 10 rows:")
    first_ten = session.query(imgs_dscrpt).limit(10).all()
    for row in first_ten:
        print(row.id, row.artist_id, row.artist_name, row.description)

    print("\nMiddle 10 rows:")
    middle_row_num = total_rows // 2
    middle_ten = session.query(imgs_dscrpt).offset(middle_row_num - 5).limit(10).all()
    for row in middle_ten:
        print(row.id, row.artist_id, row.artist_name, row.description)

    print("\nLast 10 rows:")
    last_ten = session.query(imgs_dscrpt).order_by(desc(imgs_dscrpt.id)).limit(10).all()
    for row in last_ten:
        print(row.id, row.artist_id, row.artist_name, row.description)

check_imgs_dscrpt_data(session)


In [ ]:
# Function to process and insert tags in the db
def process_tags(tags_string, image_id, artist_id, artist_name, session):
    try:
        tags_list = ast.literal_eval(tags_string)
        for tag in tags_list:
            if pd.isnull(tag) or tag is None:
                continue
            else:
                new_image_tag = imgs_tags(
                    image_id=image_id,
                    artist_id=artist_id,
                    artist_name=artist_name,
                    tags=tag.strip()
                )
                session.add(new_image_tag)
    except (SyntaxError, ValueError):
        print(f"Error parsing tags for image ID: {image_id}")


In [ ]:
# Function to load data (combining all loading logic)
def load_data_to_database():
    engine = create_engine(DATABASE_URL)
    Session = sessionmaker(bind=engine)
    session = Session()
    image_dscrpt_data = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_metaDataSnwBall/uniqueDev_metaData_SnwBall_02.csv.gz")
    for index, row in image_dscrpt_data.iterrows():
        artist = session.query(Artist).filter_by(artist_name=row['Author_Name']).first()
        if artist:
            artist_id = artist.id
            artist_name = artist.artist_name
            cleaned_description = clean_description(row['Devtn_Descp'])
    
            # Create and add imgs_dscrpt object
            new_image_dscrpt = imgs_dscrpt(id=row['Devtn_Id'], artist_id=artist_id,
                                           artist_name=artist_name, description=cleaned_description)
            session.add(new_image_dscrpt)
            session.flush()  # Flush to get image_id
    
            # Get the image_id from the flushed object
            image_id = new_image_dscrpt.id
    
            # Now you can use image_id in process_tags
            process_tags(row['tag_name'], image_id, artist_id, artist_name, session)
        else:
            print(f"Artist not found for image with ID: {row['Devtn_Id']}")

    # Load image date data
    image_date_data = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_gallData_4_5_6/uniqueDev_gall_SnwBall03_6.2.csv.gz")
    for index, row in image_date_data.iterrows():
        artist = session.query(Artist).filter_by(artist_name=row['Author_name']).first()
        if artist:
            artist_id = artist.id
            
            # Check if imgs_date record already exists
            existing_date_record = session.query(imgs_date).filter_by(id=row['Deviation_id']).first()
            if existing_date_record:
                print(f"Skipping duplicate imgs_date record with id: {row['Deviation_id']}")
                continue  # Skip to the next row
    
            new_image_date = imgs_date(id=row['Deviation_id'], artist_id=artist_id, date=row['Published_on'])
            session.add(new_image_date)
        else:
            print(f"Artist not found for image date with ID: {row['Deviation_id']}")
     
    try:
        # Reset the tag_id sequence using SQLAlchemy
        with engine.connect() as conn:  # Use a separate connection for the sequence update
            max_tag_id = session.query(func.max(imgs_tags.tag_id)).scalar() or 0  # Get max tag_id
            conn.execute(text("UPDATE sqlite_sequence SET seq = :next_val WHERE name = 'imgs_tags'"), 
                         {'next_val': max_tag_id + 1})
        session.commit()  # Commit all changes
    except Exception as e:
        session.rollback()  # Rollback changes in case of an error
        print(f"Error loading data: {e}")
    finally:
        session.close()  # Always close the session


load_data_to_database()

In [ ]:
# ... (Function to clean the artists profile data before sending it to the db and to generate an id for them) ...
def clean_and_convert_data(data, column_types):
    """Cleans and converts data based on column types."""
    for column, data_type in column_types.items():
        if data_type == Boolean:  # Handle boolean columns
            data[column] = data[column].fillna(False)  # Replace NaN with False (or True if preferred)
            data[column] = data[column].astype(bool)  # Convert to boolean type
        elif data_type in (Integer, BigInteger):  # Handle integer columns
            data[column] = data[column].fillna(0)  # Replace NaN with 0 (or another appropriate default)
            data[column] = data[column].astype(int)  # Convert to integer type
        elif data_type == String:  # Handle string columns
            data[column] = data[column].fillna('')  # Replace NaN with empty string
    return data

# Load user data from CSV
artist_data = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_profileSnwball_fin1_clean.csv.gz", on_bad_lines = "skip")  # Replace "your_users_csv.csv"


# Define column types for cleaning and conversion
column_types = {
    'profil_url': String,
    'country_name': String,
    'user_is_artist': Boolean,
    'user_deviations': Integer,
    'user_favourites': Integer,
    'user_comments': Integer,
    'profile_pageviews': Integer,
    'profile_comments': Integer,
    'user_deviations': Integer,
    'specialty': String,
    'level': String 
    
    # ... (add other columns and their types) ...
}

# Clean and convert user data
artist_data = clean_and_convert_data(artist_data, column_types)

def generate_artist_no(session):
    last_artist_no = session.query(func.max(Artist.id)).scalar()
    if last_artist_no is None:
        return 1
    else:
        return last_artist_no + 1



# Insert user data, generating artist_no
for index, row in artist_data.iterrows():
    existing_user = session.query(Artist).filter_by(artist_name=row['user']).first() # Assuming 'username_csv_column' is the username column in your CSV
    if existing_user is None:  # Check if user already exists to avoid duplicates
        artist_no = generate_artist_no(session)
        #country level registration_date artists_deviations artists_favourites artists_comments profile_pageviews
        # profile_comments is_artist gender speciality
        new_artist = Artist(id=artist_no, artist_name=row['user'], profile_url=row['profil_url'], country = row['country_name'], 
                         level = row['level'], no_of_deviations= row['user_deviations'], no_of_favourites= row['user_favourites'], 
                          no_of_user_comments = row['user_comments'], no_of_pageviews = row['profile_pageviews'], 
                            no_of_profile_comments = row['profile_comments'], is_artist = row['user_is_artist'], speciality = row['specialty'])  # Map CSV columns to User attributes
        session.add(new_artist)

session.commit()  # Save changes to the database
session.close()

In [ ]:
#To recreate tables

def recreate_imgs_tags_table():
    session = Session()
    try:
        # 1. Drop the existing imgs_tags table (if it exists)
        imgs_tags.__table__.drop(engine)  
        print("Existing imgs_tags table dropped.")
    except OperationalError:
        print("imgs_tags table does not exist, skipping drop.")

    # 2. Create the new imgs_tags table
    Base.metadata.create_all(engine)
    print("New imgs_tags table created.")

    session.close()

# Call the function to recreate the table
recreate_imgs_tags_table()

In [ ]:


# ... (Your other imports and functions) ...

def add_unique_constraint_to_imgs_date(engine):
    """Adds a unique constraint to the 'id' column of the imgs_date table in SQLite."""
    with engine.connect() as conn:
        # Check if the constraint already exists (using PRAGMA)
        constraint_name = 'unique_imgs_date_id'
        result = conn.execute(text(f"PRAGMA index_list(imgs_date)"))  
        existing_constraints = [row[1] for row in result]  
        if constraint_name in existing_constraints:
            print(f"Unique constraint '{constraint_name}' already exists on imgs_date.id")
            return

        try:
            # Create temporary table with unique constraint
            conn.execute(text(f"CREATE TABLE imgs_date_temp (id TEXT PRIMARY KEY, artist_id INTEGER, artist_name TEXT, date INTEGER)"))
            
            # Copy data from original table
            conn.execute(text(f"INSERT INTO imgs_date_temp SELECT * FROM imgs_date"))

            # Drop original table
            conn.execute(text(f"DROP TABLE imgs_date"))

            # Rename temporary table
            conn.execute(text(f"ALTER TABLE imgs_date_temp RENAME TO imgs_date"))
            print(f"Unique constraint added to imgs_date.id")  

        except Exception as e:
            print(f"Error adding unique constraint: {e}")

# Call the function after creating the engine
engine = create_engine(DATABASE_URL)
add_unique_constraint_to_imgs_date(engine)

In [ ]:
# Old not working Function to load data (combining all loading logic)
def load_data_to_database():
    engine = create_engine(DATABASE_URL)
    Session = sessionmaker(bind=engine)
    session = Session()
    image_dscrpt_data = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_metaDataSnwBall/uniqueDev_metaData_SnwBall_02.csv.gz")
    for index, row in image_dscrpt_data.iterrows():
        artist = session.query(Artist).filter_by(artist_name=row['Author_Name']).first()
        if artist:
            artist_id = artist.id
            artist_name = artist.artist_name
            cleaned_description = clean_description(row['Devtn_Descp'])
            new_image_dscrpt = imgs_dscrpt(id=row['Devtn_Id'], artist_id=artist_id, 
                                           artist_name = artist_name, description=cleaned_description)
            session.add(new_image_dscrpt)
            session.flush()  # Flush the session to get the image_id
        else:
            print(f"User not found for image description with ID: {row['Devtn_Id']}")

        # Load image tags data
        #image_tags_data = pd.read_csv("your_image_tags_csv.csv")
    for index, row in image_dscrpt_data.iterrows():
        artist = session.query(Artist).filter_by(artist_name=row['Author_Name']).first()
        dvtn = session.query(imgs_dscrpt).filter_by(artist_name=row['Author_Name']).first()
        if artist:
            artist_id = artist.id
            artist_name = artist.artist_name
            process_tags(id=,row['tag_name'], , artist_id=artist_id, 
                         artist_name = artist_name, session)
            #new_image_tags = imgs_tags(id=row['id_csv_column'], user_id=artist_no, description=cleaned_description)  # Assuming you have artist_no from previous step

        else:
            print(f"User not found for image tags with ID: {row['Devtn_Id']}")

        # Load image date data
    image_date_data = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_gallData_4_5_6/uniqueDev_gall_SnwBall03_6.2.csv.gz")
    for index, row in image_date_data.iterrows():
        artist = session.query(Artist).filter_by(artist_name=row['Author_name']).first()  # Assumed 'artist_name_csv_column' in this CSV as well
        if artist:
           artist_id = artist.id
           artist_name = artist.artist_name 
           new_image_date = imgs_date(id=row['Deviation_id'], artist_id=artist_id, artist_name=artist_name,
                                      date=row['Published_on'])
           session.add(new_image_date)
        else:
           print(f"User not found for image date with ID: {row['Deviation_id']}")

    try:
        session.commit()  # Commit all changes
    except Exception as e:
        session.rollback()  # Rollback changes in case of an error
        print(f"Error loading data: {e}")
    finally:
        session.close()  # Always close the session

load_data_to_database()

In [ ]:
#Tags to db
import pandas as pd
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, BigInteger, Boolean, func, MetaData, Table
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
import re, ast
import numpy as np
from sqlalchemy import UniqueConstraint, inspect
from sqlalchemy.exc import SQLAlchemyError
from sqlalchemy import text


def process_tags(tags_string, image_id, artist_id, artist_name, session):
    try:
        tags_list = ast.literal_eval(tags_string)
        for tag in tags_list:
            if pd.isnull(tag) or tag is None:
                continue
            else:
                # tag_id is auto-incremented, no need to provide a value here
                new_image_tag = imgs_tags(
                    image_id=image_id,
                    artist_id=artist_id,
                    artist_name=artist_name,
                    tags=tag.strip()  # Store each tag individually
                )
                session.add(new_image_tag)
    except (SyntaxError, ValueError):
        print(f"Error parsing tags for image ID: {image_id}")


#tag_id INTEGER image_id VARCHAR artist_id INTEGER artist_name VARCHAR tags VARCHAR
def save_imgs_tags_incrementally(session, image_tags_data_path):
    """Saves imgs_dscrpt data incrementally, row by row."""
    engine = session.get_bind()  # Get the engine from the session
    total_rows_saved = 0

    try:
        for chunk in pd.read_csv(image_tags_data_path, chunksize=1000, on_bad_lines='skip'):
            for index, row in chunk.iterrows():
                artist = session.query(Artist).filter_by(artist_name=row['Author_Name']).first()
                if artist:
                    artist_id = artist.id
                    artist_name = artist.artist_name

                    # Call process_tags to handle tag creation and insertion
                    process_tags(row['tag_name'], row['Devtn_Id'], artist_id, artist_name, session) 

                    # Check if record already exists (using image_id)
                    existing_record = session.query(imgs_tags).filter_by(image_id=row['Devtn_Id']).first()
                    if existing_record:
                        print(f"Skipping duplicate imgs_tags record with id: {row['Devtn_Id']}")
                        continue 

                    # --- The new_image_tags object is created within process_tags ---
                    # new_image_tags = imgs_tags(id=)  <-- This line is removed

                    # --- session.add(new_image_dscrpt) is incorrect; removed ---

                    try:
                        session.commit()  # Commit after each row
                        total_rows_saved += 1
                        # --- This print statement is adjusted to reflect imgs_tags ---
                        # print(f"Saved imgs_dscrpt record: {new_image_dscrpt.id}, Total saved: {total_rows_saved}")
                        print(f"Saved imgs_tags records for image: {row['Devtn_Id']}, Total saved: {total_rows_saved}") 
                    except SQLAlchemyError as e:
                        session.rollback()
                        print(f"Error saving record: {e}")
                        # Handle the error (e.g., log, retry, skip)

                else:
                    print(f"Artist not found for image with ID: {row['Devtn_Id']}")
        # Place the print statement BEFORE the exceptions:
        print(f"All data from CSV '{image_tags_data_path}' saved to table 'imgs_tags'")  # Changed to imgs_tags
    
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        session.close()

# ... (In your main function or script) ...

if __name__ == "__main__":  # This block will only execute when the script is run directly
    DATABASE_URL =  'sqlite:////mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_main05.db'
    engine = create_engine(DATABASE_URL)  # Create a SQLite database file
    Base = declarative_base()
    Session = sessionmaker(bind=engine)
    session = Session()

    image_tags_data_path = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_metaDataSnwBall/uniqueDev_metaData_SnwBall_02.csv.gz"

    try:
        save_imgs_tags_incrementally(session, image_tags_data_path)
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        session.close()  # Close the session in the finally block to ensure it's closed even if errors occur

In [ ]:
# Function to clean description
#def clean_description(description):
#       description = re.sub(r"<br\s*/?>", "\n", description)  # Replace <br> tags with newlines
#       description = re.sub(r"<[^>]+>", "", description)  # Remove other HTML tags
       # Remove special characters (except spaces and alphanumeric characters)
#       description = re.sub(r"[^a-zA-Z0-9 ]", "", description) 
#       return description


# Function to load data (combining all loading logic)
def load_data_to_database():
    engine = create_engine(DATABASE_URL)
    Session = sessionmaker(bind=engine)
    session = Session()
    image_dscrpt_data = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_metaDataSnwBall/uniqueDev_metaData_SnwBall_02.csv.gz")
    for index, row in image_dscrpt_data.iterrows():
            # Get user_id based on username
        artist = session.query(Artist).filter_by(artist_name=row['Author_Name']).first()
        if artist:
            artist_id = artist.id
            cleaned_description = clean_description(row['Author_Name'])
            new_image_dscrpt = imgs_dscrpt(id=row['Devtn_Id'], artist_id=artist_id, description=cleaned_description)
            session.add(new_image_dscrpt)
        else:
            print(f"User not found for image description with ID: {row['Devtn_Id']}")

        # Load image tags data
        #image_tags_data = pd.read_csv("your_image_tags_csv.csv")
    for index, row in image_dscrpt_data.iterrows():
        artist = session.query(Artist).filter_by(artist_name=row['Author_Name']).first()
        if artist:
            artist_id = artist.id
            process_tags(row['tag_name'], row['Devtn_Id'], artist_id, session)
        else:
            print(f"User not found for image tags with ID: {row['Devtn_Id']}")

        # Load image date data
    image_date_data = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_gallData_4_5_6/uniqueDev_gall_SnwBall03_6.2.csv.gz")
    for index, row in image_date_data.iterrows():
        artist = session.query(Artist).filter_by(artist_name=row['Author_name']).first()  # Assumed 'artist_name_csv_column' in this CSV as well
        if artist:
           artist_id = artist.id
           new_image_date = imgs_date(id=row['Deviation_id'], artist_id=artist_id, date=row['Published_on'])
           session.add(new_image_date)
        else:
           print(f"User not found for image date with ID: {row['Deviation_id']}")

    session.commit()  # Commit all changes

    #except Exception as e:
    session.rollback()
    print(f"Error loading data: {e}")
    #finally:
    session.close()

load_data_to_database()
    
#    try:
        # Load user data
#        user_data = pd.read_csv("your_users_csv.csv") 
#        for index, row in user_data.iterrows():
            # Check if user already exists
#            existing_user = session.query(User).filter_by(username=row['username_csv_column']).first()
#            if existing_user is None:
#                new_user = User(username=row['username_csv_column'], profile_url=row['profile_url_csv_column'], ...)
#                session.add(new_user)
#        session.commit()  # Commit to get generated user IDs

        # Load image description data 
        

In [ ]:
from sqlalchemy import inspect

In [ ]:
inspector = inspect(engine)

In [ ]:
tbl_names = inspector.get_table_names()
print(tbl_names)

In [ ]:
columns = inspector.get_columns('imgs_tags')  # Replace 'artists' with other table names
for column in columns:
        print(column['name'], column['type'])  # Print column name and data type

In [ ]:
from sqlalchemy import func, desc  # Import necessary functions

# ... (Your existing code) ...

def check_imgs_dscrpt_data(session):
    """Checks the first, middle, and last ten rows of the imgs_dscrpt table."""

    total_rows = session.query(func.count(Artist.id)).scalar()

    if total_rows == 0:
        print("artists table is empty.")
        return

    print("First 10 rows:")
    first_ten = session.query(imgs_dscrpt).limit(10).all()
    for row in first_ten:
        print(row.id, row.artist_id, row.artist_name, row.description)  # Adjust columns as needed

    print("\nMiddle 10 rows:")
    middle_row_num = total_rows // 2
    middle_ten = session.query(imgs_dscrpt).offset(middle_row_num - 5).limit(10).all()
    for row in middle_ten:
        print(row.id, row.artist_id, row.artist_name, row.description)  # Adjust columns as needed

    print("\nLast 10 rows:")
    last_ten = session.query(imgs_dscrpt).order_by(desc(imgs_dscrpt.id)).limit(10).all()
    for row in last_ten:
        print(row.id, row.artist_id, row.artist_name, row.description)  # Adjust columns as needed

# ... (In your load_data_to_database function or after calling it) ...
check_imgs_dscrpt_data(session)

In [ ]:
from sqlalchemy import func, desc  # Import necessary functions

# ... (Your existing code) ...

def check_imgs_date_data(session):
    """Checks the first, middle, and last ten rows of the imgs_dscrpt table."""

    total_rows = session.query(func.count(Artist.id)).scalar()

    if total_rows == 0:
        print("artists table is empty.")
        return

    print("First 10 rows:")
    first_ten = session.query(imgs_date).limit(10).all()
    for row in first_ten:
        print(row.id, row.artist_id, row.artist_name, row.date)  # Adjust columns as needed

    print("\nMiddle 10 rows:")
    middle_row_num = total_rows // 2
    middle_ten = session.query(imgs_date).offset(middle_row_num - 5).limit(10).all()
    for row in middle_ten:
        print(row.id, row.artist_id, row.artist_name, row.date)  # Adjust columns as needed

    print("\nLast 10 rows:")
    last_ten = session.query(imgs_date).order_by(desc(imgs_date.id)).limit(10).all()
    for row in last_ten:
        print(row.id, row.artist_id, row.artist_name, row.date)  # Adjust columns as needed

# ... (In your load_data_to_database function or after calling it) ...
check_imgs_date_data(session)

In [ ]:
# Create a session
#Session = sessionmaker(bind=engine)
#session = Session()

# Get the total number of rows in the table
total_rows = session.query(func.count(Artist.id)).scalar()

# Fetch the first 10 rows
first_10 = session.query(Artist).limit(10).all()

# Fetch the middle 10 rows
middle_start = total_rows // 2 - 5  # Calculate the starting index for the middle rows
middle_10 = session.query(Artist).offset(middle_start).limit(10).all()

# Fetch the last 10 rows
last_10 = session.query(Artist).order_by(Artist.id.desc()).limit(10).all()

# Print the data
def print_artist_data(artists):
    for artist in artists:
        print(f"ID: {artist.id}, Artist Name: {artist.artist_name}, Profile URL: {artist.profile_url}, ... (other columns)")  # Include other columns as needed

print("First 10 rows:")
print_artist_data(first_10)

print("\nMiddle 10 rows:")
print_artist_data(middle_10)

print("\nLast 10 rows:")
print_artist_data(last_10)

session.close()

In [ ]:
##Code from gemini
def load_data_to_database():
    engine = create_engine("your_database_connection_string")
    Session = sessionmaker(bind=engine)
    session = Session()

    try:
        # Load and process user data
        user_data = pd.read_csv("your_users_csv.csv")
        # ... (Insert user data as described earlier) ...

        # Load and process image description data
        image_dscrpt_data = pd.read_csv("your_image_descriptions_csv.csv")
        for index, row in image_dscrpt_data.iterrows():
            # ... (Clean description and insert into imgs_dscrpt) ...
            #cleaned_description = clean_description(row['description_csv_column']) 
            #new_image_dscrpt = imgs_dscrpt(id=row['id_csv_column'], user_id=artist_no, description=cleaned_description)  # Assuming you have artist_no from previous step
            #session.add(new_image_dscrpt)
            
        # Load and process image tags data
        image_tags_data = pd.read_csv("your_image_tags_csv.csv")
        for index, row in image_tags_data.iterrows():
            # ... (Process tags and insert into imgs_tags) ...
            processed_tags = process_tags(row['tags_csv_column'], row['id_csv_column'], artist_no)
            new_image_tags = imgs_tags(id=row['id_csv_column'], user_id=artist_no, description=cleaned_description)  # Assuming you have artist_no from previous step
            session.add(new_image_tags)
        # Load and process image date data
        image_date_data = pd.read_csv("your_image_dates_csv.csv")
        for index, row in image_date_data.iterrows():
            # ... (Insert into imgs_date) ...
            processed_tags = process_tags(row['tags_csv_column'], row['id_csv_column'], artist_no)
            new_image_tags = imgs_tags(id=row['id_csv_column'], user_id=artist_no, description=cleaned_description)  # Assuming you have artist_no from previous step
            session.add(new_image_date)
        session.commit()
    except Exception as e:
        session.rollback()  # Roll back if any error occurs
        print(f"Error loading data: {e}")
    finally:
        session.close()

load_data_to_database()

In [ ]:
import ast
import pandas as pd

# ... (Your database setup, imgs_dscrpt, and imgs_tags class definitions) ...

def process_tags_from_csv(image_id_column, tags_column):
    """Processes tags from a CSV file and links them to image IDs.

    Args:
        csv_file (str): Path to the CSV file.
        image_id_column (str): Name of the column containing image IDs.
        tags_column (str): Name of the column containing tags (as a string 
                          representation of a list).
    """
    
    all_tag_rows = []

    for index, row in image_dscrpt_data.iterrows():
        image_id = row[image_id_column]
        tags_string = row[tags_column]

        try:
            tags_list = ast.literal_eval(tags_string)  # Safely evaluate tags string
            for tag in tags_list:
                if pd.isnull(tag) or tag is None:
                    continue  # Skip invalid tags
                else:
                    new_image_tag = imgs_tags(tags=tag.strip(), image_id=image_id)
                    all_tag_rows.append(new_image_tag)

        except (SyntaxError, ValueError):
            print(f"Error parsing tags for image ID: {image_id}")

    # Check and print the data before inserting into the database (optional)
    if all_tag_rows:
        print("Extracted tags and image IDs:")
        for tag_row in all_tag_rows:
            print(f"- Image ID: {tag_row.image_id}, Tag: {tag_row.tags}")
    else:
        print("No valid tags found in the CSV file.")

    return all_tag_rows  # Return a list of all tag rows


# Example usage:
#csv_file = "your_csv_file.csv"  # Replace with your CSV file path
image_id_column = 'Devtn_Id'  # Replace with the actual column name for image IDs
tags_column = "tag_name"  # Replace with the actual column name for tags

all_tag_rows = process_tags_from_csv(image_id_column, tags_column)

# ... (Your code to add all_tag_rows to the database session if the data is valid) ...

In [ ]:
#from sqlalchemy import create_engine, MetaData, Table, Column, String

# ... (For adding new columns to the table using alchemy) ...

#metadata = MetaData(bind=engine)  # Get metadata associated with your engine
#imgs_tags_table = Table('imgs_tags', metadata, autoload=True)  # Load the 'imgs_tags' table

# Add the new column
#artist_name_column = Column('artist_name', String)
#imgs_tags_table.append_column(artist_name_column)

# Apply the changes to the database
#metadata.create_all(engine)

In [ ]:
# ... (Your existing code for User, imgs_date, imgs_dscrpt, imgs_tags classes) ...

# ... (Functions clean_description, process_tags, generate_artist_no defined earlier) ...

def load_data_to_database():
    engine = create_engine("your_database_connection_string")
    Session = sessionmaker(bind=engine)
    session = Session()

    try:
        # Load and process user data
        user_data = pd.read_csv("your_users_csv.csv")
        # ... (Insert user data as described earlier) ...

        # Load and process image description data
        image_dscrpt_data = pd.read_csv("your_image_descriptions_csv.csv")
        for index, row in image_dscrpt_data.iterrows():
            # ... (Clean description and insert into imgs_dscrpt) ...

        # Load and process image tags data
        image_tags_data = pd.read_csv("your_image_tags_csv.csv")
        for index, row in image_tags_data.iterrows():
            # ... (Process tags and insert into imgs_tags) ...

        # Load and process image date data
        image_date_data = pd.read_csv("your_image_dates_csv.csv")
        for index, row in image_date_data.iterrows():
            # ... (Insert into imgs_date) ...

        session.commit()
    except Exception as e:
        session.rollback()  # Roll back if any error occurs
        print(f"Error loading data: {e}")
    finally:
        session.close()

load_data_to_database()

In [ ]:
class Artist(Base):
    __tablename__ = 'artists'
    id = Column(Integer, primary_key=True)
    artist_name = Column(String)
    profile_url = Column(String)
    country = Column(String)
    level = Column(String)
    registration_date = Column(Integer)
    artists_deviations = Column(Integer)  
    artists_favourites = Column(Integer)
    artists_comments = Column(Integer)
    profile_pageviews = Column(Integer)
    profile_comments = Column(Integer)
    is_artist  = Column(Bool)
    gender = Column(String)
    speciality = Column(String)
    no_of_images = Column(Integer)
    no_of_AI_images = Column(Integer)
    ai_adopter = Column(Bool)
    ai_adoption_first_time = Column(Integer) 

#Images date class
class imgs_date(Base):  
    __tablename__ = 'imgs_date'
    id = Column(Integer, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'))  # Link to User table
    artist_name = Column(String)
    date = Column(Integer)

    user = relationship("Artist", back_populates="imgs_dscrpt")  # Define the relationship

#Images description class
class imgs_dscrpt(Base):  
    __tablename__ = 'imgs_dscrpt'
    id = Column(Integer, primary_key=True)
    artist_id = Column(Integer, ForeignKey('artists.id'))  # Link to User table
    artist_name = Column(String)
    description = Column(String)

    user = relationship("Artist", back_populates="imgs_dscrpt")  # Define the relationship

#Images tags class
class imgs_tags(Base):  
    __tablename__ = 'imgs_tags'
    id = Column(Integer, primary_key=True)
    user_id = Column(Integer, ForeignKey('artists.id'))  # Link to User table
    tags = Column(String)

    user = relationship("Artist", back_populates="imgs_tags")  # Define the relationship

#Images meta class
#class imgs_meta(Base):  
#    __tablename__ = 'imgs_meta'
#    id = Column(Integer, primary_key=True)
#    user_id = Column(Integer, ForeignKey('users.id'))  # Link to User table
#    description = Column(String)

#    user = relationship("User", back_populates="imgs_meta")  # Define the relationship



Artist.imgs_date = relationship("imgs_date", order_by=_artist_id, back_populates="artists")  # Establish relationship from User to Gallery
Artist.imgs_dscrpt = relationship("imgs_dscrpt", order_by=_artist_id, back_populates="artists")  # Establish relationship from User to Gallery
Artist.imgs_tags = relationship("imgs_", order_by=_artist_id, back_populates="artists")  # Establish relationship from User to Gallery

In [ ]:
import deviantart
import requests, threading
import random
import sqlite3
from bs4 import BeautifulSoup
import json
import time
import requests.auth
import datetime
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient
import threading
import os, re
import pandas as pd
from requests.exceptions import HTTPError
import gc, pickle 



class GatherMetaData:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")

    
    def get_response_rate(self, response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'


    # Function to get metadata
    def get_metadata(self, devIds):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params_meta = {"deviationids[]": devIds}
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
               
                api_url_devMeta = f"https://www.deviantart.com/api/v1/oauth2/deviation/metadata"
                response_devMeta = requests.get(api_url_devMeta, headers=headers, params=params_meta)
                return response_devMeta

            except requests.exceptions.RequestException as e:
                print(f"Error getting info: {e}")
                return None
    

    def parse_metadata(self, devMeta):
        # Get the metadata
        deviations_metadata = pd.DataFrame()

        try:
            data = devMeta.json()  # Extract JSON data from the response
            for i in data['metadata']:
                a = {"Devtn_Id": i['deviationid'],
                     "Devtn_Title": i["title"],
                     "Devtn_Descp": i["description"],
                     "Author_Id": i["author"]["userid"],
                     "Author_Name": i["author"]["username"],
                     "Allows_Comments": i["allows_comments"],
                     "Is_Favourited": i["is_favourited"],
                     "Is_Mature": i["is_mature"],
                     "Can_post_comments": i["can_post_comment"],
                     "Tags_Info": i["tags"]
                }
                dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
                dict_pd['tag_name'] = dict_pd['Tags_Info'].apply(lambda tags_list: [tag['tag_name'] for tag in tags_list])
                dict_pd['Sponsered'] = dict_pd['Tags_Info'].apply(lambda tags_list: [tag['sponsored'] for tag in tags_list])
                dict_pd['Sponser'] = dict_pd['Tags_Info'].apply(lambda tags_list: [tag['sponsor'] for tag in tags_list])
                dict_pd['tag_name'] = dict_pd['Tags_Info'].apply(lambda tags_list: [tag['tag_name'] for tag in tags_list])
                dict_pd['Sponsered'] = dict_pd['Tags_Info'].apply(lambda tags_list: [tag['sponsored'] for tag in tags_list])
                dict_pd['Sponser'] = dict_pd['Tags_Info'].apply(lambda tags_list: [tag['sponsor'] for tag in tags_list])
                
                deviations_metadata = pd.concat([deviations_metadata, dict_pd])
        except (requests.exceptions.RequestException, KeyError, ValueError) as e:
            print(f"Error parsing metadata: {e}")
            return pd.DataFrame()  # Return an empty DataFrame in case of error
            
        return deviations_metadata

    def fetch_deviations_metaData(self):
        """Executes the algorithm to fetch and save metadata for each deviation ID."""
        visited_meta_deviants = set()
        gallery_data_path = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_gallData_4_5_6/uniqueDev_gall_SnwBall03_6.2.csv.gz"
        metadata_path = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_metaDataSnwBall/uniqueDev_metaData_SnwBall_02.csv.gz"
        chunk_size = 10000
        columns_to_append = ['Author_name', 'Deviation_id']
        visited_deviants_file = "visited_deviants_forMetaData.pkl"  # Define the pickle file path

        # Load existing metadata deviants (if file exists)
        if os.path.exists(metadata_path):
            metadata_df = pd.read_csv(metadata_path, usecols=['Author_Name'], header=0, on_bad_lines='skip')
            visited_meta_deviants.update(metadata_df['Author_Name'].unique())
            print(f"Loaded existing metadata deviants: {len(visited_meta_deviants)}")
    
        # Load gallery data
        appended_gall_df = pd.DataFrame(columns=columns_to_append)
        if os.path.exists(gallery_data_path):
            for chunk in pd.read_csv(gallery_data_path, chunksize=chunk_size, header=0, usecols=columns_to_append, on_bad_lines='skip'):
                appended_gall_df = pd.concat([appended_gall_df, chunk[columns_to_append]], ignore_index=True)
            print(f"Loaded gallery data: {appended_gall_df.nunique()}")

        # Load visited deviants from pickle file (if exists)
        try:
            if os.path.exists(visited_deviants_file):
                with open(visited_deviants_file, "rb") as f:
                    visited_deviants.update(pickle.load(f))  # Update, not replace
        except EOFError:
            print("Warning: 'visited_deviants.pkl' is empty or corrupted. Ignoring it.")

    
        # Main execution
        deviant_count = 0
        metaId_count = 0
        
        unique_deviants_to_check = set(appended_gall_df['Author_name'].tolist())
        try:
            for deviant in unique_deviants_to_check:
                if deviant not in visited_meta_deviants:
                    visited_meta_deviants.add(deviant)
                    deviant_count += 1
                    print(f"Gathering metadata for unique deviant: {deviant}, count: {deviant_count}")
    
                    deviant_gall_data = appended_gall_df.loc[appended_gall_df['Author_name'] == deviant, 'Deviation_id']
                    devIds = deviant_gall_data.unique().tolist()
                    print(f"Total unique devIds are {len(devIds)} for {deviant}")
    
                    meta = pd.DataFrame()  # Reset meta for each deviant
                    for devId in devIds:
                        metaId_count += 1
                        print(f"Gathering metadata for deviant: {deviant}, devId: {devId}, count: {metaId_count}")
                        
                        # Check if devId has already been processed (before fetching metadata)
                        if os.path.exists(metadata_path):
                            existing_deviation_ids = pd.read_csv(metadata_path, usecols=['Devtn_Id'], on_bad_lines='skip', low_memory=False)['Devtn_Id'].tolist()
                            if devId in existing_deviation_ids:
                                print(f"Skipping devId: {devId} for deviant: {deviant} (already exists)")
                                continue  # Skip to the next devId
                
                        metadata = self.get_metadata(devId)
                
                        if metadata is not None:
                            parsed_df = self.parse_metadata(metadata)
                            
                            # Save metadata immediately if not empty
                            if not parsed_df.empty:
                                parsed_df.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)
                                print(f"Saved metadata for deviant: {deviant}, devId: {devId}")
                            else:
                                print(f"No metadata for devId: {devId} of {deviant} is available")
                        else:
                            print(f"No metadata for devId: {devId} of {deviant} is available")
                        
                        time.sleep(random.uniform(1, 2))
    
                    # Save unique metadata to CSV
                    if not meta.empty:
                        if os.path.exists(metadata_path):
                            existing_deviation_ids = pd.read_csv(metadata_path, usecols=['Devtn_Id'], on_bad_lines='skip', low_memory=False)['Devtn_Id'].tolist()
                            meta = meta[~meta['Devtn_Id'].isin(existing_deviation_ids)]  # Filter out existing deviations
                        
                        if not meta.empty:
                            meta.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)
                            print(f"Saved metadata for {deviant}")
                        else:
                            print(f"Skipping saving meta info for {deviant} (already exists)") 
    
                else:
                    print(f"Skipping already processed deviant: {deviant}")
                
                self.refresh_token()
        except requests.exceptions.RequestException as e:
            print(f"Exception occurred: {e}")

        # Save visited deviants to pickle file after processing
        with open(visited_deviants_file, "wb") as f:
            pickle.dump(visited_meta_deviants, f)
        print(f"Saved visited deviants to pickle file: {len(visited_meta_deviants)}")
        
        print("Metadata fetching completed.")

# Provide API credentials
client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"
TOKEN_URL = "https://www.deviantart.com/oauth2/token"
REDIRECT_URI = "https://www.deviantart.com/oauth2/authorize"

#Call the class and the function
# Initialize token refresh timer
metaDat = GatherMetaData(client_id, client_secret, TOKEN_URL, REDIRECT_URI)
metaDat.get_token()
metaDat.refresh_token()
metaDat.fetch_deviations_metaData()

In [ ]:
df_gall = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_gallData_4_5_6/uniqueDev_gall_RndmWalk03_6_unq2.csv.gz", index_col=False, low_memory = False)

In [ ]:
df_gall.info()

In [ ]:
df_gall.nunique()

In [ ]:
import deviantart
import requests, threading
import random
import time
import requests.auth
import datetime
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient
import threading
import os, re
import pandas as pd
from requests.exceptions import HTTPError
import gc, pickle 
'''This code calls Deviant Art APIs and gathers artists gallery information'''


class DeviantArtGalleryInfo:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")

    
    def get_response_rate(self, response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'

    def parse_gallery_data_fin2(self, devGallery):
        gallery_meta = []  # Store parsed data as a list of dictionaries
        if devGallery is not None:
            get_gallery = devGallery.get("results")
            if get_gallery is not None:
                for i in get_gallery:
                    content = i.get('content')
    
                    # Extract data and handle potential single-item lists/tuples
                    deviation_id = i['deviationid']
                    deviation_url = i['url']
                    deviation_title = i['title']
                    author_id = i['author']['userid']
                    author_name = i['author']['username']
                    author_type = i['author']['type']
                    published_on = i['published_time']
                    deviation_source = content.get('src') if content else None
                    deviation_height = content.get('height') if content else None
                    deviation_width = content.get('width') if content else None
                    deviation_transparency = content.get('transparency') if content else None
                    comments = i['stats']['comments']
                    is_mature = i['is_mature']
                    is_downloadable = i['is_downloadable']
                    favourites = i['stats']['favourites']
    
                    # Check and extract values if necessary
                    deviation_title = deviation_title[0] if isinstance(deviation_title, list) and len(deviation_title) > 0 else deviation_title
                    # Apply similar logic to other fields if they might be single-item lists/tuples
    
                    # Append data as a dictionary to the list
                    gallery_meta.append({
                        'Deviation_id': deviation_id,
                        'Deviation_url': deviation_url,
                        'Deviation_title': deviation_title,
                        'Author_id': author_id,
                        'Author_name': author_name,
                        'Author_type': author_type,
                        'Published_on': published_on,
                        'Deviation_source': deviation_source,
                        'Deviation_height': deviation_height,
                        'Deviation_width': deviation_width,
                        'Deviation_transparency': deviation_transparency,
                        'Comments': comments,
                        'is_Mature': is_mature,
                        'is_Downloadable': is_downloadable,
                        'Favourites': favourites
                    })
    
                # Create DataFrame outside the loop
                return pd.DataFrame(gallery_meta)
            else:
                print("Empty Gallery Data")
                return pd.DataFrame()  # Return an empty DataFrame
        else:
            print("No gallery found")
            return pd.DataFrame()  # Return an empty DataFrame

    def get_gallery(self,username):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params = {"username": username, "offset": 0}  # Start with offset 0
        has_more = True
        gallery_pd = pd.DataFrame()
        consecutive_empty_results = 0  # Counter for consecutive empty results
        MAX_CONSECUTIVE_EMPTY_RESULTS = 3  # Maximum allowed consecutive empty results
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            while has_more:
                try:
                        response = requests.get(
                            "https://www.deviantart.com/api/v1/oauth2/gallery/all",
                            headers=headers,
                            params=params,
                        )
                        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
                        gallery_data = response.json()
            
                        if gallery_data.get("results"):  # Check if 'results' key exists and is not empty
                            parsed_gallery = self.parse_gallery_data_fin2(gallery_data)  # Assuming parse_gallery is defined
                             # --- Exclude empty or all-NA columns before concatenation ---
                            parsed_gallery = parsed_gallery.dropna(axis=1, how='all') # Drop columns with all NA values
                            parsed_gallery = parsed_gallery[parsed_gallery.columns[parsed_gallery.notna().any()]] # Drop empty columns
                            # --- End of exclusion ---
                            
                            if len(parsed_gallery) > 0:
                                gallery_pd = pd.concat([gallery_pd, parsed_gallery], ignore_index=True) # Add ignore_index=True to avoid duplicate indices

                        else:
                            print(f"Warning: Empty 'results' for user {username}, offset {params['offset']}")
                            consecutive_empty_results += 1  # Increment counter for empty results
            
                        has_more = gallery_data["has_more"]
            
                        # Check for consecutive empty results
                        if consecutive_empty_results >= MAX_CONSECUTIVE_EMPTY_RESULTS:
                            print(f"Stopping due to {MAX_CONSECUTIVE_EMPTY_RESULTS} consecutive empty results.")
                            has_more = False  # Force stop if too many consecutive empty results
            
                        if has_more:
                            params["offset"] += len(gallery_data.get("results", []))
            
                        time.sleep(1)
                except requests.exceptions.RequestException as e:
                       print(f"Error: {e}")
                       has_more = False
            return gallery_pd

    def update_csv_with_new_data(self, new_data):
            """Updates the CSV file with new, unique data."""
            
            if os.path.exists(gall_data_path):
                existing_data = pd.read_csv(gall_data_path, on_bad_lines='skip', low_memory=False)
            else:
                existing_data = pd.DataFrame(columns=new_data.columns)  # Empty DataFrame with same columns
        
            # Filter for new data based on 'Deviation_id'
            data_to_append = new_data[~new_data['Deviation_id'].isin(existing_data['Deviation_id'])]
        
            if not data_to_append.empty:
                data_to_append.to_csv(gall_data_path, mode='a', header=not os.path.exists(gall_data_path), index=False)
                print(f"Updated CSV file with {len(data_to_append)} new rows.")
        
    def fetch_deviants_galleryInfo(self):
        """Fethc gallery data"""
        #gallery_unique_devs = "/mnt/hdd/maittewa/gallery_deviant_name_nd_deviationIds.csv"
        gall_data_path = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_gallData_4_5_6/uniqueDev_gall_RndmWalk03_6_unq2.csv.gz"
        unique_deviants = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_profileSnwball_fin1.csv.gz", low_memory=False)
        
        columns_to_append = ['Author_name']
        visited_deviants_file = "visited_deviants.pkl"
        visited_deviants = set()
        update_csv_interval = 100  # Update CSV every 100 deviants processed

        # Always read from CSV to get the most updated list of already_saved_deviants
        already_saved_deviants = pd.DataFrame(columns=columns_to_append)
        if os.path.exists(gall_data_path):
            for chunk in pd.read_csv(gall_data_path, chunksize=1000, header=0, usecols=columns_to_append, on_bad_lines='skip', low_memory=False):
                already_saved_deviants = pd.concat([already_saved_deviants, chunk[columns_to_append]], ignore_index=True)
                visited_deviants.update(already_saved_deviants["Author_name"].unique())
            print(f"Loaded unique deviants already saved on the gallery path: {len(visited_deviants)}")
    
        # Then, attempt to load additional visited deviants from the pickle file
        try:
            if os.path.exists(visited_deviants_file):
                with open(visited_deviants_file, "rb") as f:
                    visited_deviants.update(pickle.load(f))  # Update, not replace
        except EOFError:
            print("Warning: 'visited_deviants.pkl' is empty or corrupted. Ignoring it.")
            
        #save_interval = 2
        deviant_count = 0
        deviant_gall_data = pd.DataFrame()
        unique_deviants_to_check = unique_deviants['user'].tolist()
        #2.Call API for gathering the data and parsing it
        try:
            for deviant in unique_deviants_to_check:
                print(f"Total number of unique deviants are {len(unique_deviants_to_check) - len(visited_deviants)}")
                if deviant not in visited_deviants:
                    deviant_count += 1
                    visited_deviants.add(deviant)  # Add deviant to visited set
                    print(f"Gathering gallery info {deviant}, count {deviant_count}")

                    gallery = self.get_gallery(deviant)
                    if gallery is not None:
                        deviant_gall_data = pd.concat([deviant_gall_data, gallery])
                        print(f'Gathered and parsed gallery data for {deviant}')
                    else:
                        print(f"No gallery data of {deviant} available")
                    
                    time.sleep(random.uniform(1, 2))

                    # 3. Save Data
                    if not deviant_gall_data.empty:
                        if os.path.exists(gall_data_path):
                            # Load existing deviation ids
                            existing_deviation_ids = pd.read_csv(gall_data_path, usecols=['Deviation_id'], on_bad_lines='skip', low_memory=False)['Deviation_id'].tolist()
                        else:
                            existing_deviation_ids = []

                        # Filter out existing deviations before saving
                        deviant_gall_data = deviant_gall_data[~deviant_gall_data['Deviation_id'].isin(existing_deviation_ids)]
                        
                        # Save new deviations
                        if not deviant_gall_data.empty:
                            deviant_gall_data.to_csv(gall_data_path, mode="a", header=not os.path.exists(gall_data_path), index=False)
                            print(f"Saved gallery info for {deviant}")
                            
                        # --- Embedded else block ---
                        else:
                            print(f"Skipping saving gallery info for {deviant} (already exists)") 
                        # --- End of embedded else block ---

                else:
                    print(f"Skipping already visited {deviant}")
                    
                self.refresh_token()
        except requests.exceptions.RequestException as e:
            print(f"Exception occurred: {e}")

        print("Gallery information fetching completed.")

    


client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"
TOKEN_URL = "https://www.deviantart.com/oauth2/token"
REDIRECT_URI = "https://www.deviantart.com/oauth2/authorize"

# Initialize token refresh timer
Deviants_gall = DeviantArtGalleryInfo(client_id, client_secret, TOKEN_URL, REDIRECT_URI)
Deviants_gall.get_token()
Deviants_gall.refresh_token()
Deviants_gall.fetch_deviants_galleryInfo()

In [ ]:
import deviantart
import requests, threading
import random
import time
import requests.auth
import datetime
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient
import threading
import os, re
import pandas as pd
from requests.exceptions import HTTPError
import gc, pickle 
'''This code calls Deviant Art APIs and gathers artists gallery information'''


class DeviantArtGalleryInfo:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")

    
    def get_response_rate(self, response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'

    def parse_gallery_data_fin2(self, devGallery):
        gallery_meta = []  # Store parsed data as a list of dictionaries
        if devGallery is not None:
            get_gallery = devGallery.get("results")
            if get_gallery is not None:
                for i in get_gallery:
                    content = i.get('content')
    
                    # Extract data and handle potential single-item lists/tuples
                    deviation_id = i['deviationid']
                    deviation_url = i['url']
                    deviation_title = i['title']
                    author_id = i['author']['userid']
                    author_name = i['author']['username']
                    author_type = i['author']['type']
                    published_on = i['published_time']
                    deviation_source = content.get('src') if content else None
                    deviation_height = content.get('height') if content else None
                    deviation_width = content.get('width') if content else None
                    deviation_transparency = content.get('transparency') if content else None
                    comments = i['stats']['comments']
                    is_mature = i['is_mature']
                    is_downloadable = i['is_downloadable']
                    favourites = i['stats']['favourites']
    
                    # Check and extract values if necessary
                    deviation_title = deviation_title[0] if isinstance(deviation_title, list) and len(deviation_title) > 0 else deviation_title
                    # Apply similar logic to other fields if they might be single-item lists/tuples
    
                    # Append data as a dictionary to the list
                    gallery_meta.append({
                        'Deviation_id': deviation_id,
                        'Deviation_url': deviation_url,
                        'Deviation_title': deviation_title,
                        'Author_id': author_id,
                        'Author_name': author_name,
                        'Author_type': author_type,
                        'Published_on': published_on,
                        'Deviation_source': deviation_source,
                        'Deviation_height': deviation_height,
                        'Deviation_width': deviation_width,
                        'Deviation_transparency': deviation_transparency,
                        'Comments': comments,
                        'is_Mature': is_mature,
                        'is_Downloadable': is_downloadable,
                        'Favourites': favourites
                    })
    
                # Create DataFrame outside the loop
                return pd.DataFrame(gallery_meta)
            else:
                print("Empty Gallery Data")
                return pd.DataFrame()  # Return an empty DataFrame
        else:
            print("No gallery found")
            return pd.DataFrame()  # Return an empty DataFrame

    def get_gallery(self,username):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params = {"username": username, "offset": 0}  # Start with offset 0
        has_more = True
        gallery_pd = pd.DataFrame()
        consecutive_empty_results = 0  # Counter for consecutive empty results
        MAX_CONSECUTIVE_EMPTY_RESULTS = 3  # Maximum allowed consecutive empty results
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            while has_more:
                try:
                        response = requests.get(
                            "https://www.deviantart.com/api/v1/oauth2/gallery/all",
                            headers=headers,
                            params=params,
                        )
                        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
                        gallery_data = response.json()
            
                        if gallery_data.get("results"):  # Check if 'results' key exists and is not empty
                            parsed_gallery = self.parse_gallery_data_fin2(gallery_data)  # Assuming parse_gallery is defined
                            if len(parsed_gallery) > 0:
                                gallery_pd = pd.concat([gallery_pd, parsed_gallery])
                        else:
                            print(f"Warning: Empty 'results' for user {username}, offset {params['offset']}")
                            consecutive_empty_results += 1  # Increment counter for empty results
            
                        has_more = gallery_data["has_more"]
            
                        # Check for consecutive empty results
                        if consecutive_empty_results >= MAX_CONSECUTIVE_EMPTY_RESULTS:
                            print(f"Stopping due to {MAX_CONSECUTIVE_EMPTY_RESULTS} consecutive empty results.")
                            has_more = False  # Force stop if too many consecutive empty results
            
                        if has_more:
                            params["offset"] += len(gallery_data.get("results", []))
            
                        time.sleep(1)
                except requests.exceptions.RequestException as e:
                       print(f"Error: {e}")
                       has_more = False
            return gallery_pd

        
    def fetch_deviants_galleryInfo(self):
        """Executes the random walk algorithm."""
        visited_deviants = set() 
        #gallery_unique_devs = "/mnt/hdd/maittewa/gallery_deviant_name_nd_deviationIds.csv"
        gall_data_path = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_gallData_4_5_6/uniqueDev_gall_RndWalk03_6.csv.gz"
        unique_deviants = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_profileSnwball_fin1.csv.gz")
        columns_to_append = ['Author_name']

        
        already_saved_deviants = pd.DataFrame(columns = columns_to_append)
        if os.path.exists(gall_data_path):
            for chunk in pd.read_csv(gall_data_path, chunksize=1000, header=0, usecols=columns_to_append, on_bad_lines='skip'):
                already_saved_deviants = pd.concat([already_saved_deviants, chunk[columns_to_append]], ignore_index=True)
                visited_deviants.update(already_saved_deviants["Author_name"].unique())
        print(f"Loaded unique deviants already saved in the gallery: {len(visited_deviants)}, {visited_deviants}")    
        save_interval = 2
        unique_deviants_to_check = unique_deviants['user'].tolist()
        deviant_count = 0
        deviant_gall_data = pd.DataFrame()
        
        #2.Call API for gathering the data and parsing it
        try:
            for deviant in unique_deviants_to_check:
                print(f"Total number of unique deviants are {len(unique_deviants_to_check) - len(visited_deviants)}")
                if deviant not in visited_deviants:
                    deviant_count += 1
                    visited_deviants.add(deviant)
                    print(f"Gathering gallery info {deviant}, count {deviant_count}")

                    gallery = self.get_gallery(deviant)
                    if gallery is not None:
                        deviant_gall_data = pd.concat([deviant_gall_data, gallery])
                        print(f'Gathered and parsed gallery data for {deviant}')
                    else:
                        print(f"No gallery data of {deviant} available")
                    
                    time.sleep(random.uniform(1, 2))

                    # 3. Save Data
                    if not deviant_gall_data.empty:
                        deviant_gall_data.to_csv(gall_data_path, mode="a", header=not os.path.exists(gall_data_path), index=False)
                        print(f"Saved gallery info for {deviant}")  
                        #gall_df = pd.read_csv(gall_data_path)

                else:
                    print(f"Skipping already visited {deviant}")
                
                self.refresh_token()

        except requests.exceptions.RequestException as e:
            print(f"Exception occurred: {e}")
    
        print("Gallery information fetching completed.")

In [ ]:
import deviantart
import requests, threading
import random
import time
import requests.auth
import datetime
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient
import threading
import os, re
import pandas as pd
from requests.exceptions import HTTPError
import gc, pickle 
'''This code calls Deviant Art APIs and gathers artists gallery information'''


class DeviantArtGalleryInfo:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")

    
    def get_response_rate(self, response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'

    def parse_gallery_data_fin2(self, devGallery):
        gallery_meta = []  # Store parsed data as a list of dictionaries
        if devGallery is not None:
            get_gallery = devGallery.get("results")
            if get_gallery is not None:
                for i in get_gallery:
                    content = i.get('content')
    
                    # Extract data and handle potential single-item lists/tuples
                    deviation_id = i['deviationid']
                    deviation_url = i['url']
                    deviation_title = i['title']
                    author_id = i['author']['userid']
                    author_name = i['author']['username']
                    author_type = i['author']['type']
                    published_on = i['published_time']
                    deviation_source = content.get('src') if content else None
                    deviation_height = content.get('height') if content else None
                    deviation_width = content.get('width') if content else None
                    deviation_transparency = content.get('transparency') if content else None
                    comments = i['stats']['comments']
                    is_mature = i['is_mature']
                    is_downloadable = i['is_downloadable']
                    favourites = i['stats']['favourites']
    
                    # Check and extract values if necessary
                    deviation_title = deviation_title[0] if isinstance(deviation_title, list) and len(deviation_title) > 0 else deviation_title
                    # Apply similar logic to other fields if they might be single-item lists/tuples
    
                    # Append data as a dictionary to the list
                    gallery_meta.append({
                        'Deviation_id': deviation_id,
                        'Deviation_url': deviation_url,
                        'Deviation_title': deviation_title,
                        'Author_id': author_id,
                        'Author_name': author_name,
                        'Author_type': author_type,
                        'Published_on': published_on,
                        'Deviation_source': deviation_source,
                        'Deviation_height': deviation_height,
                        'Deviation_width': deviation_width,
                        'Deviation_transparency': deviation_transparency,
                        'Comments': comments,
                        'is_Mature': is_mature,
                        'is_Downloadable': is_downloadable,
                        'Favourites': favourites
                    })
    
                # Create DataFrame outside the loop
                return pd.DataFrame(gallery_meta)
            else:
                print("Empty Gallery Data")
                return pd.DataFrame()  # Return an empty DataFrame
        else:
            print("No gallery found")
            return pd.DataFrame()  # Return an empty DataFrame

    def get_gallery(self,username):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params = {"username": username, "offset": 0}  # Start with offset 0
        has_more = True
        gallery_pd = pd.DataFrame()
        consecutive_empty_results = 0  # Counter for consecutive empty results
        MAX_CONSECUTIVE_EMPTY_RESULTS = 3  # Maximum allowed consecutive empty results
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            while has_more:
                try:
                        response = requests.get(
                            "https://www.deviantart.com/api/v1/oauth2/gallery/all",
                            headers=headers,
                            params=params,
                        )
                        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
                        gallery_data = response.json()
            
                        if gallery_data.get("results"):  # Check if 'results' key exists and is not empty
                            parsed_gallery = self.parse_gallery_data_fin2(gallery_data)  # Assuming parse_gallery is defined
                            if len(parsed_gallery) > 0:
                                gallery_pd = pd.concat([gallery_pd, parsed_gallery])
                        else:
                            print(f"Warning: Empty 'results' for user {username}, offset {params['offset']}")
                            consecutive_empty_results += 1  # Increment counter for empty results
            
                        has_more = gallery_data["has_more"]
            
                        # Check for consecutive empty results
                        if consecutive_empty_results >= MAX_CONSECUTIVE_EMPTY_RESULTS:
                            print(f"Stopping due to {MAX_CONSECUTIVE_EMPTY_RESULTS} consecutive empty results.")
                            has_more = False  # Force stop if too many consecutive empty results
            
                        if has_more:
                            params["offset"] += len(gallery_data.get("results", []))
            
                        time.sleep(1)
                except requests.exceptions.RequestException as e:
                       print(f"Error: {e}")
                       has_more = False
            return gallery_pd

        
    def fetch_deviants_galleryInfo(self):
        """Executes the random walk algorithm."""
        visited_deviants = set() 
        #gallery_unique_devs = "/mnt/hdd/maittewa/gallery_deviant_name_nd_deviationIds.csv"
        gall_data_path = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_gallData_4_5_6/uniqueDev_gall_RndWalk03_6.2.csv.gz"
        unique_deviants = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_profileSnwball_fin1.csv.gz",low_memory=False)
        columns_to_append = ['Author_name']

        
        already_saved_deviants = pd.DataFrame(columns = columns_to_append)
        if os.path.exists(gall_data_path):
            for chunk in pd.read_csv(gall_data_path, chunksize=1000, header=0, usecols=columns_to_append, on_bad_lines='skip',low_memory=False):
                already_saved_deviants = pd.concat([already_saved_deviants, chunk[columns_to_append]], ignore_index=True)
                visited_deviants.update(already_saved_deviants["Author_name"].unique())
        print(f"Loaded unique deviants already saved on the gallery path: {len(visited_deviants)}")    
        save_interval = 2
        unique_deviants_to_check = unique_deviants['user'].tolist()
        deviant_count = 0
        deviant_gall_data = pd.DataFrame()
        
        #2.Call API for gathering the data and parsing it
        try:
            for deviant in unique_deviants_to_check:
                print(f"Total number of unique deviants are {len(unique_deviants_to_check) - len(visited_deviants)}")
                if deviant not in visited_deviants:
                    deviant_count += 1
                    visited_deviants.add(deviant)
                    print(f"Gathering gallery info {deviant}, count {deviant_count}")

                    gallery = self.get_gallery(deviant)
                    if gallery is not None:
                        deviant_gall_data = pd.concat([deviant_gall_data, gallery])
                        print(f'Gathered and parsed gallery data for {deviant}')
                    else:
                        print(f"No gallery data of {deviant} available")
                    
                    time.sleep(random.uniform(1, 2))

                    # 3. Save Data
                    if not deviant_gall_data.empty:
                        deviant_gall_data.to_csv(gall_data_path, mode="a", header=not os.path.exists(gall_data_path), index=False)
                        print(f"Saved gallery info for {deviant}")  
                        #gall_df = pd.read_csv(gall_data_path)
                    #self.refresh_token()


                else:
                    print(f"Skipping already visited {deviant}")
                    
                self.refresh_token()
        except requests.exceptions.RequestException as e:
            print(f"Exception occurred: {e}")
    
        print("Gallery information fetching completed.")

In [ ]:
import deviantart
import requests, threading
import random
import time
import requests.auth
import datetime
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient
import threading
import os, re
import pandas as pd
from requests.exceptions import HTTPError
import gc, pickle 
'''This code calls Deviant Art APIs and gathers artists gallery information'''


class DeviantArtGalleryInfo:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")

    
    def get_response_rate(self, response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'

    def parse_gallery_data_fin2(self, devGallery):
        gallery_meta = []  # Store parsed data as a list of dictionaries
        if devGallery is not None:
            get_gallery = devGallery.get("results")
            if get_gallery is not None:
                for i in get_gallery:
                    content = i.get('content')
    
                    # Extract data and handle potential single-item lists/tuples
                    deviation_id = i['deviationid']
                    deviation_url = i['url']
                    deviation_title = i['title']
                    author_id = i['author']['userid']
                    author_name = i['author']['username']
                    author_type = i['author']['type']
                    published_on = i['published_time']
                    deviation_source = content.get('src') if content else None
                    deviation_height = content.get('height') if content else None
                    deviation_width = content.get('width') if content else None
                    deviation_transparency = content.get('transparency') if content else None
                    comments = i['stats']['comments']
                    is_mature = i['is_mature']
                    is_downloadable = i['is_downloadable']
                    favourites = i['stats']['favourites']
    
                    # Check and extract values if necessary
                    deviation_title = deviation_title[0] if isinstance(deviation_title, list) and len(deviation_title) > 0 else deviation_title
                    # Apply similar logic to other fields if they might be single-item lists/tuples
    
                    # Append data as a dictionary to the list
                    gallery_meta.append({
                        'Deviation_id': deviation_id,
                        'Deviation_url': deviation_url,
                        'Deviation_title': deviation_title,
                        'Author_id': author_id,
                        'Author_name': author_name,
                        'Author_type': author_type,
                        'Published_on': published_on,
                        'Deviation_source': deviation_source,
                        'Deviation_height': deviation_height,
                        'Deviation_width': deviation_width,
                        'Deviation_transparency': deviation_transparency,
                        'Comments': comments,
                        'is_Mature': is_mature,
                        'is_Downloadable': is_downloadable,
                        'Favourites': favourites
                    })
    
                # Create DataFrame outside the loop
                return pd.DataFrame(gallery_meta)
            else:
                print("Empty Gallery Data")
                return pd.DataFrame()  # Return an empty DataFrame
        else:
            print("No gallery found")
            return pd.DataFrame()  # Return an empty DataFrame

    def get_gallery(self,username):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params = {"username": username, "offset": 0}  # Start with offset 0
        has_more = True
        gallery_pd = pd.DataFrame()
        consecutive_empty_results = 0  # Counter for consecutive empty results
        MAX_CONSECUTIVE_EMPTY_RESULTS = 3  # Maximum allowed consecutive empty results
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            while has_more:
                try:
                        response = requests.get(
                            "https://www.deviantart.com/api/v1/oauth2/gallery/all",
                            headers=headers,
                            params=params,
                        )
                        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
                        gallery_data = response.json()
            
                        if gallery_data.get("results"):  # Check if 'results' key exists and is not empty
                            parsed_gallery = self.parse_gallery_data_fin2(gallery_data)  # Assuming parse_gallery is defined
                            if len(parsed_gallery) > 0:
                                gallery_pd = pd.concat([gallery_pd, parsed_gallery])
                        else:
                            print(f"Warning: Empty 'results' for user {username}, offset {params['offset']}")
                            consecutive_empty_results += 1  # Increment counter for empty results
            
                        has_more = gallery_data["has_more"]
            
                        # Check for consecutive empty results
                        if consecutive_empty_results >= MAX_CONSECUTIVE_EMPTY_RESULTS:
                            print(f"Stopping due to {MAX_CONSECUTIVE_EMPTY_RESULTS} consecutive empty results.")
                            has_more = False  # Force stop if too many consecutive empty results
            
                        if has_more:
                            params["offset"] += len(gallery_data.get("results", []))
            
                        time.sleep(1)
                except requests.exceptions.RequestException as e:
                       print(f"Error: {e}")
                       has_more = False
            return gallery_pd

        
    def fetch_deviants_galleryInfo(self):
        """Executes the random walk algorithm."""
        visited_deviants = set() 
        #gallery_unique_devs = "/mnt/hdd/maittewa/gallery_deviant_name_nd_deviationIds.csv"
        gall_data_path = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_gallData_4_5_6/uniqueDev_gall_RndWalk03_6.2.csv.gz"
        unique_deviants = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_profileSnwball_fin1.csv.gz",low_memory=False)
        columns_to_append = ['Author_name']

        
        already_saved_deviants = pd.DataFrame(columns = columns_to_append)
        if os.path.exists(gall_data_path):
            for chunk in pd.read_csv(gall_data_path, chunksize=1000, header=0, usecols=columns_to_append, on_bad_lines='skip',low_memory=False):
                already_saved_deviants = pd.concat([already_saved_deviants, chunk[columns_to_append]], ignore_index=True)
                visited_deviants.update(already_saved_deviants["Author_name"].unique())
        print(f"Loaded unique deviants already saved on the gallery path: {len(visited_deviants)}")    
        save_interval = 2
        unique_deviants_to_check = unique_deviants['user'].tolist()
        deviant_count = 0
        deviant_gall_data = pd.DataFrame()
        
        #2.Call API for gathering the data and parsing it
        try:
            for deviant in unique_deviants_to_check:
                print(f"Total number of unique deviants are {len(unique_deviants_to_check) - len(visited_deviants)}")
                if deviant not in visited_deviants:
                    deviant_count += 1
                    visited_deviants.add(deviant)
                    print(f"Gathering gallery info {deviant}, count {deviant_count}")

                    gallery = self.get_gallery(deviant)
                    if gallery is not None:
                        deviant_gall_data = pd.concat([deviant_gall_data, gallery])
                        print(f'Gathered and parsed gallery data for {deviant}')
                    else:
                        print(f"No gallery data of {deviant} available")
                    
                    time.sleep(random.uniform(1, 2))

                    # 3. Save Data
                    if not deviant_gall_data.empty:
                        deviant_gall_data.to_csv(gall_data_path, mode="a", header=not os.path.exists(gall_data_path), index=False)
                        print(f"Saved gallery info for {deviant}")  
                        #gall_df = pd.read_csv(gall_data_path)
                    #self.refresh_token()


                else:
                    print(f"Skipping already visited {deviant}")
                    
                self.refresh_token()
        except requests.exceptions.RequestException as e:
            print(f"Exception occurred: {e}")
    
        print("Gallery information fetching completed.")

In [ ]:
client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"
TOKEN_URL = "https://www.deviantart.com/oauth2/token"
REDIRECT_URI = "https://www.deviantart.com/oauth2/authorize"

# Initialize token refresh timer
Deviants_gall = DeviantArtGalleryInfo(client_id, client_secret, TOKEN_URL, REDIRECT_URI)
Deviants_gall.get_token()
Deviants_gall.refresh_token()
Deviants_gall.fetch_deviants_galleryInfo()

In [ ]:
import deviantart
import requests, threading
import random
import sqlite3
import time
import requests.auth
import datetime
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient
import threading
import os, re
import pandas as pd
from requests.exceptions import HTTPError
import gc, pickle 
'''This code calls Deviant Art APIs and gathers artists gallery information'''


class DeviantArtGalleryInfo:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")

    
    def get_response_rate(self, response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'


    def parse_gallery_data(self, devGallery):
        gallery_meta = pd.DataFrame()
        if devGallery is not None:
            get_gallery = devGallery.get("results")
            if get_gallery is not None:
                for i in get_gallery: # Handle potential missing 'metadata' key
                    content = i.get('content') # Get 'content' value, or None if missing
                    a = {'Deviation_id': i['deviationid'],
                         'Deviation_url': i['url'],
                         'Deviation_title': i['title'],
                         'Author_id': i['author']['userid'],
                         'Author_name': i['author']['username'],
                         'Author_type': i['author']['type'],
                         'Published_on': i['published_time'],
                         'Deviation_source': content.get('src') if content else None,
                         'Deviation_height': content.get('height') if content else None,
                         'Deviation_width': content.get('width') if content else None,
                         'Deviation_transparency': content.get('transparency') if content else None,
                         'Comments': i['stats']['comments'],
                         'is_Mature': i['is_mature'],
                         'is_Downloadable': i['is_downloadable'],
                         'Favourites': i['stats']['favourites']}
                    dict_pd = pd.DataFrame([a]).T  # Create DataFrame from a list of dictionaries
                    dict_pd.columns = dict_pd.iloc[0]  # Set columns to the keys of the dictionary
                    dict_pd = dict_pd[1:]  # Remove the first row (which contained the keys)
                    #    dict_pd['tag_name'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['tag_name'] for tag in tags_list])
                    #    dict_pd['Sponsered'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['sponsored'] for tag in tags_list])
                    #    dict_pd['Sponser'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['sponsor'] for tag in tags_list])
                        
                    gallery_meta = pd.concat([gallery_meta, dict_pd], ignore_index=True)
                return (gallery_meta)
            else:
                print("Empty Gallery Data")
        else:
            print("No gallery found")
            return []

    def parse_gallery_data_fin2(self, devGallery):
        gallery_meta = []  # Store parsed data as a list of dictionaries
        if devGallery is not None:
            get_gallery = devGallery.get("results")
            if get_gallery is not None:
                for i in get_gallery:
                    content = i.get('content')
    
                    # Extract data and handle potential single-item lists/tuples
                    deviation_id = i['deviationid']
                    deviation_url = i['url']
                    deviation_title = i['title']
                    author_id = i['author']['userid']
                    author_name = i['author']['username']
                    author_type = i['author']['type']
                    published_on = i['published_time']
                    deviation_source = content.get('src') if content else None
                    deviation_height = content.get('height') if content else None
                    deviation_width = content.get('width') if content else None
                    deviation_transparency = content.get('transparency') if content else None
                    comments = i['stats']['comments']
                    is_mature = i['is_mature']
                    is_downloadable = i['is_downloadable']
                    favourites = i['stats']['favourites']
    
                    # Check and extract values if necessary
                    deviation_title = deviation_title[0] if isinstance(deviation_title, list) and len(deviation_title) > 0 else deviation_title
                    # Apply similar logic to other fields if they might be single-item lists/tuples
    
                    # Append data as a dictionary to the list
                    gallery_meta.append({
                        'Deviation_id': deviation_id,
                        'Deviation_url': deviation_url,
                        'Deviation_title': deviation_title,
                        'Author_id': author_id,
                        'Author_name': author_name,
                        'Author_type': author_type,
                        'Published_on': published_on,
                        'Deviation_source': deviation_source,
                        'Deviation_height': deviation_height,
                        'Deviation_width': deviation_width,
                        'Deviation_transparency': deviation_transparency,
                        'Comments': comments,
                        'is_Mature': is_mature,
                        'is_Downloadable': is_downloadable,
                        'Favourites': favourites
                    })
    
                # Create DataFrame outside the loop
                return pd.DataFrame(gallery_meta)
            else:
                print("Empty Gallery Data")
                return pd.DataFrame()  # Return an empty DataFrame
        else:
            print("No gallery found")
            return pd.DataFrame()  # Return an empty DataFrame

    def parse_gallery_data_fin(self, gallery_data):
        """Parses the gallery data returned by the DeviantArt API."""
        results = gallery_data.get("results", [])  # Get results or an empty list if no 'results' key
        parsed_data = pd.DataFrame()
        if results is not None:
            for item in results:
                deviation_id = item.get("deviationid")  # Use .get() to avoid KeyError
                url = item.get("url")
                title = item.get("title")
                author_id = item.get("author",{}).get('userid')
                author_name = item.get("author",{}).get('username')
                author_type = item.get("author",{}).get('type')
                published_on = item.get('published_time')
                deviation_source= item.get('content', {}).get('src')
                deviation_height= item.get('content', {}).get('height')
                deviation_width= item.get('content', {}).get('width')
                deviation_transparency = item.get('content', {}).get('transparency')
                deviation_comments = item.get('stats', {}).get('comments')
                deviation_is_mature = item.get('is_mature')
                deviation_is_downloadable = item.get('is_downloadable')
                deviation_Favourites = item.get('stats', {}).get('favourites')
                # Add more fields as needed
                if deviation_id is not None:
                    parsed_data.append({
                        "Deviation_id": deviation_id,
                        "URL": url,
                        "Title": title,
                        "Author_id": author_id,
                        "Author_name": author_name,
                        "Author_type": author_type,
                        "Published_on": published_on,
                        "Deviation_source": deviation_source,
                        "Deviation_height": deviation_height,
                        "Deviation_width": deviation_width,
                        "Deviation_transparency": deviation_transparency,
                        "Deviation_comments": deviation_comments,
                        "Deviation_Favourites": deviation_Favourites,
                        "Deviation_is_mature" : deviation_is_mature,
                        "Deviation_is_downloadable" : deviation_is_downloadable
                        # Add more fields here
                    })
                else:
                    print("Item skipped because no deviation_id found")
        
            return 

    def get_gallery(self,username):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params = {"username": username, "offset": 0}  # Start with offset 0
        has_more = True
        gallery_pd = pd.DataFrame()
        consecutive_empty_results = 0  # Counter for consecutive empty results
        MAX_CONSECUTIVE_EMPTY_RESULTS = 3  # Maximum allowed consecutive empty results
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            while has_more:
                try:
                        response = requests.get(
                            "https://www.deviantart.com/api/v1/oauth2/gallery/all",
                            headers=headers,
                            params=params,
                        )
                        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
                        gallery_data = response.json()
            
                        if gallery_data.get("results"):  # Check if 'results' key exists and is not empty
                            parsed_gallery = self.parse_gallery_data_fin2(gallery_data)  # Assuming parse_gallery is defined
                            if len(parsed_gallery) > 0:
                                gallery_pd = pd.concat([gallery_pd, parsed_gallery])
                        else:
                            print(f"Warning: Empty 'results' for user {username}, offset {params['offset']}")
                            consecutive_empty_results += 1  # Increment counter for empty results
            
                        has_more = gallery_data["has_more"]
            
                        # Check for consecutive empty results
                        if consecutive_empty_results >= MAX_CONSECUTIVE_EMPTY_RESULTS:
                            print(f"Stopping due to {MAX_CONSECUTIVE_EMPTY_RESULTS} consecutive empty results.")
                            has_more = False  # Force stop if too many consecutive empty results
            
                        if has_more:
                            params["offset"] += len(gallery_data.get("results", []))
            
                        time.sleep(1)
                except requests.exceptions.RequestException as e:
                       print(f"Error: {e}")
                       has_more = False
            return gallery_pd

        
    def fetch_deviants_galleryInfo(self):
        """Executes the random walk algorithm."""
        visited_deviants = set() 
        #gallery_unique_devs = "/mnt/hdd/maittewa/gallery_deviant_name_nd_deviationIds.csv"
        gall_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_testing2.csv"
        unique_deviants = pd.read_csv("/mnt/hdd/maittewa/uniqueDevs_gall_prof_RndmWalkr03.csv")
        columns_to_append = ['Author_name']
        #appended_gall_df = pd.DataFrame(columns=columns_to_append)
        #if os.path.exists(gallery_unique_devs):
         #   for chunk in pd.read_csv(gallery_unique_devs, chunksize=10000, header=0, usecols=columns_to_append, on_bad_lines='skip'):
         #       appended_gall_df = pd.concat([appended_gall_df, chunk[columns_to_append]], ignore_index=True)
         #       visited_deviants.update(appended_gall_df["Author_name"].unique())
        #print(f"Loaded unique deviants in gallery: {len(visited_deviants)}")
        
        already_saved_deviants = pd.DataFrame(columns = columns_to_append)
        if os.path.exists(gall_data_path):
            for chunk in pd.read_csv(gall_data_path, chunksize=1000, header=0, usecols=columns_to_append, on_bad_lines='skip'):
                already_saved_deviants = pd.concat([already_saved_deviants, chunk[columns_to_append]], ignore_index=True)
                visited_deviants.update(already_saved_deviants["Author_name"].unique())
        print(f"Loaded unique deviants already saved in the gallery: {len(visited_deviants)}, {visited_deviants}")    
        save_interval = 2
        unique_deviants_to_check = unique_deviants['UniqueValues'].tolist()
        deviant_count = 0
        deviant_gall_data = pd.DataFrame()
        
        #2.Call API for gathering the data and parsing it
        try:
            for deviant in unique_deviants_to_check:
                print(f"Total number of unique deviants are {len(unique_deviants_to_check) - len(visited_deviants)}")
                if deviant not in visited_deviants:
                    deviant_count += 1
                    visited_deviants.add(deviant)
                    print(f"Gathering gallery info {deviant}, count {deviant_count}")

                    gallery = self.get_gallery(deviant)
                    if gallery is not None:
                        deviant_gall_data = pd.concat([deviant_gall_data, gallery])
                        print(f'Gathered and parsed gallery data for {deviant}')
                        #print(f"Gallery dataframe: {deviant_gall_data.info()}")
                        #print(f"Gallery dataframe: {deviant_gall_data.head()}")
                        #print(f"Gallery dataframe: {deviant_gall_data.tail()}")
                    else:
                        print(f"No gallery data of {deviant} available")
                    
                    time.sleep(random.uniform(1, 2))

                    # 3. Save Data
                    if not deviant_gall_data.empty:
                        deviant_gall_data.to_csv(gall_data_path, mode="a", header=not os.path.exists(gall_data_path), index=False)
                     #   deviant_gall_data[columns_to_append].to_csv(gallery_unique_devs, mode = "a", header=False, index=False)
                        print(f"Saved gallery info for {deviant}")  
                        #gall_df = pd.read_csv(gall_data_path)
                        


                    self.refresh_token()


                else:
                    print(f"Skipping already visited {deviant}")
        except requests.exceptions.RequestException as e:
            print(f"Exception occurred: {e}")
    
        print("Gallery information fetching completed.")


client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"
TOKEN_URL = "https://www.deviantart.com/oauth2/token"
REDIRECT_URI = "https://www.deviantart.com/oauth2/authorize"

# Initialize token refresh timer
Deviants_gall = DeviantArtGalleryInfo(client_id, client_secret, TOKEN_URL, REDIRECT_URI)
Deviants_gall.get_token()
Deviants_gall.refresh_token()
Deviants_gall.fetch_deviants_galleryInfo()

In [ ]:
unique_devs = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/uniqueDev_gall_RndWalk03_6.csv")

In [ ]:
unique_devs.nunique()

In [ ]:
unique_devs.head()

In [ ]:
import deviantart
import requests, threading
import random
import sqlite3
from bs4 import BeautifulSoup
import json
import time
import requests.auth
import datetime
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient
import threading
import os, re
import pandas as pd
from requests.exceptions import HTTPError
import gc, pickle 
import watchers_friends_new, deviants_profile_snwball

In [ ]:
class DeviantArtSnowballDataGathering:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")

    def get_response_rate(self, response):
            if response.status_code == 200:
                return json.loads(response.content.decode('utf-8'))
            elif response.status_code == 404:
                return 'user_done'
            elif response.status_code == 429:
                return 'too_many_requests'
            elif response.status_code == 500:
                return 'server error'
            elif response.status_code == 401:
                return 'get new token'
        
    def get_profile(self,username):
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
                url=f"https://www.deviantart.com/api/v1/oauth2/user/profile/{username}?access_token={self.access_token}"
                response = requests.get(url)
                response.raise_for_status()
            except requests.exceptions.RequestException as e:
                print(f"Error getting profile info: {e}")
                return None
            return response

    def parse_user_profile(self,user):
        s = self.get_profile(user)
        profile_df = pd.DataFrame()
        if s is not None:
            d = json.loads(s.text)
            if 'error' not in d.keys():
                a = {'user': user,
                'user_id': d.get('userid'),
                'user_type': d.get("type"),     
                'real_name': d.get('real_name'),
                'profil_url': d.get('profile_url'),
                'tag_line': d.get('tagline'),
                'country': d.get('country'),
                'user_is_artist': d.get('user_is_artist'),
                'website': d.get('website'),
                'bio': d.get('bio'),
                'cover_photo': d.get('cover_photo'),
                'last_status': d.get('last_status'),     
                'bio': d['bio'],
                'level': d['artist_level'],
                'specialty': d['artist_specialty']}
                a.update(d['stats'])
                dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
                profile_df = pd.concat([profile_df, dict_pd]) 
                return profile_df
                print(f'Gathered profile information for the {user}')
            else:
                return pd.DataFrame()  

    def append_unique_usernames(self, df1, df2, df3, otpt_df):
        """
        Compares usernames in three DataFrames and appends unique usernames to a separate DataFrame.
    
        Args:
            df1, df2, df3: The three DataFrames to compare.
            output_df: The DataFrame to append the unique usernames to.
        """
    
        # Combine usernames from all three DataFrames
        all_usernames = pd.concat([df1["user"], df2["Deviant"], df3["Deviant"]])
    
        # Get unique usernames
        unique_usernames = all_usernames.unique()
    
        # Create a DataFrame for unique usernames
        unique_usernames_df = pd.DataFrame({"unique_dev": unique_usernames})
    
        # Append unique usernames to output_df
        otpt_df = pd.concat([otpt_df, unique_usernames_df], ignore_index=True)
    
        return otpt_df
    # Function to get friends
    def get_friends_watchers(self, username, page):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params_gallery = {"username": username}
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
                api_url_friends = f"https://www.deviantart.com/api/v1/oauth2/user/friends/{username}?access_token={self.access_token}"
                api_url_watchers = f"https://www.deviantart.com/api/v1/oauth2/user/watchers/{username}?access_token={self.access_token}"
                
                response_watchers = requests.get(api_url_watchers, params={'offset': page, 'limit': 50})
                response_friends = requests.get(api_url_friends, params={'offset': page, 'limit': 50})

                return (response_watchers, response_friends)

            except requests.exceptions.RequestException as e:
                print(f"Error getting info: {e}")
                return None

    # Function to parse friends
    def parse_friends(self, friends):
        users = pd.DataFrame()
        # print(friends.keys())
        # next_offset=friends['next_offset']
        has_more = friends.get('has_more')
        for i in friends['results']:
            a = {'Friends name': i['user']['username'],
                 'user_icon': i['user']['usericon'],
                 'type': i['user']['type'],
                 'is_watching': i['is_watching'],
                 'last_visit': i['lastvisit'],
                 'friends': i['watch']['friend'],
                 'deviations': i['watch']['deviations'],
                 'journals': i['watch']['journals'],
                 'forum_threads': i['watch']['forum_threads'],
                 'critiques': i['watch']['critiques'],
                 'scraps': i['watch']['scraps'],
                 'activity': i['watch']['activity'],
                 'collections': i['watch']['collections']}
            dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
            users = pd.concat([users, dict_pd])
        return (has_more, users)

    def parse_watchers(self, watchers):
        users = pd.DataFrame()
        # print(friends.keys())
        # next_offset=friends['next_offset']
        has_more = watchers.get('has_more')
        for i in watchers['results']:
            a = {'Watchers name': i['user']['username'],
                 'user_icon': i['user']['usericon'],
                 'type': i['user']['type'],
                 'is_watching': i['is_watching'],
                 'last_visit': i['lastvisit'],
                 'activity': i['watch']['activity'],
                 'collections': i['watch']['collections'],
                 'critiques': i['watch']['critiques'],
                 'deviations': i['watch']['deviations'],
                 'forum_threads': i['watch']['forum_threads'],
                 'friend': i['watch']['friend'],
                 'journals': i['watch']['journals'],
                 'scraps': i['watch']['scraps']}
            dict_pd = pd.DataFrame.from_dict(a, orient='index').T
            users = pd.concat([users, dict_pd], ignore_index=True)
        return (has_more, users)

        
    def watchers_friends_data(self, username):
        """Gathers watchers and watching using API."""

        watchers_pd = pd.DataFrame()
        friends_pd = pd.DataFrame()
        has_more = True
        try:
                # Get the initial batch of watchers
                for i in range(0, 5):
                    if has_more == True:
                        resp_watchers, resp_friends = self.get_friends_watchers(username, i)
                        watchers = self.get_response_rate(resp_watchers)
                        friends = self.get_response_rate(resp_friends)
                        if watchers is not None:
                            has_more, parsed_watchers = self.parse_watchers(watchers)
                            if len(parsed_watchers) > 0:
                                parsed_watchers["Deviant"] = username
                                watchers_pd = pd.concat([watchers_pd, parsed_watchers])
                        else: 
                            print(f"No watchers found for {username}")
                            return []
                    
                        if friends is not None:
                            has_more, parsed_frnds = self.parse_friends(friends)
                            if len(parsed_frnds) > 0:
                                parsed_frnds["Deviant"] = username
                                friends_pd = pd.concat([friends_pd, parsed_frnds])
                        else: 
                            print(f"No friends found for {username}")
                            return []




        except requests.exceptions.RequestException as e:
                print(f"Error scraping DeviantArt watchers, friends because of: {e}")
                return [], []
        except Exception as e:
            print(f"Error fetching friends or watchers for {username}: {e}")
            return [], []

        return watchers_pd, friends_pd    
        
    def fetch_deviant_profile(self):
        """Fetches unique deviants profile from watchers, friends and already gathered deviants."""
        profile = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRndmWalk03.csv")
        watchers = pd.read_csv("/mnt/hdd/maittewa/deviants_watchersRndmWalk03_2.csv")
        friends = pd.read_csv("/mnt/hdd/maittewa/deviants_friendsRndmWalk03_2.csv")
        profile_new = "/mnt/hdd/maittewa/deviants_profileSnwball_3.csv"
        already_existing_profile = pd.read_csv(profile_new)
        deviant_count = 0
        visited_deviants = set()
        unique_deviants = pd.DataFrame()
        visited_deviants.update(set(already_existing_profile["user"].tolist()))
        profile_data = pd.DataFrame()
        unique_deviants_to_gather = self.append_unique_usernames(profile,watchers,friends,unique_deviants)
        unique_dev_list = unique_deviants_to_gather["unique_dev"].tolist()
        for deviant in unique_dev_list:
            #print(f"length of unique deviants ")
            if deviant not in visited_deviants:
                deviant_count += 1
                visited_deviants.add(deviant)
                print(f"Gathering profile data for {deviant}, count {deviant_count}")
                profile_data = self.parse_user_profile(deviant)
                time.sleep(3)
                if profile_data is not None and not profile_data.empty:
                        print(f"fetched user profile data for {deviant}")
                        profile_data.to_csv(profile_new, mode="a", header=not os.path.exists(profile_new), index=False)
                        print(f"Saved {deviant} profile and the profile data has the shape {profile_data.shape}")
                else:
                    print(f"Empty user profile data for {deviant}")
                profile_data = pd.DataFrame()
                self.refresh_token()
            else:
                    print(f"Skipping already visited {deviant}")
        print("Snowball sampled user profile data gtahering completed.")

    def fetch_deviant_profile(self):
        """Fetches unique deviants profile from watchers, friends and already gathered deviants."""
        profile = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRndmWalk03.csv")
        watchers = pd.read_csv("/mnt/hdd/maittewa/deviants_watchersRndmWalk03_2.csv")
        friends = pd.read_csv("/mnt/hdd/maittewa/deviants_friendsRndmWalk03_2.csv")
        profile_new = "/mnt/hdd/maittewa/deviants_profileSnwball_3.csv"
        already_existing_profile = pd.read_csv(profile_new)
        deviant_count = 0
        visited_deviants = set()
        unique_deviants = pd.DataFrame()
        visited_deviants.update(set(already_existing_profile["user"].tolist()))
        profile_data = pd.DataFrame()
        unique_deviants_to_gather = self.append_unique_usernames(profile,watchers,friends,unique_deviants)
        unique_dev_list = unique_deviants_to_gather["unique_dev"].tolist()
        for deviant in unique_dev_list:
            #print(f"length of unique deviants ")
            if deviant not in visited_deviants:
                deviant_count += 1
                visited_deviants.add(deviant)
                print(f"Gathering profile data for {deviant}, count {deviant_count}")
                profile_data = self.parse_user_profile(deviant)
                time.sleep(3)
                if profile_data is not None and not profile_data.empty:
                        print(f"fetched user profile data for {deviant}")
                        profile_data.to_csv(profile_new, mode="a", header=not os.path.exists(profile_new), index=False)
                        print(f"Saved {deviant} profile and the profile data has the shape {profile_data.shape}")
                else:
                    print(f"Empty user profile data for {deviant}")
                profile_data = pd.DataFrame()
                self.refresh_token()
            else:
                    print(f"Skipping already visited {deviant}")
        print("Snowball sampled user profile data gtahering completed.")

    def snowball_sampling(self, profile, watchers, friends, rounds=100):
            """
            Performs snowball sampling to gather deviant data.
        
            Args:
                friends_df: DataFrame containing friends data.
                watchers_df: DataFrame containing watchers data.
                profile_df: DataFrame containing profile data.
                rounds: Maximum number of sampling rounds.
        
            Returns:
                A tuple containing the updated friends, watchers, and profile DataFrames.
            """
            """Fetches unique deviants profile from watchers, friends and already gathered deviants."""
            prof_new = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_profileSnwball_5.csv"
            watcrs_new = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_watchersSnwball_3.csv"
            frnds_new = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_friendsSnwBall_3.csv"
            visited_deviants = set(profile['user'])  # Start with deviants in profile_df
            current_round = 0
            deviant_no = 0
            while current_round < rounds:
                current_round += 1
                print(f"Starting round {current_round}...")
        
                # Combine usernames from all DataFrames to find new unique deviants
                all_usernames = pd.concat([friends['Friends name'], watchers['Watchers name'], profile['user']])
                new_deviants = set(all_usernames) - visited_deviants
                print(f"Total number of new deviants: {len(new_deviants)}")
        
                if not new_deviants:
                    print("No new unique deviants found. Stopping snowball sampling.")
                    break
        
                # Gather data for new deviants and update DataFrames
                for deviant in new_deviants:
                        profile_data = self.parse_user_profile(deviant)
                        if profile_data is not None and not profile_data.empty:
                            print(f"fetched user profile data for {deviant}")
                            profile_data.to_csv(prof_new, mode="a", header=not os.path.exists(prof_new), index=False)
                            print(f"Saved {deviant} profile and the profile data has the shape {profile_data.shape}")
                            profile_data = pd.concat([profile, profile_data])
                            print("Concatenated gathered profile with previous profiles")
                        else:
                            print(f"Empty user profile data for {deviant}")
                        time.sleep(3)
                        watchers_friends = self.watchers_friends_data(deviant)
                        if len(watchers_friends) == 2:
                            deviant_watchers, deviant_friends = watchers_friends
                        else:
                            print(f"No watchers, friends of {deviant} available")  
                                
                        if deviant_watchers is not None and not isinstance(deviant_watchers, list) and not deviant_watchers.empty:  # Save only if not None, not a list and not empty:
                            print(f"fetched deviant watchers for {deviant}")
                            deviant_watchers.to_csv(watcrs_new, mode="a", header=not os.path.exists(watcrs_new), index=False)
                            print(f"Saved {deviant} watchers")
                            watchers = pd.concat([watchers, deviant_watchers])
                            print("Concatenated gathered watchers with previous watchers")
                            time.sleep(5)
                        else:
                            print(f"Empty deviant watchers for {deviant}")
                            continue
                                        
                        if deviant_friends is not None and not isinstance(deviant_friends, list) and not deviant_friends.empty:  # Save only if not None, not a list and not empty:
                            print(f"fetched deviant friends for {deviant}")
                            deviant_friends.to_csv(frnds_new, mode="a", header=not os.path.exists(frnds_new), index=False)
                            print(f"Saved {deviant} friends")
                            friends = pd.concat([friends,deviant_friends])
                            print("Concatenated gathered friends with previous friends")
                        else:
                            print(f"Empty deviant friends for {deviant}")
                            
                        visited_deviants.update(new_deviants) 
                        self.refresh_token()
        
            print("Snowball sampling completed.")
            return friends, watchers, profile

In [ ]:
#Specify credentials
client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"
TOKEN_URL = "https://www.deviantart.com/oauth2/token"
REDIRECT_URI = "https://www.deviantart.com/oauth2/authorize"

In [ ]:
profile = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_profileSnwball_3.csv")
watchers = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_watchersSnwball_1.csv")
friends = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_friendsSnwBall_1.csv")
snwball_smplng = DeviantArtSnowballDataGathering(client_id, client_secret, TOKEN_URL, REDIRECT_URI)
snwball_smplng.get_token()
snwball_smplng.refresh_token()
snwball_smplng.snowball_sampling(profile, watchers, friends)

In [ ]:
friends.nunique()

In [ ]:
all_usernames = pd.concat([friends['Friends name'], watchers['Watchers name'], profile['user']])
len(set(all_usernames))

In [ ]:
def snowball_sampling(friends_df, watchers_df, profile_df, rounds=100):
    """
    Performs snowball sampling to gather deviant data.

    Args:
        friends_df: DataFrame containing friends data.
        watchers_df: DataFrame containing watchers data.
        profile_df: DataFrame containing profile data.
        rounds: Maximum number of sampling rounds.

    Returns:
        A tuple containing the updated friends, watchers, and profile DataFrames.
    """

    visited_deviants = set(profile_df['username'])  # Start with deviants in profile_df
    current_round = 0

    while current_round < rounds:
        current_round += 1
        print(f"Starting round {current_round}...")

        # Combine usernames from all DataFrames to find new unique deviants
        all_usernames = pd.concat([friends_df['Deviant'], watchers_df['Deviant'], profile_df['user']])
        new_deviants = set(all_usernames) - visited_deviants

        if not new_deviants:
            print("No new unique deviants found. Stopping snowball sampling.")
            break

        # Gather data for new deviants and update DataFrames
        for deviant in new_deviants:
            # Gather profile, friends, and watchers data for the deviant (replace with your data gathering logic)
            deviant_profile = gather_profile_data(deviant)  
            deviant_friends = gather_friends_data(deviant)
            deviant_watchers = gather_watchers_data(deviant)

            # Append new data to existing DataFrames
            profile_df = pd.concat([profile_df, deviant_profile])
            friends_df = pd.concat([friends_df, deviant_friends])
            watchers_df = pd.concat([watchers_df, deviant_watchers])

        visited_deviants.update(new_deviants)

    print("Snowball sampling completed.")
    return friends_df, watchers_df, profile_df

In [ ]:
def find_bad_lines(filepath, delimiter=','):
    """
    Finds lines in a CSV that don't have a consistent number of columns.

    Args:
        filepath: Path to the CSV file.
        delimiter: The column delimiter (default is comma).

    Returns:
        A list of line numbers (starting from 1) that have a different
        number of columns than the first row.
    """
    bad_lines = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            header = f.readline().strip()
            expected_cols = len(header.split(delimiter))
            for i, line in enumerate(f, start=2):  # Start at line 2 (line 1 is the header)
                line = line.strip()
                if not line:
                    continue  # Skip empty lines
                cols = len(line.split(delimiter))
                if cols != expected_cols:
                    bad_lines.append((i, cols))
    except UnicodeDecodeError:
        print(f"Error decoding file {filepath}. Trying with 'latin-1' encoding.")
        with open(filepath, 'r', encoding='latin-1') as f:
            header = f.readline().strip()
            expected_cols = len(header.split(delimiter))
            for i, line in enumerate(f, start=2):  # Start at line 2 (line 1 is the header)
                line = line.strip()
                if not line:
                    continue  # Skip empty lines
                cols = len(line.split(delimiter))
                if cols != expected_cols:
                    bad_lines.append((i, cols))

    return bad_lines


class DeviantArtRandomWalk:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI, database_path="/mnt/hdd/maittewa/random_walker_since2003_apiData.db"):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.database_path = database_path
        self.create_database()
        self.conn = sqlite3.connect(self.database_path)
        self.cursor = self.conn.cursor()
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")

    
    def get_response_rate(self, response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'

    def random_days_since_2001(self):
        """Returns a random number of days since January 1, 2001."""
        today = datetime.date.today()
        start_date = datetime.date(2001, 1, 1)
        total_days = (today - start_date).days
        random_days = random.randint(0, total_days)
        return random_days

    def create_database(self):
        """Creates the SQLite database if it doesn't exist."""
        conn = sqlite3.connect(self.database_path)
        cursor = conn.cursor()

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS deviants (
                username TEXT PRIMARY KEY,
                profile_url TEXT
            )
        """)

        conn.commit()
        conn.close()
        
    def store_data(self, deviant, profile, dev_watchers, dev_friends):
        """Stores the data in the SQLite database."""
        conn = sqlite3.connect(self.database_path)
        cursor = conn.cursor()

        cursor.execute("INSERT OR IGNORE INTO deviants (username, profile_url) VALUES (?, ?)",
                       (deviant, f"https://www.deviantart.com/{deviant}"))
        if not profile.empty:
            profile.to_sql("profile", conn, if_exists='replace', index=False)
        else:
            print("Skipping insertion of profile data because its empty")

        if not dev_watchers.empty:
            dev_watchers.to_sql("watchers", conn, if_exists='replace', index=False)
        else:
            print("Skipping insertion of watchers data because its empty")

        if not dev_friends.empty:
            dev_friends.to_sql("friends", conn, if_exists='replace', index=False)
        else:
            print("Skipping insertion of friends data because its empty")
        #metadata.to_sql("metadata", conn, if_exists='replace', index=False)

        print("Stored data for deviant:", deviant)


        
    def get_random_deviants_from_daily_deviations(self, num_deviants, date):
        """Fetches a list of random deviants from a specific tag and page."""
        # Check if we have a valid token
        if not self.access_token: 
            self.access_token = self.get_token()
        url = f"https://www.deviantart.com/api/v1/oauth2/browse/dailydeviations?access_token={self.access_token}"
        params = {
            "client_id": client_id,
            "client_secret": client_secret,
            "date": date.strftime("%Y-%m-%d"),  # Format date as YYYY-MM-DD
            # ... (other parameters if needed, e.g., limit, offset) ...
        }
        try:
            response = requests.get(url, params=params)
            response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
            data = response.json()
            # results = self.da.browse_dailydeviations()
            # results = self.da.browse(tag=tag, offset=(page - 1) * 24)  # 24 results per page by default
            deviants = [deviation["author"]["username"] for deviation in data["results"]]
            random_deviants = random.sample(deviants, min(num_deviants, len(deviants)))
            return random_deviants
            # Get as many as possible
        except Exception as e:
            print(f"Error fetching deviants: {e}")
            return []


    # Function to get friends
    def get_friends_watchers_gallery(self, username, page):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params_gallery = {"username": username}
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
                api_url_friends = f"https://www.deviantart.com/api/v1/oauth2/user/friends/{username}?access_token={self.access_token}"
                api_url_watchers = f"https://www.deviantart.com/api/v1/oauth2/user/watchers/{username}?access_token={self.access_token}"
                api_url_gallery = f"https://www.deviantart.com/api/v1/oauth2/gallery/folders"
                response_watchers = requests.get(api_url_watchers, params={'offset': page, 'limit': 50})
                response_friends = requests.get(api_url_friends, params={'offset': page, 'limit': 50})
                response_gallery = requests.get(api_url_gallery, headers=headers, params=params_gallery)

                return (response_watchers, response_friends, response_gallery)

            except requests.exceptions.RequestException as e:
                print(f"Error getting info: {e}")
                return None

    # Function to get metadata
    def get_metadata(self, devIds):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params_meta = {"deviationids[]": devIds}
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
               
                api_url_devMeta = f"https://www.deviantart.com/api/v1/oauth2/deviation/metadata"
                response_devMeta = requests.get(api_url_devMeta, headers=headers, params=params_meta)
                return response_devMeta

            except requests.exceptions.RequestException as e:
                print(f"Error getting info: {e}")
                return None
    


        
    def get_profile(self,username):
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
                api_url_profile = f"https://www.deviantart.com/api/v1/oauth2/user/profile/{username}?access_token={self.access_token}"
                response_profile = requests.get(api_url_profile)
                return response_profile
            except requests.exceptions.RequestException as e:
                print(f"Error getting profile info: {e}")
                return None

    def parse_user_profile(self, profile_response):
        profile_df = pd.DataFrame()
        if 'error' not in d.keys():
                a = {'user': user,
                     'real_name': d.get('real_name'),
                     'profil_url': d['profile_url'],
                     'tags': d['tagline'],
                     'country': d['countryid'],
                     'bio': d['bio'],
                     'level': d['artist_level'],
                     'specialty': d['artist_specialty']}
                a.update(d['stats'])
                dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
                profile_df = pd.concat([profile_df, dict_pd]) 
                return profile_df
                print(f'Gathered profile information for the {user}')
        else:
            return 'error'    

    # Function to parse friends
    def parse_friends(self, friends):
        users = pd.DataFrame()
        # print(friends.keys())
        # next_offset=friends['next_offset']
        has_more = friends.get('has_more')
        for i in friends['results']:
            a = {'username': i['user']['username'],
                 'user_icon': i['user']['usericon'],
                 'type': i['user']['type'],
                 'is_watching': i['is_watching'],
                 'last_visit': i['lastvisit'],
                 'friends': i['watch']['friend'],
                 'deviations': i['watch']['deviations'],
                 'journals': i['watch']['journals'],
                 'forum_threads': i['watch']['forum_threads'],
                 'critiques': i['watch']['critiques'],
                 'scraps': i['watch']['scraps'],
                 'activity': i['watch']['activity'],
                 'collections': i['watch']['collections']}
            dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
            users = pd.concat([users, dict_pd])
        return (has_more, users)

    def parse_watchers(self, watchers):
        users = pd.DataFrame()
        # print(friends.keys())
        # next_offset=friends['next_offset']
        has_more = watchers.get('has_more')
        for i in watchers['results']:
            a = {'username': i['user']['username'],
                 'user_icon': i['user']['usericon'],
                 'type': i['user']['type'],
                 'is_watching': i['is_watching'],
                 'last_visit': i['lastvisit'],
                 'activity': i['watch']['activity'],
                 'collections': i['watch']['collections'],
                 'critiques': i['watch']['critiques'],
                 'deviations': i['watch']['deviations'],
                 'forum_threads': i['watch']['forum_threads'],
                 'friend': i['watch']['friend'],
                 'journals': i['watch']['journals'],
                 'scraps': i['watch']['scraps']}
            dict_pd = pd.DataFrame.from_dict(a, orient='index').T
            users = pd.concat([users, dict_pd], ignore_index=True)
        return (has_more, users)

    def parse_metadata(self, devMeta):
        # Get the metadata
        deviations_metadata = pd.DataFrame()

        try:
            data = devMeta.json()  # Extract JSON data from the response
            for i in data['metadata']:
                a = {"DevtnId": i['deviationid'],
                     "DevtnTitle": i["title"],
                     "DevtnDescp": i["description"],
                     "AuthorId": i["author"]["userid"],
                     "AuthorName": i["author"]["username"],
                     "AuthorIcon": i["author"]["usericon"],
                     "AuthorType": i["author"]["type"],
                     "License": i["license"],
                     "AllowsComments": i["allows_comments"],
                     "IsFavourited": i["is_favourited"],
                     "IsMature": i["is_mature"],
                     "CanPostComments": i["can_post_comment"],
                     "TagsInfo": i["tags"]
                }
                dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
                dict_pd['tag_name'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['tag_name'] for tag in tags_list])
                dict_pd['Sponsered'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['sponsored'] for tag in tags_list])
                dict_pd['Sponser'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['sponsor'] for tag in tags_list])
                
                deviations_metadata = pd.concat([deviations_metadata, dict_pd])
        except (requests.exceptions.RequestException, KeyError, ValueError) as e:
            print(f"Error parsing metadata: {e}")
            return pd.DataFrame()  # Return an empty DataFrame in case of error
            
        return deviations_metadata

    def parse_gallery_data(self, devGallery):
        gallery_meta = pd.DataFrame()
        if devGallery is not None:
            get_gallery = devGallery.get("results")
            if get_gallery is not None:
                for i in get_gallery: # Handle potential missing 'metadata' key
                    content = i.get('content') # Get 'content' value, or None if missing
                    a = {'Deviation_id': i['deviationid'],
                         'Deviation_url': i['url'],
                         'Deviation_title': i['title'],
                         'Author_id': i['author']['userid'],
                         'Author_name': i['author']['username'],
                         'Author_type': i['author']['type'],
                         'Published_on': i['published_time'],
                         'Deviation_source': content.get('src') if content else None,
                         'Deviation_height': content.get('height') if content else None,
                         'Deviation_width': content.get('width') if content else None,
                         'Deviation_transparency': content.get('transparency') if content else None,
                         'Comments': i['stats']['comments'],
                         'is_Mature': i['is_mature'],
                         'is_Downloadable': i['is_downloadable'],
                         'Favourites': i['stats']['favourites']}
                    dict_pd = pd.DataFrame([a]).T  # Create DataFrame from a list of dictionaries
                    dict_pd.columns = dict_pd.iloc[0]  # Set columns to the keys of the dictionary
                    dict_pd = dict_pd[1:]  # Remove the first row (which contained the keys)
                    #    dict_pd['tag_name'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['tag_name'] for tag in tags_list])
                    #    dict_pd['Sponsered'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['sponsored'] for tag in tags_list])
                    #    dict_pd['Sponser'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['sponsor'] for tag in tags_list])
                        
                    gallery_meta = pd.concat([gallery_meta, dict_pd], ignore_index=True)
                return (gallery_meta)
            else:
                print("Empty Gallery Data")
        else:
            print("No gallery found")
            return []

    def get_gallery(self,username):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params = {"username": username, "offset": 0}  # Start with offset 0
        has_more = True
        gallery_pd = pd.DataFrame()
        consecutive_empty_results = 0  # Counter for consecutive empty results
        MAX_CONSECUTIVE_EMPTY_RESULTS = 3  # Maximum allowed consecutive empty results
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            while has_more:
                try:
                        response = requests.get(
                            "https://www.deviantart.com/api/v1/oauth2/gallery/all",
                            headers=headers,
                            params=params,
                        )
                        response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
                        gallery_data = response.json()
            
                        if gallery_data.get("results"):  # Check if 'results' key exists and is not empty
                            parsed_gallery = self.parse_gallery1(gallery_data)  # Assuming parse_gallery is defined
                            if len(parsed_gallery) > 0:
                                gallery_pd = pd.concat([gallery_pd, parsed_gallery])
                        else:
                            print(f"Warning: Empty 'results' for user {username}, offset {params['offset']}")
                            consecutive_empty_results += 1  # Increment counter for empty results
            
                        has_more = gallery_data["has_more"]
            
                        # Check for consecutive empty results
                        if consecutive_empty_results >= MAX_CONSECUTIVE_EMPTY_RESULTS:
                            print(f"Stopping due to {MAX_CONSECUTIVE_EMPTY_RESULTS} consecutive empty results.")
                            has_more = False  # Force stop if too many consecutive empty results
            
                        if has_more:
                            params["offset"] += len(gallery_data.get("results", []))
            
                        time.sleep(1)
                except requests.exceptions.RequestException as e:
                       print(f"Error: {e}")
                       has_more = False
            return gallery_pd


    def get_gallery1(self, username):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params = {"username": username, "offset": 0}
        gallery_pd = pd.DataFrame()
    
        has_more = True
        while has_more:
            try:
                response = requests.get(
                    "https://www.deviantart.com/api/v1/oauth2/gallery/all",
                    headers=headers,
                    params=params,
                )
                response.raise_for_status()
                gallery_data = response.json()
    
                #print(f"API Response for user {username}, offset {params['offset']}:")
                #print(gallery_data)
    
                if "error" in gallery_data.keys():
                    print(f"Error in API response for user {username}: {gallery_data['error_description']}")
                    has_more = False  # Stop if there's an error
                    continue
    
                if gallery_data.get("results"):
                     
                    parsed_gallery = self.parse_gallery1(gallery_data)
                    if not parsed_gallery.empty:
                        gallery_pd = pd.concat([gallery_pd, parsed_gallery])
                    else:
                        print(f"Warning: Empty 'results' for user {username}, offset {params['offset']}")
                
                has_more = gallery_data.get("has_more", False)
                if has_more:
                    params["offset"] += len(gallery_data.get("results", []))
                time.sleep(1)
    
            except requests.exceptions.RequestException as e:
                print(f"Error: {e}")
                has_more = False
        return gallery_pd

    def parse_gallery1(self, gallery_data):
        """Parses the gallery data returned by the DeviantArt API."""
        results = gallery_data.get("results", [])  # Get results or an empty list if no 'results' key
        parsed_data = []
        for item in results:
            deviation_id = item.get("deviationid")  # Use .get() to avoid KeyError
            url = item.get("url")
            title = item.get("title")
            author_id = item.get('userid')
            author_name = item.get('username')
            author_type = item.get('type')
            published_on = item.get('published_time')
            deviation_source= item.get('src') if item else None
            deviation_height= item.get('height') if item else None,
            deviation_width= item.get('width') if item else None,
            deviation_transparency = item.get('transparency') if item else None,
            deviation_comments = item.get('stats', {}).get('comments', None),
            deviation_Favourites = item.get('stats', {}).get('favourites', None)
            # Add more fields as needed
            if deviation_id is not None:
                parsed_data.append({
                    "Deviation_id": deviation_id,
                    "URL": url,
                    "Title": title,
                    "Author_id": author_id,
                    "Author_name": author_name,
                    "Author_type": author_type,
                    "Published_on": published_on,
                    "Deviation_source": deviation_source,
                    "Deviation_height": deviation_height,
                    "Deviation_width": deviation_width,
                    "Deviation_transparency": deviation_transparency,
                    "Deviation_comments": deviation_comments,
                    "Deviation_Favourites": deviation_Favourites
                    # Add more fields here
                })
            else:
                print("Item skipped because no deviation_id found")
    
        return pd.DataFrame(parsed_data)
        
    def watchers_friends_data(self, username):
        """Gathers watchers and watching using API."""

        watchers_pd = pd.DataFrame()
        friends_pd = pd.DataFrame()
        has_more = True
        try:
                # Get the initial batch of watchers
                for i in range(0, 5):
                    if has_more == True:
                        resp_watchers, resp_friends, resp_gallery = self.get_friends_watchers_gallery(username, i)
                        watchers = self.get_response_rate(resp_watchers)
                        friends = self.get_response_rate(resp_friends)
                        if watchers is not None:
                            has_more, parsed_watchers = self.parse_watchers(watchers)
                            if len(parsed_watchers) > 0:
                                watchers_pd = pd.concat([watchers_pd, parsed_watchers])
                        else: 
                            print(f"No watchers found for {username}")
                            return []
                    
                        if friends is not None:
                            has_more, parsed_frnds = self.parse_friends(friends)
                            if len(parsed_frnds) > 0:
                                friends_pd = pd.concat([friends_pd, parsed_frnds])
                        else: 
                            print(f"No friends found for {username}")
                            return []




        except requests.exceptions.RequestException as e:
                print(f"Error scraping DeviantArt watchers, friends and gallery because of: {e}")
                return [], []
        except Exception as e:
            print(f"Error fetching gallery, friends or watchers for {username}: {e}")
            return [], []

        return watchers_pd, friends_pd, gallery_pd

    def load_visited_deviants(self, gallery_data_path):
        visited_deviants = set()
    
        # 1. Load from pickle file first
        visited_deviants_file = "visited_deviants.pkl"
        try:
            with open(visited_deviants_file, 'rb') as f:
                visited_deviants = pickle.load(f)
        except FileNotFoundError:
            pass  # Handle case where pickle file doesn't exist yet
    
        # 2. Load from gallery_data_path (using pandas for CSV)
        try:
            if os.path.exists(gallery_data_path):
                print(f"Loading visited deviants from {gallery_data_path}...")
                # Use iterators to read only the 'Author_name' column efficiently
                for chunk in pd.read_csv(gallery_data_path, chunksize=10000, header=0, usecols=['Author_name'],
                                         on_bad_lines='skip'):
                    # Directly update the set from the column.
                    visited_deviants.update(chunk['Author_name'].unique())
                print(
                    f"Finished loading visited deviants from {gallery_data_path}. Total: {len(set(visited_deviants))}")
            else:
              print("No data of visited deviants")
        except pd.errors.EmptyDataError:
            print(f"Warning: {gallery_data_path} is empty.")
        except FileNotFoundError:
            print(f"Warning: {gallery_data_path} not found.")
        except Exception as e:
            print(f"An error occured trying to load the file {gallery_data_path}, {e}")
    
        return visited_deviants
        
    def fetch_gallery_deviationids_metaData(self):
        """Executes the random walk algorithm."""
        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        already_saved_deviants_file = "/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv"
        metadata_path = "/mnt/hdd/maittewa/uniqueDev_dvtnMetaRndmWalk03_1.csv"
        
        # Load visited deviants (using pickle for persistence)
        visited_deviants_file = "visited_deviants.pkl"  
        try:
            with open(visited_deviants_file, 'rb') as f:
                visited_deviants = pickle.load(f)
                print(f"Loaded {len(visited_deviants)} visited deviants from pickle file.") # Debugging line
        except FileNotFoundError:
            visited_deviants = set()  # Start with an empty set if the file doesn't exist
            print("Pickle file not found. Starting with an empty set of visited deviants.") # Debugging line

        # Load from gallery data file 
        visited_deviants = self.load_visited_deviants(gallery_data_path)

        print(f"Initial size of visited_deviants: {len(set(visited_deviants))}")  # Debugging line

        
        save_interval = 10

        if os.path.exists(already_saved_deviants_file):
            try:
                already_saved_deviants = pd.read_csv(already_saved_deviants_file)
            except pd.errors.EmptyDataError:
                print(f"Error: {already_saved_deviants_file} is empty.")
                return  # Exit the function if the file is empty
            except FileNotFoundError:
                print(f"Error: {already_saved_deviants_file} not found.")
                return  # Exit the function if the file is not found
            except Exception as e:
                print(f"An error occurred trying to load the file: {already_saved_deviants_file}, {e}")
                return

            unique_deviants_to_check = already_saved_deviants['user'].tolist()
            deviant_count = 0
            deviant_gall_data = pd.DataFrame()
            meta = pd.DataFrame()
            for deviant in unique_deviants_to_check:
                #print(f"length of unique deviants ")
                if deviant not in visited_deviants:
                    deviant_count += 1
                    visited_deviants.add(deviant)
                    print(f"Gathering gallery info and metadata for {deviant}, count {deviant_count}")
                    
                    gallery = self.get_gallery(deviant)
                    if gallery is not None:
                        deviant_gall_data = pd.concat([deviant_gall_data, gallery])
                        print(f'Gathered and parsed gallery data for {deviant}')
                    else:
                        print(f"No gallery data of {deviant} available")
                    
                    time.sleep(random.uniform(1, 2))

                    if not deviant_gall_data.empty:
                        devIds = deviant_gall_data["Deviation_id"].tolist()
                        metadata = self.get_metadata(devIds)
                        if metadata:
                            parsed_df = self.parse_metadata(metadata)
                            meta = pd.concat([meta, parsed_df], ignore_index=True)
                        print(f'Gathered and parsed deviation metadata for {deviant}')
                    
                    # 3. Save Data in Batches:
                    if deviant_count % save_interval == 0:
                        print(f"Saving data for {deviant_count} deviants...")  # Debugging line
                        deviant_gall_data.to_csv(gallery_data_path, mode="a", header=not os.path.exists(gallery_data_path), index=False)
                        if not meta.empty:
                            meta.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)

                        # Count unique deviants in gallery_data_path
                        gallery_df = pd.read_csv(gallery_data_path, usecols=['Author_name'], on_bad_lines='skip')
                        unique_gallery_deviants = len(gallery_df['Author_name'].unique())
                        print(f"Total unique deviants in gallery_data_path: {unique_gallery_deviants}")

                        # Count unique deviants in metadata_path (if it exists and has 'author')
                        if os.path.exists(metadata_path) and 'AuthorName' in pd.read_csv(metadata_path, nrows=1).columns:
                            metadata_df = pd.read_csv(metadata_path, usecols=['AuthorName'], on_bad_lines='skip')
                            unique_metadata_deviants = len(metadata_df['AuthorName'].unique())
                            print(f"Total unique deviants in metadata_path: {unique_metadata_deviants}")
                        else:
                            print("metadata_path does not exist or does not have an 'author' column.")

                        deviant_gall_data = pd.DataFrame()
                        meta = pd.DataFrame()
                        print(f"Saved data for {deviant_count} deviants.")
                        # Save visited_deviants to pickle file incrementally
                        with open(visited_deviants_file, 'wb') as f:
                            pickle.dump(visited_deviants, f)
                        print(f"Saved visited_deviants to pickle file.")

                    self.refresh_token()
                    
                    # Option 1: Using `to_csv()` to append deviants (recommended)
                    #visited_deviants_df = pd.DataFrame({'Author_name': list(visited_deviants)}) 
                    #visited_deviants_df.to_csv(gallery_data_path, mode='a', header=not os.path.exists(gallery_data_path), index=False)

                    # Option 2: Saving `visited_deviants` separately for debugging
                    # with open('visited_deviants.csv', 'w') as vd_file:
                    #    for deviant_name in visited_deviants:
                    #        vd_file.write(deviant_name + '\n')


                else:
                    print(f"Skipping already visited {deviant}")
        else:
            print(f"The file {already_saved_deviants_file} does not exist.")

        # Save visited deviants to file (before exiting the function)
        with open(visited_deviants_file, 'wb') as f:
            pickle.dump(visited_deviants, f)    
        print("Random walk completed.")

    def fetch_gallery_info(self):
        """Executes the random walk algorithm."""
        # 1. Load Existing Data and Visited Deviants:
        visited_gall_deviants = set()
        already_saved_deviants = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")

        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        chunk_size = 10000
        if os.path.exists(gallery_data_path):
            print(f"Loading visited deviants from {gallery_data_path}...")
            # Use iterators to read only the 'Author_name' column efficiently
            for chunk in pd.read_csv(gallery_data_path, chunksize=chunk_size, header=0, usecols=['Author_name'],on_bad_lines='skip'):
                # Directly update the set from the column.
                visited_gall_deviants.update(chunk['Author_name'].unique())
            print(f"Finished loading visited deviants from {gallery_data_path}. Total: {len(visited_gall_deviants)}")
        
        deviant_count = 0  # Keep track of processed deviants
        # Establish a single database connection outside the loop
        #count_for_reauth = 0
        deviant_batch = []
        user_name = [] 
        deviant_gall_data = pd.DataFrame()
        save_interval = 2  # Save data after processing this many deviants
        deviant_count = 0
        batch_size = 10

        # Main execution
        # Split the deviation IDs into chunks to avoid exceeding API limits
        try:
            for i in range(0, len(already_saved_deviants), batch_size):
                batch_usernames = already_saved_deviants['user'][i:i + batch_size].tolist()

                for deviant in batch_usernames:
                    if deviant not in visited_gall_deviants:
                        visited_gall_deviants.add(deviant)
                        deviant_count += 1
                        print(f"Gathering gallery info for unique {deviant}, count {deviant_count}")
                        gallery = self.get_gallery(deviant)
                        if gallery is not None:
                            gallery_df = pd.DataFrame(gallery)
                            deviant_gall_data = pd.concat([deviant_gall_data,gallery_df])
                            print(f'Gathered and parsed gallery data for {deviant}')
                            print(deviant_gall_data.info())
                            try:
                                deviant_gall_data.to_csv(gallery_data_path, mode="a", header=False, index=False)
                                print(f"Saved gallery info for {deviant}")
                            except Exception as e:
                               print(f"Error saving to CSV: {e}")
                               print("DataFrame contents:", deviant_gall_data.head()) # Print a sample of the DataFrame
                            
                        else:
                            print(f"No gallery data of {deviant} available")
                            # 3. Save Data in Batches:
                            deviant_gall_data = pd.DataFrame()
                            self.refresh_token()
                            time.sleep(random.uniform(1,3)) 
                        # Count unique deviants in gallery_data_path
                        unique_gall_deviants = set()  # Use a set to efficiently store unique values
                        # Iterate through chunks and update the set of unique deviants
                        for chunk in pd.read_csv(gallery_data_path, chunksize=chunk_size, usecols=['Author_name'], on_bad_lines='skip'):
                            unique_gall_deviants.add(chunk['Author_name'].unique())
                        print(f"Total unique deviants in the gallery_data: {len(unique_gall_deviants)}")
                    else:
                        print(f"Skipping already visited {deviant}")

        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")
            
        print("Random walk completed.")
    

In [ ]:
class DeviantArtProfileData:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")


        
    def get_profile(self,username):
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
                url=f"https://www.deviantart.com/api/v1/oauth2/user/profile/{username}?access_token={self.access_token}"
                response = requests.get(url)
                response.raise_for_status()
            except requests.exceptions.RequestException as e:
                print(f"Error getting profile info: {e}")
                return None
            return response

    def parse_user_profile(self,user):
        s = self.get_profile(user)
        profile_df = pd.DataFrame()
        if s is not None:
            d = json.loads(s.text)
            if 'error' not in d.keys():
                a = {'user': user,
                'user_id': d.get('userid'),
                'user_type': d.get("type"),     
                'real_name': d.get('real_name'),
                'profil_url': d.get('profile_url'),
                'tag_line': d.get('tagline'),
                'country': d.get('country'),
                'user_is_artist': d.get('user_is_artist'),
                'website': d.get('website'),
                'bio': d.get('bio'),
                'cover_photo': d.get('cover_photo'),
                'last_status': d.get('last_status'),     
                'bio': d['bio'],
                'level': d['artist_level'],
                'specialty': d['artist_specialty']}
                a.update(d['stats'])
                dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
                profile_df = pd.concat([profile_df, dict_pd]) 
                return profile_df
                print(f'Gathered profile information for the {user}')
            else:
                return pd.DataFrame()  

    def append_unique_usernames(self, df1, df2, df3, otpt_df):
        """
        Compares usernames in three DataFrames and appends unique usernames to a separate DataFrame.
    
        Args:
            df1, df2, df3: The three DataFrames to compare.
            output_df: The DataFrame to append the unique usernames to.
        """
    
        # Combine usernames from all three DataFrames
        all_usernames = pd.concat([df1["user"], df2["Deviant"], df3["Deviant"]])
    
        # Get unique usernames
        unique_usernames = all_usernames.unique()
    
        # Create a DataFrame for unique usernames
        unique_usernames_df = pd.DataFrame({"unique_dev": unique_usernames})
    
        # Append unique usernames to output_df
        otpt_df = pd.concat([otpt_df, unique_usernames_df], ignore_index=True)
    
        return otpt_df
        
    def fetch_deviant_profile(self):
        """Fetches unique deviants profile from watchers, friends and already gathered deviants."""
        profile = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRndmWalk03.csv")
        watchers = pd.read_csv("/mnt/hdd/maittewa/deviants_watchersRndmWalk03_2.csv")
        friends = pd.read_csv("/mnt/hdd/maittewa/deviants_friendsRndmWalk03_2.csv")
        profile_new = "/mnt/hdd/maittewa/deviants_profileSnwball_3.csv"
        already_existing_profile = pd.read_csv(profile_new)
        deviant_count = 0
        visited_deviants = set()
        unique_deviants = pd.DataFrame()
        visited_deviants.update(set(already_existing_profile["user"].tolist()))
        profile_data = pd.DataFrame()
        unique_deviants_to_gather = self.append_unique_usernames(profile,watchers,friends,unique_deviants)
        unique_dev_list = unique_deviants_to_gather["unique_dev"].tolist()
        for deviant in unique_dev_list:
            #print(f"length of unique deviants ")
            if deviant not in visited_deviants:
                deviant_count += 1
                visited_deviants.add(deviant)
                print(f"Gathering profile data for {deviant}, count {deviant_count}")
                profile_data = self.parse_user_profile(deviant)
                time.sleep(3)
                if profile_data is not None and not profile_data.empty:
                        print(f"fetched user profile data for {deviant}")
                        profile_data.to_csv(profile_new, mode="a", header=not os.path.exists(profile_new), index=False)
                        print(f"Saved {deviant} profile and the profile data has the shape {profile_data.shape}")
                else:
                    print(f"Empty user profile data for {deviant}")
                profile_data = pd.DataFrame()
                self.refresh_token()
            else:
                    print(f"Skipping already visited {deviant}")
        print("Random walk completed.")

In [ ]:
import pandas as pd
import numpy as np
#gallery_unique_devs = pd.read_csv("/mnt/hdd/maittewa/gallery_deviant_name_nd_deviationIds.csv")
#already_saved_deviants = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRndmWlkr03_1.csv")

In [ ]:
uniqueDevs_gall = gallery_unique_devs["Author_name"].unique()

In [ ]:
uniqueDev_prof = already_saved_deviants["user"].unique()

In [ ]:
# Concatenate DataFrames
combined_df = pd.concat([gallery_unique_devs["Author_name"], already_saved_deviants["user"]])

# Find unique rows based on both columns
unique_rows = pd.DataFrame(combined_df.unique(), columns=['UniqueValues'])

print(unique_rows.info())

In [ ]:
unique_rows.to_csv("/mnt/hdd/maittewa/uniqueDevs_gall_prof_RndmWalkr03.csv")

In [ ]:
new_gall = pd.read_csv("/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_6.csv")
new_gall.nunique()

In [ ]:
import pandas as pd
new_meta = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_watchersRndmWalk03_2.csv")
new_meta.nunique()

In [ ]:
new_meta.head()

In [ ]:
# Example usage:
client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"
TOKEN_URL = "https://www.deviantart.com/oauth2/token"
REDIRECT_URI = "https://www.deviantart.com/oauth2/authorize"

In [ ]:
# Initialize token refresh timer
random_walk = DeviantArtProfileData(client_id, client_secret, TOKEN_URL, REDIRECT_URI)
random_walk.get_token()
random_walk.refresh_token()
random_walk.fetch_deviant_profile()

In [ ]:
import sqlite3

# Connect to the database
conn = sqlite3.connect("/mnt/hdd/maittewa/deviantArt_data1.1.db")  # Replace with your database file name
cursor = conn.cursor()

# Example query: Get the first 10 rows from the deviants_profile table
cursor.execute("SELECT * FROM deviants_profile LIMIT 10")
results = cursor.fetchall()

# Print the results
for row in results:
    print(row)

# Example query: Get the number of rows in a table
cursor.execute("SELECT COUNT(*) FROM deviants_watchersRndmWalk03_2")
count = cursor.fetchone()[0]
print(f"Number of rows in deviants_friendsRndmWalk03_2: {count}")

# Close the connection
conn.close()

In [ ]:
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect("/mnt/hdd/maittewa/deviantArt_data1.1.db")  # Replace with your database file name
cursor = conn.cursor()

# Get a list of all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [table[0] for table in cursor.fetchall()]

# Get unique values summary for each table
for table in tables:
    print(f"\n--- Table: {table} ---")

    # Get column names
    cursor.execute(f"PRAGMA table_info({table})")
    columns = [column[1] for column in cursor.fetchall()]

    # Get unique values summary for each column
    for column in columns:
        cursor.execute(f"SELECT DISTINCT {column} FROM {table}")
        unique_values = cursor.fetchall()
        print(f"Column: {column}, Unique Values: {len(unique_values)}")

        # Print the unique values if you want to see them (optional)
        # if len(unique_values) <= 10:  # Only print if there are 10 or fewer unique values
        #     print(f"   {unique_values}")

# Get summary of all tables
print("\n--- Table Summaries ---")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    row_count = cursor.fetchone()[0]
    print(f"Table: {table}, Number of Rows: {row_count}")

# Close the connection
conn.close()

In [ ]:
friends = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_profileRndmWalk03.csv")

In [ ]:
friends.nunique()

Google Colab code Testing

In [ ]:
import deviantart
import requests, threading
import random
import sqlite3
from bs4 import BeautifulSoup
import json
import time
import requests.auth
import datetime
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient
import threading
import os, re
import pandas as pd
from requests.exceptions import HTTPError
import gc, pickle



class GatherMetaData:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()

    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret,
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")


    # Function to get metadata
    def get_metadata(self, devIds):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params_meta = {"deviationids[]": devIds}
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:

                api_url_devMeta = f"https://www.deviantart.com/api/v1/oauth2/deviation/metadata"
                response_devMeta = requests.get(api_url_devMeta, headers=headers, params=params_meta)
                return response_devMeta

            except requests.exceptions.RequestException as e:
                print(f"Error getting info: {e}")
                return None


    def parse_metadata(self, devMeta):
        # Get the metadata
        deviations_metadata = pd.DataFrame()

        try:
            data = devMeta.json()  # Extract JSON data from the response
            for i in data['metadata']:
                a = {"Devtn_Id": i['deviationid'],
                     "Devtn_Title": i["title"],
                     "Devtn_Descp": i["description"],
                     "Author_Name": i["author"]["username"],
                     "Allows_Comments": i["allows_comments"],
                     "Is_Favourited": i["is_favourited"],
                     "Is_Mature": i["is_mature"],
                     "Can_post_comments": i["can_post_comment"],
                     "Tags_Info": i["tags"]
                }
                dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
                dict_pd['tag_name'] = dict_pd['Tags_Info'].apply(lambda tags_list: [tag['tag_name'] for tag in tags_list])
                dict_pd['Sponsered'] = dict_pd['Tags_Info'].apply(lambda tags_list: [tag['sponsored'] for tag in tags_list])
                dict_pd['Sponser'] = dict_pd['Tags_Info'].apply(lambda tags_list: [tag['sponsor'] for tag in tags_list])

                deviations_metadata = pd.concat([deviations_metadata, dict_pd])
        except (requests.exceptions.RequestException, KeyError, ValueError) as e:
            print(f"Error parsing metadata: {e}")
            return pd.DataFrame()  # Return an empty DataFrame in case of error

        return deviations_metadata

    def fetch_deviations_metaData(self):
        """Executes the algorithm to fetch and save metadata for each deviation ID."""
        visited_meta_deviants = set()
        gallery_data_path = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviants_gallData_4_5_6/uniqueDev_gall_SnwBall03_6.2.csv.gz"
        metadata_path = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_metaDataSnwBall/uniqueDev_metaData_SnwBall_02.csv.gz"
        chunk_size = 10000
        columns_to_append = ['Author_name', 'Deviation_id']
        visited_deviants_file = "visited_deviants_forMetaData.pkl"  # Define the pickle file path

        # Load existing metadata deviants (if file exists)
        if os.path.exists(metadata_path):
            metadata_df = pd.read_csv(metadata_path, usecols=['Author_Name'], header=0, on_bad_lines='skip')
            visited_meta_deviants.update(metadata_df['Author_Name'].unique())
            print(f"Loaded existing metadata deviants: {len(visited_meta_deviants)}")

        # Load gallery data
        appended_gall_df = pd.DataFrame(columns=columns_to_append)
        if os.path.exists(gallery_data_path):
            for chunk in pd.read_csv(gallery_data_path, chunksize=chunk_size, header=0, usecols=columns_to_append, on_bad_lines='skip'):
                appended_gall_df = pd.concat([appended_gall_df, chunk[columns_to_append]], ignore_index=True)
            print(f"Loaded gallery data: {appended_gall_df.nunique()}")

        # Load visited deviants from pickle file (if exists)
        try:
            if os.path.exists(visited_deviants_file):
                with open(visited_deviants_file, "rb") as f:
                    visited_meta_deviants.update(pickle.load(f))  # Update, not replace
        except EOFError:
            print("Warning: 'visited_deviantsforMeta.pkl' is empty or corrupted. Ignoring it.")


        # Main execution
        deviant_count = 0
        metaId_count = 0

        unique_deviants_to_check = set(appended_gall_df['Author_name'].tolist())
        try:
            for deviant in unique_deviants_to_check:
                if deviant not in visited_meta_deviants:
                    visited_meta_deviants.add(deviant)
                    deviant_count += 1
                    print(f"Gathering metadata for unique deviant: {deviant}, count: {deviant_count}")

                    deviant_gall_data = appended_gall_df.loc[appended_gall_df['Author_name'] == deviant, 'Deviation_id']
                    devIds = deviant_gall_data.unique().tolist()
                    print(f"Total unique devIds are {len(devIds)} for {deviant}")

                    meta = pd.DataFrame()  # Reset meta for each deviant
                    for devId in devIds:
                        metaId_count += 1
                        print(f"Gathering metadata for deviant: {deviant}, devId: {devId}, count: {metaId_count}")

                        # Check if devId has already been processed (before fetching metadata)
                        if os.path.exists(metadata_path):
                            existing_deviation_ids = pd.read_csv(metadata_path, usecols=['Devtn_Id'], on_bad_lines='skip', low_memory=False)['Devtn_Id'].tolist()
                            if devId in existing_deviation_ids:
                                print(f"Skipping devId: {devId} for deviant: {deviant} (already exists)")
                                continue  # Skip to the next devId

                        metadata = self.get_metadata(devId)

                        if metadata is not None:
                            parsed_df = self.parse_metadata(metadata)

                            # Save metadata immediately if not empty
                            if not parsed_df.empty:
                                parsed_df.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)
                                print(f"Saved metadata for deviant: {deviant}, devId: {devId}")
                                #Saving the pickel file
                                visited_meta_deviants.add(deviant)
                                with open(visited_deviants_file, "wb") as f:
                                    pickle.dump(visited_meta_deviants, f)
                                print(f"Saved visited deviants to pickle file: {visited_deviants_file}")
                            else:
                                print(f"No metadata for devId: {devId} of {deviant} is available")
                        else:
                            print(f"No metadata for devId: {devId} of {deviant} is available")

                        time.sleep(random.uniform(1, 2))

                else:
                    print(f"Skipping already processed deviant: {deviant}")

                self.refresh_token()



        except requests.exceptions.RequestException as e:
            print(f"Exception occurred: {e}")


        print("Metadata fetching completed.")

# Provide API credentials
client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"
TOKEN_URL = "https://www.deviantart.com/oauth2/token"
REDIRECT_URI = "https://www.deviantart.com/oauth2/authorize"

#Call the class and the function
# Initialize token refresh timer
metaDat = GatherMetaData(client_id, client_secret, TOKEN_URL, REDIRECT_URI)
metaDat.get_token()
metaDat.refresh_token()
metaDat.fetch_deviations_metaData()

In [ ]:
class GatherMetaData:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")

    
    def get_response_rate(self, response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'


    # Function to get metadata
    def get_metadata(self, devIds):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params_meta = {"deviationids[]": devIds}
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
               
                api_url_devMeta = f"https://www.deviantart.com/api/v1/oauth2/deviation/metadata"
                response_devMeta = requests.get(api_url_devMeta, headers=headers, params=params_meta)
                return response_devMeta

            except requests.exceptions.RequestException as e:
                print(f"Error getting info: {e}")
                return None
    

    def parse_metadata(self, devMeta):
        # Get the metadata
        deviations_metadata = pd.DataFrame()

        try:
            data = devMeta.json()  # Extract JSON data from the response
            for i in data['metadata']:
                a = {"DevtnId": i['deviationid'],
                     "DevtnTitle": i["title"],
                     "DevtnDescp": i["description"],
                     "AuthorId": i["author"]["userid"],
                     "AuthorName": i["author"]["username"],
                     "AuthorIcon": i["author"]["usericon"],
                     "AuthorType": i["author"]["type"],
                     "License": i["license"],
                     "AllowsComments": i["allows_comments"],
                     "IsFavourited": i["is_favourited"],
                     "IsMature": i["is_mature"],
                     "CanPostComments": i["can_post_comment"],
                     "TagsInfo": i["tags"]
                }
                dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
                dict_pd['tag_name'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['tag_name'] for tag in tags_list])
                dict_pd['Sponsered'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['sponsored'] for tag in tags_list])
                dict_pd['Sponser'] = dict_pd['TagsInfo'].apply(lambda tags_list: [tag['sponsor'] for tag in tags_list])
                
                deviations_metadata = pd.concat([deviations_metadata, dict_pd])
        except (requests.exceptions.RequestException, KeyError, ValueError) as e:
            print(f"Error parsing metadata: {e}")
            return pd.DataFrame()  # Return an empty DataFrame in case of error
            
        return deviations_metadata

    def fetch_deviations_metaData(self):
        """Executes the random walk algorithm."""
        visited_meta_deviants = set()
        already_saved_deviants = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")
        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        metadata_path = "/mnt/hdd/maittewa/uniqueDev_dvtnMetaRndmWalk03_1.csv"
        chunk_size = 10000
        columns_to_append = ['Author_name', 'Deviation_id'] 
    
        # Load existing metadata deviants (if file exists)
        if os.path.exists(metadata_path):
            metadata_df = pd.read_csv(metadata_path, usecols=['AuthorName'], header=0, on_bad_lines='skip')
            visited_meta_deviants.update(metadata_df['AuthorName'].unique())
            print(f"Loaded existing metadata deviants: {len(visited_meta_deviants)}")
    
        # Load gallery data (if file exists)
        appended_gall_df = pd.DataFrame(columns=columns_to_append)
        if os.path.exists(gallery_data_path):
            for chunk in pd.read_csv(gallery_data_path, chunksize=chunk_size, header=0, usecols=columns_to_append, on_bad_lines='skip'):
                appended_gall_df = pd.concat([appended_gall_df, chunk[columns_to_append]], ignore_index=True)
            print(f"Loaded gallery data: {len(appended_gall_df)} rows")
    
        # Main execution
        deviant_count = 0
        batch_size = 10  
        try:
            unique_deviants_in_gallery = appended_gall_df['Author_name'].unique()
            for deviant in unique_deviants_in_gallery:
                if deviant not in visited_meta_deviants:
                    visited_meta_deviants.add(deviant)
                    deviant_count += 1
                    print(f"Gathering metadata for unique deviant: {deviant}, count: {deviant_count}")
    
                    deviant_gall_data = appended_gall_df.loc[appended_gall_df['Author_name'] == deviant, 'Deviation_id']
                    devIds = deviant_gall_data.tolist()
                    metadata_chunks = [devIds[i:i + chunk_size] for i in range(0, len(devIds), chunk_size)]
                    
                    meta = pd.DataFrame()  # Reset meta for each deviant
                    for chunk in metadata_chunks:
                        metadata = self.get_metadata(chunk)
                        if metadata:
                            parsed_df = self.parse_metadata(metadata)
                            meta = pd.concat([meta, parsed_df], ignore_index=True) 
    
                    # Save metadata only if it was fetched
                    if not meta.empty:
                        meta.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)
                        print(f"Saved metadata for {deviant}")
    
                        # Refresh token occasionally (e.g., every 10 deviants)
                        if deviant_count % 10 == 0:
                            self.refresh_token()
                else:
                    print(f"Skipping already processed deviant: {deviant}")
    
        except requests.exceptions.RequestException as e:
            print(f"Exception occurred: {e}")
    
        print("Metadata fetching completed.")
    

In [ ]:
meta = pd.read_csv("/mnt/hdd/maittewa/uniqueDev_dvtnMetaRndmWalk03_1.csv")

In [ ]:
#When loading deviants - total unique deviants on gallery_data path = 4834
meta.info()

In [ ]:
#When loading deviants - total unique deviants on gallery_data path = 4834
meta.nunique()

In [ ]:
import pandas as pd
gall1 = pd.read_csv("/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv", on_bad_lines='skip')
#gall2 = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")
#df_meta = pd.read_csv("/mnt/hdd/maittewa/deviants_deviationMetaDataRandomWalkerSince2003.csv")
#df_meta_unique = df_meta.drop_duplicates(subset=['Deviation_id'], keep='first', inplace = False)
#df_meta_unique.to_csv("/mnt/hdd/maittewa/uniqueDeviant_deviationMetaDataRandomWalkerSince2003.csv")

In [ ]:
#df = pd.read_csv("/mnt/hdd/maittewa/uniqueDev_gall_RandomWalkSince2003.csv")
#df = pd.read_csv("/mnt/hdd/maittewa/deviants_deviationMetaDataRandomWalkerSince2003.csv")
gall1.nunique()

In [ ]:
##One that was running but printing too many times the same thing
def get_random_deviants_from_daily_deviations(self, num_deviants, date):
        """Fetches a list of random deviants from a specific tag and page."""
        # Check if we have a valid token
        url = f"https://www.deviantart.com/api/v1/oauth2/browse/dailydeviations?access_token={self.access_token}"  
        params = {
        "client_id": client_id,
        "client_secret": client_secret,
        "date": date.strftime("%Y-%m-%d"),  # Format date as YYYY-MM-DD
        # ... (other parameters if needed, e.g., limit, offset) ...
        }
        try:
            response = requests.get(url, params=params)
            response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
            data = response.json() 
            #results = self.da.browse_dailydeviations()  
            #results = self.da.browse(tag=tag, offset=(page - 1) * 24)  # 24 results per page by default
            deviants = [deviation["author"]["username"] for deviation in data["results"]]
            random_deviants = random.sample(deviants, min(num_deviants, len(deviants))) 
            return random_deviants
            # Get as many as possible
        except Exception as e:
            print(f"Error fetching deviants: {e}")
            self.refresh_token()
            self.get_random_deviants_from_daily_deviations(num_deviants,date)
    def fetch_gallery_deviationids_metaData(self):
        """
        Executes the random walk algorithm to gather gallery info and metadata.
        """
        # 1. Load Existing Data and Visited Deviants:
        visited_deviants = set()
        already_saved_deviants = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")
        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        metadata_path = "/mnt/hdd/maittewa/uniqueDeviant_deviationMetaDataRandomWalkerSince2003.csv"

        # Find the bad lines in gallery_data_path
        bad_lines_gallery = find_bad_lines(gallery_data_path)

        # Load the list of existing authors from gallery_data_path into visited_deviants
        if os.path.exists(gallery_data_path):
            print(f"Loading visited deviants from {gallery_data_path}...")
            for chunk in pd.read_csv(gallery_data_path, chunksize=100000, header=0, usecols=['Author_name'], on_bad_lines='skip'):
                # Filter out the rows with bad lines numbers
                if bad_lines_gallery: # only filter if we have bad lines
                    chunk = chunk[~chunk.index.isin(bad_lines_gallery)]
                newly_visited = set(chunk['Author_name'].tolist())
                visited_deviants.update(newly_visited)
                print(f"  Added {len(newly_visited)} deviants from chunk to visited_deviants. Total: {len(visited_deviants)}")
            print(f"Finished loading visited deviants from {gallery_data_path}. Total: {len(visited_deviants)}")

        deviant_count = 0  # Keep track of processed deviants
        save_interval = 5  # Save data after processing this many deviants
        batch_size = 10
        chunk_size = 50  # Adjust chunk size as needed for metadata retrieval
        chunk_size_csv = 10000 # Adjust chunk size as needed for csv retrieval

        # Main execution
        try:
            for i in range(0, len(already_saved_deviants), batch_size):
                batch_usernames = already_saved_deviants['user'][i:i + batch_size].tolist()
                for deviant in batch_usernames:
                    if deviant not in visited_deviants:
                        visited_deviants.add(deviant)
                        deviant_count += 1
                        deviant_gall_data = pd.DataFrame(columns=['Author_name', 'Deviation_id'])
                        meta = pd.DataFrame()
                        print(f"Gathering gallery info and metadata for {deviant}, count {deviant_count}")
                        gallery = self.get_gallery(deviant)
                        if gallery is not None:
                            gallery_df = pd.DataFrame(gallery)
                            #Ensure the column name is consistent
                            if 'deviation_id' in gallery_df.columns:
                                gallery_df.rename(columns={'deviation_id': 'Deviation_id'}, inplace=True)
                            
                            deviant_gall_data = pd.concat([deviant_gall_data,gallery_df], ignore_index=True)
                            print(f'Gathered and parsed gallery data for {deviant}')

                        else:
                            print(f"No gallery data of {deviant} available")
                        time.sleep(random.uniform(1,2))
                        if deviant_gall_data is not None and 'Deviation_id' in deviant_gall_data.columns:
                            devIds = deviant_gall_data["Deviation_id"].tolist()
                            metadata_chunks = [devIds[i:i + chunk_size] for i in range(0, len(devIds), chunk_size)]
                            for chunk in metadata_chunks:
                                metadata = self.get_metadata(chunk)
                                if metadata:
                                    parsed_df = self.parse_metadata(metadata)
                                    meta = pd.concat([meta, parsed_df], ignore_index=True)  # Append parsed data
                                print(f'Gathered and parsed deviation metadata for {deviant}')
                                # 3. Save Data in Batches:
                                if deviant_count % save_interval == 0:
                                # Append to CSV with `mode="a"`
                                    deviant_gall_data.to_csv(gallery_data_path, mode="a", header=not os.path.exists(gallery_data_path), index=False)
                                    meta.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)

                                    print(f"      Saved data for {deviant_count} deviants.")

                            # Delete only if not saving
                            if deviant_count % save_interval != 0:
                                del deviant_gall_data
                                del meta
                                gc.collect()

            else:
                print(f"The path {already_saved_deviants_path} does not exists")
        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")
        except pd.errors.ParserError as e:
            print(f"A parser error occurred: {e}")

        print("Random walk completed.")

In [ ]:
###For reducing the too many print statements
def fetch_gallery_deviationids_metaData(self):
        """
        Executes the random walk algorithm to gather gallery info and metadata.
        """
        # 1. Load Existing Data and Visited Deviants:
        visited_deviants = set()
        already_saved_deviants = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")
        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        metadata_path = "/mnt/hdd/maittewa/uniqueDeviant_deviationMetaDataRandomWalkerSince2003.csv"

        # Find the bad lines in gallery_data_path
        bad_lines_gallery = self.find_bad_lines(gallery_data_path)

        # Load the list of existing authors from gallery_data_path into visited_deviants
        if os.path.exists(gallery_data_path):
            print(f"Loading visited deviants from {gallery_data_path}...")
            for chunk in pd.read_csv(gallery_data_path, chunksize=10000, header=0, usecols=['Author_name'], on_bad_lines='skip'):
                # Filter out the rows with bad lines numbers
                if bad_lines_gallery: # only filter if we have bad lines
                    chunk = chunk[~chunk.index.isin(bad_lines_gallery)]
                newly_visited = set(chunk['Author_name'].tolist())
                visited_deviants.update(newly_visited)
                print(f"  Added {len(newly_visited)} deviants from chunk to visited_deviants. Total: {len(visited_deviants)}")
            print(f"Finished loading visited deviants from {gallery_data_path}. Total: {len(visited_deviants)}")

        deviant_count = 0  # Keep track of processed deviants
        save_interval = 5  # Save data after processing this many deviants
        batch_size = 10
        chunk_size = 50  # Adjust chunk size as needed for metadata retrieval
        chunk_size_csv = 10000 # Adjust chunk size as needed for csv retrieval
        deviant_gall_data = pd.DataFrame()
        meta = pd.DataFrame()

        # Main execution
        try:
            for i in range(0, len(already_saved_deviants), batch_size):
                batch_usernames = already_saved_deviants['user'][i:i + batch_size].tolist()

                for deviant in batch_usernames:
                    if deviant not in visited_deviants:
                        visited_deviants.add(deviant)
                        deviant_count += 1
                        print(f"Gathering gallery info and metadata for {deviant}")

                        gallery = self.get_gallery(deviant)
                        if gallery is not None:
                            gallery_df = pd.DataFrame(gallery)
                            deviant_gall_data = pd.concat([deviant_gall_data, gallery_df], ignore_index=True)
                            print(f'Gathered and parsed gallery data for {deviant}')
                            # ---------------------------------------------------------------
                            # All metadata for a user should be gathered here
                            devIds = deviant_gall_data["Deviation_id"].tolist()
                            metadata_chunks = [devIds[j:j + chunk_size] for j in range(0, len(devIds), chunk_size)]
                            for chunk in metadata_chunks:
                                metadata = self.get_metadata(chunk)
                                if metadata:
                                    parsed_df = self.parse_metadata(metadata)
                                    meta = pd.concat([meta, parsed_df], ignore_index=True)

                            print(f'Gathered and parsed deviation metadata for {deviant}')

                            # ---------------------------------------------------------------
                        else:
                            print(f"No gallery data of {deviant} available")

                        if deviant_count % save_interval == 0:
                            # Save gallery data in chunks
                            for start in range(0, len(deviant_gall_data), chunk_size_csv):
                                end = start + chunk_size_csv
                                chunk_to_save = deviant_gall_data.iloc[start:end]
                                chunk_to_save.to_csv(gallery_data_path, mode='a', header=not os.path.exists(gallery_data_path), index=False)
                            deviant_gall_data = pd.DataFrame()  # Reset for the next batch

                            # Save metadata in chunks
                            for start in range(0, len(meta), chunk_size_csv):
                                end = start + chunk_size_csv
                                chunk_to_save = meta.iloc[start:end]
                                chunk_to_save.to_csv(metadata_path, mode='a', header=not os.path.exists(metadata_path), index=False)
                            meta = pd.DataFrame()  # Reset for the next batch

                            print(f"Saved data for {deviant_count} deviants.")

                        self.refresh_token()
                        time.sleep(random.uniform(1, 2))
                    else:
                        print(f"Skipping already visited {deviant}")

        except RequestException as e:
            print(f"Request Exception: {e}")

        except Exception as e:
            print(f"An unexpected error occurred: {e}")

        finally:
            # Save any remaining data
            if not deviant_gall_data.empty:
                for start in range(0, len(deviant_gall_data), chunk_size_csv):
                    end = start + chunk_size_csv
                    chunk_to_save = deviant_gall_data.iloc[start:end]
                    chunk_to_save.to_csv(gallery_data_path, mode='a', header=not os.path.exists(gallery_data_path), index=False)
            if not meta.empty:
                for start in range(0, len(meta), chunk_size_csv):
                    end = start + chunk_size_csv
                    chunk_to_save = meta.iloc[start:end]
                    chunk_to_save.to_csv(metadata_path, mode='a', header=not os.path.exists(metadata_path), index=False)

        print("Random walk completed.")

In [ ]:
def fetch_gallery_deviationids_metaData(self):
        """Executes the random walk algorithm."""
        # 1. Load Existing Data and Visited Deviants:
        visited_deviants = set()
        already_saved_deviants = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")

        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        #metadata_path = "/mnt/hdd/maittewa/uniqueDeviant_deviationMetaDataRandomWalkerSince2003.csv"

        if os.path.exists(gallery_data_path):
            existing_gallery_df = pd.read_csv(gallery_data_path)
            visited_deviants.update(set(existing_gallery_df['Author_name'].tolist()))
        
        deviant_count = 0  # Keep track of processed deviants
        # Establish a single database connection outside the loop
        #count_for_reauth = 0
        deviant_batch = []
        user_name = [] 
        deviant_gall_data = pd.DataFrame()
        meta = pd.DataFrame()
        save_interval = 10  # Save data after processing this many deviants
        deviant_count = 0
        batch_size = 10

        # Main execution
        # Split the deviation IDs into chunks to avoid exceeding API limits
        try:
            for i in range(0, len(already_saved_deviants), batch_size):
                batch_usernames = already_saved_deviants['user'][i:i + batch_size].tolist()

                for deviant in batch_usernames:
                    if deviant not in visited_deviants:
                        visited_deviants.add(deviant)
                        deviant_count += 1
                        print(f"Gathering gallery info and metadata for {deviant}")
                        gallery = self.get_gallery(deviant)
                        if gallery is not None:
                            gallery_df = pd.DataFrame(gallery)
                            deviant_gall_data = pd.concat([deviant_gall_data,gallery_df])
                            print(f'Gathered and parsed gallery data for {deviant}')
                            
                        else:
                            print(f"No gallery data of {deviant} available")
                        time.sleep(random.uniform(1,2)) 
                        if deviant_gall_data is not None:
                            devIds = deviant_gall_data["Deviation_id"].tolist()     
                            metadata_chunks = [devIds[i:i + chunk_size] for i in range(0, len(devIds), chunk_size)]
                            for chunk in metadata_chunks:
                                metadata = self.get_metadata(chunk)
                            if metadata:
                                parsed_df = self.parse_metadata(metadata)
                                meta = pd.concat([meta, parsed_df], ignore_index=True)  # Append parsed data
                            print(f'Gathered and parsed deviation metadata for {deviant}')
                            # 3. Save Data in Batches:
                            if deviant_count % save_interval == 0:
                            # Append to CSV with `mode="a"`
                                deviant_gall_data.to_csv(gallery_data_path, mode="a", header=not os.path.exists(gallery_data_path), index=False)
                                #meta.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)

                                # Reset DataFrames for the next batch
                                deviant_gall_data = pd.DataFrame()
                                #syyyyyyyyyyyyyxmeta = pd.DataFrame()
                                print(f"Saved data for {deviant_count} deviants.")

                            self.refresh_token()
                           # Clear the list for the next batch
                           # time.sleep(5) 
                            
                    else:
                        print(f"Skipping already visited {deviant}")

        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")
            
        print("Random walk completed.")

In [ ]:

    def fetch_gallery_deviationids_metaData(self):
        """
        Executes the random walk algorithm to gather gallery info and metadata.
        """
        # 1. Load Existing Data and Visited Deviants:
        visited_deviants = set()
        already_saved_deviants_path = "/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv"
        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        metadata_path = "/mnt/hdd/maittewa/uniqueDev_dvtnMetaRndmWalk03_1.csv"

        if os.path.exists(gallery_data_path):
            existing_gallery_df = pd.read_csv(gallery_data_path)
            visited_deviants.update(set(existing_gallery_df['Author_name'].tolist()))

        deviant_count = 0  # Keep track of processed deviants
        save_interval = 5  # Save data after processing this many deviants
        batch_size = 10
        chunk_size = 50  # Adjust chunk size as needed for metadata retrieval
        chunk_size_csv = 1000 # Adjust chunk size as needed for csv retrieval
        # Find the bad lines
        bad_lines = find_bad_lines(gallery_data_path)
        bad_lines_numbers = {line_number for line_number, _ in bad_lines}
        #Main execution
       #Main execution
        try:
            # Check if the file exists
            if os.path.exists(already_saved_deviants_path):
                # Iterate over the file in chunks
                # In this line we read the already_saved_deviants_path file
                for chunk in pd.read_csv(already_saved_deviants_path, chunksize=chunk_size_csv, header=None, on_bad_lines="skip"):

                    # Iterate over each row of the current chunk to get the usernames
                    for index, row in chunk.iterrows():

                        deviant = row['user']

                        # check that the deviant is a string and not a number
                        if type(deviant) is not str:
                            continue

                        if deviant not in visited_deviants:
                            deviant_gall_data = pd.DataFrame(columns=['Author_name', 'Deviation_id'])
                            meta = pd.DataFrame()
                            visited_deviants.add(deviant)
                            deviant_count += 1
                            print(f"Gathering gallery info and metadata for {deviant}, count {deviant_count}")
                            gallery = self.get_gallery(deviant)
                            if gallery is not None:
                                gallery_df = pd.DataFrame(gallery)
                                #Ensure the column name is consistent
                                if 'deviation_id' in gallery_df.columns:
                                    gallery_df.rename(columns={'deviation_id': 'Deviation_id'}, inplace=True)

                                deviant_gall_data = pd.concat([deviant_gall_data,gallery_df], ignore_index=True)
                                print(f'Gathered and parsed gallery data for {deviant}')

                            else:
                                print(f"No gallery data of {deviant} available")
                            time.sleep(random.uniform(1,2))
                            if deviant_gall_data is not None and 'Deviation_id' in deviant_gall_data.columns:
                                devIds = deviant_gall_data["Deviation_id"].tolist()
                                metadata_chunks = [devIds[i:i + chunk_size] for i in range(0, len(devIds), chunk_size)]
                                for chunk in metadata_chunks:
                                    metadata = self.get_metadata(chunk)
                                    if metadata:
                                        parsed_df = self.parse_metadata(metadata)
                                        meta = pd.concat([meta, parsed_df], ignore_index=True)  # Append parsed data
                                print(f'Gathered and parsed deviation metadata for {deviant}')
                                # 3. Save Data in Batches:
                                if deviant_count % save_interval == 0:
                                # Append to CSV with `mode="a"`
                                    deviant_gall_data.to_csv(gallery_data_path, mode="a", header=not os.path.exists(gallery_data_path), index=False)
                                    meta.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)

                                    print(f"Saved data for {deviant_count} deviants.")
                            
                            # Delete only if not saving
                            if deviant_count % save_interval != 0:
                                del deviant_gall_data
                                del meta
                                gc.collect()
                        else:
                            print(f"Skipping already visited {deviant}")

            else:
              print(f"The path {already_saved_deviants_path} does not exists")
        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")

        print("Random walk completed.")



In [ ]:
#Actual random Walk for gathering deviant data
def run_random_walk(self, num_devs, num_dates):
        """Executes the random walk algorithm."""
        #current_tag = start_tag
        visited_deviants = set()
        dev_df = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")
        already_saved_deviants = set(dev_df['user'].tolist()) 
        deviant_count = 0  # Keep track of processed deviants
        # Establish a single database connection outside the loop
        #count_for_reauth = 0
        deviant_batch = []
        user_name = [] #, user_watchers, user_friends, deviatn_metadata = [], [], [], [] #user_profile,
        # Main execution
        start_date = datetime.date(2003, 1, 1)  # 21 years ago
        end_date = datetime.date.today()
        date_visited = set()
        
        try:
            for _ in range(num_dates):
                random_date = start_date + datetime.timedelta(days=random.randint(0, (end_date - start_date).days))

                random_deviants = self.get_random_deviants_from_daily_deviations(num_devs,random_date)
                if random_deviants is not None:
                    for deviant in random_deviants:
                        if deviant not in visited_deviants and deviant not in already_saved_deviants:
                            visited_deviants.add(deviant)
                            print(f"Fetching data for unique deviant: {deviant} on date {random_date}") 
                            profile = self.get_profile(deviant)
                            deviant_profile = self.parse_user_profile(profile)
                            time.sleep(random.uniform(1,5))                   
                            watchers_friends_gallery = self.watchers_friends_gallery_data(deviant)
                            if len(watchers_friends_gallery) == 3:
                                deviant_watchers, deviant_friends, deviant_gallery_data = watchers_friends_gallery
                            else:
                                print(f"No watchers, friends or gallery data of {deviant} available")
                            time.sleep(random.uniform(1,5))    
                            devIds = deviant_gallery_data["Deviation_Id"].tolist()
                            #deviant_deviations = self.download_deviations(deviant)
                            #print(f"Downloaded {deviant}'s gallery")
                            #time.sleep(random.uniform(1,5))                   
                            deviations_metadata = self.parse_metadata(devIds)
                            print(f'Parsed deviation metadata for {deviant}')
                            time.sleep(random.uniform(1,5))                   
                            #print (deviations_metadata)
                            user_name.append(deviant)
                            deviant_df = pd.DataFrame(user_name, columns=["deviant_username"])
                            deviant_df.to_csv("/mnt/hdd/maittewa/deviantsRandomWalkerSince2003.csv", mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviantsRandomWalkerSince2003.csv"))
                            print(f"Saved {deviant} name")
                            deviant_profile = self.parse_user_profile(profile)
                            if deviant_profile is not None:
                                print(f"fetched user profile data for {deviant}")
                                deviant_profile.to_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv", mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv"), index=False)
                                print(f"Saved {deviant} profile")
                            else:
                                print(f"Empty user profile data for {deviant}")
            
                            if deviant_watchers is not None and not isinstance(deviant_watchers, list) and not deviant_watchers.empty:  # Save only if not None, not a list and not empty:
                                print(f"fetched deviant watchers for {deviant}")
                                deviant_watchers.to_csv("/mnt/hdd/maittewa/deviants_watchersRandomWalkerSince2003.csv", mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviants_watchersRandomWalkerSince2003.csv"), index=False)
                                print(f"Saved {deviant} watchers")
                            else:
                                print(f"Empty deviant watchers for {deviant}")
                                
                            if deviant_friends is not None and not isinstance(deviant_friends, list) and not deviant_friends.empty:  # Save only if not None, not a list and not empty:
                                print(f"fetched deviant friends for {deviant}")
                                deviant_friends.to_csv("/mnt/hdd/maittewa/deviants_friendsRandomWalkerSince2003.csv", mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviants_friendsRandomWalkerSince2003.csv"), index=False)
                                print(f"Saved {deviant} friends")
                            else:
                                print(f"Empty deviant friends for {deviant}")
            
                            if deviations_metadata is not None:
                                print(f"fetched deviant deviations metadata for {deviant}")
                                deviations_metadata.to_csv("/mnt/hdd/maittewa/deviatn_metadataRandomWalkerSince2003.csv", mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviatn_metadataRandomWalkerSince2003.csv"), index=False)
                                print(f"Saved {deviant} deviations metadata")
                            else:
                                print(f"Empty deviations metadata for {deviant}")
                           
                           # Clear the list for the next batch
                            user_name = []
                            time.sleep(5) 
                            
                        else:
                            print(f"Skipping already visited {deviant}")
                else:
                    print(f"No deviant for the date {random_date}")

        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")

        # Dump any remaining data in the batch
        if deviant_batch:
            #self.store_data(deviant, user_profile, deviant_watchers, deviant_friends, deviations_metadata)
            print(f"Dumped remaining data for {len(deviant_batch)} deviations to database.")
 
                    
        print("Random walk completed.")

In [ ]:
df_check = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")

In [ ]:
df_check.user.value_counts()

In [ ]:
def get_token():
        client = BackendApplicationClient(client_id=client_id)
        oauth = OAuth2Session(client=client)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        access_token = token['access_token']
        return access_token
    
def get_response_rate(response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'

# Function to get friends
def get_friends(username, page):
    token = get_token()
    api_url = f"https://www.deviantart.com/api/v1/oauth2/user/friends/{username}?access_token={token}"
    response = requests.get(api_url, params={'offset': page, 'limit': 50})
    return response

# Function to get watchers
def get_watchers(username, page):
    token = get_token()
    api_url = f"https://www.deviantart.com/api/v1/oauth2/user/watchers/{username}?access_token={token}"
    response = requests.get(api_url, params={'offset': page, 'limit': 50})
    return response

# Function to parse friends
def parse_friends(friends):
    users = pd.DataFrame()
    # print(friends.keys())
    # next_offset=friends['next_offset']
    has_more = friends.get('has_more')
    for i in friends['results']:
        a = {'username': i['user']['username'],
                 'user_icon': i['user']['usericon'],
                 'type': i['user']['type'],
                 'is_watching': i['is_watching'],
                 'last_visit': i['lastvisit'],
                 'friends': i['watch']['friend'],
                 'deviations': i['watch']['deviations'],
                 'journals': i['watch']['journals'],
                 'forum_threads': i['watch']['forum_threads'],
                 'critiques': i['watch']['critiques'],
                 'scraps': i['watch']['scraps'],
                 'activity': i['watch']['activity'],
                 'collections': i['watch']['collections']}
        dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
        users = pd.concat([users, dict_pd])
    return has_more, users

def parse_watchers(watchers):
    users = pd.DataFrame()
    # print(friends.keys())
    # next_offset=friends['next_offset']
    has_more = watchers.get('has_more')
    for i in watchers['results']:
        a = {'username': i['user']['username'],
                 'user_icon': i['user']['usericon'],
                 'type': i['user']['type'],
                 'is_watching': i['is_watching'],
                 'last_visit': i['lastvisit'],
                 'activity': i['watch']['activity'],
                 'collections': i['watch']['collections'],
                 'critiques': i['watch']['critiques'],
                 'deviations': i['watch']['deviations'],
                 'forum_threads': i['watch']['forum_threads'],
                 'friend': i['watch']['friend'],
                 'journals': i['watch']['journals'],
                 'scraps': i['watch']['scraps']}
        dict_pd = pd.DataFrame.from_dict(a, orient='index').T
        users = pd.concat([users, dict_pd], ignore_index=True)
    return has_more, users

In [ ]:
"""Gathers watchers and watching using API."""

watchers_pd = pd.DataFrame()
# Get watching (friends)
friends_pd = pd.DataFrame()
has_more = True
username = "damter"
try:
    # Get the initial batch of watchers
    for i in range(0, 5):
        if has_more == True:
            resp = get_watchers(username, i)
            watchers = get_response_rate(resp)
            if watchers is not None:
                has_more, parsed_watchers = parse_watchers(watchers)
                if len(parsed_watchers) > 0:
                    watchers_pd = pd.concat([watchers_pd, parsed_watchers])
                
    # Get the initial batch of watching users
    for i in range(0, 5):
        if has_more == True:
            resp = get_friends(username, i)
            friends = get_response_rate(resp)
            if friends is not None:
                has_more, parsed_frnds = parse_friends(friends)
                if len(parsed_frnds) > 0:
                    friends_pd = pd.concat([friends_pd, parsed_frnds])

except requests.exceptions.RequestException as e:
    print(f"Error scraping DeviantArt watchers and watching: {e}")

In [ ]:
friends_pd.info()

In [ ]:
def get_token():
        client = BackendApplicationClient(client_id=client_id)
        oauth = OAuth2Session(client=client)
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret)
        # Extract the access token
        access_token = token['access_token']
        return access_token

In [ ]:
client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"
TOKEN_URL = "https://www.deviantart.com/oauth2/token"
REDIRECT_URI = "https://www.deviantart.com/oauth2/authorize"

In [ ]:
import pandas as pd
import requests
import os

da = deviantart.Api("42096", "97080792c6d30a4178965e41f1ca15de")

# Specify the username
username = "RavenRemo"

# Get the user's gallery
gallery = da.get_gallery_folder(username)
output_dir = f"/mnt/hdd/maittewa/downloaded_gallery_{username}"

all_deviations = []
has_more = True
while has_more:
  for deviation in gallery['results']:
    deviation_data = {}
    if hasattr(deviation, 'title'):
        deviation_data['title'] = deviation.title

    # Get deviation details including tags
    deviation_details = da.get_deviation(deviation.deviationid)
    if hasattr(deviation_details, 'tags'):
      deviation_data['tags'] = deviation_details.tags

    all_deviations.append(deviation_data)

  has_more = gallery['has_more']
  if has_more:
    gallery = da.get_gallery_folder(username, offset=gallery['next_offset'])

    metadata = {
        "title": deviation.title,
        "deviationid": deviation.deviationid,
        "url": deviation.url,
        "category": deviation.category
        # Add other metadata fields as needed
    }
    
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, username + ".json")
    
    with open(filename, "w") as f:
        json.dump(metadata, f, indent=4)
    
    print(f"Saved metadata for deviation {deviation_id} to {filename}")
# Print the list of tags for each deviation
for deviation in all_deviations:
  print(f"Deviation: {deviation['title']}")
  if 'tags' in deviation:
    print(f"Tags: {', '.join(deviation['tags'])}")
  else:
    print("No tags found for this deviation.")
  print("-" * 20)

In [ ]:
 def deviations_metadata(self, deviant):
        # Get the gallery folder
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
                gallery = self.da.get_gallery_folder(deviant)
                deviations_metadata = pd.DataFrame()
                for deviation in gallery['results']:
                    metadata = {}
                    metadata = {
                        "deviation_id": deviation.deviationid,
                        "title": deviation.title,
                        "category": deviation.category}
                    if hasattr(deviation, 'content') and deviation.content and 'src' in deviation.content:
                        metadata['image_width'] = deviation.content['width']
                        metadata["image_height"] = deviation.content["height"]
                        #filename = f"{deviation_id}.json"
                        #filepath = os.path.join(output_dir, metadata_folder, username + ".json")
                        #os.makedirs(os.path.dirname(filepath), exist_ok=True)  
                        dict_pd = pd.DataFrame.from_dict(metadata, orient='index').transpose()
                        deviations_metadata = pd.concat([deviations_metadata, dict_pd])
                return deviations_metadata
            except Exception as e:
                print(f"Encountering the following {e} error")
                self.refresh_token()
                self.download_deviations(deviant)


    def download_deviations(self,deviant):  
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        self.da.token = self.access_token
        if self.da.token:
            try:
                gallery = self.da.get_gallery_folder(deviant)
                # Create an output directory for downloaded images
                output_dir = f"/mnt/hdd/maittewa/deviantartGallery/downloaded_gallery_{deviant}"
                os.makedirs(output_dir, exist_ok=True)
                #os.makedirs(gallery_folder, exist_ok=True)
                # Download images
                has_more = True
                while has_more:
                    for deviation in gallery['results']:
                        if hasattr(deviation, 'content') and deviation.content:
                            if 'src' in deviation.content:
                                image_url = deviation.content['src']
                                if image_url:
                                    filename = f"{deviation.title}-{deviation.deviationid}.jpg"
                                    filepath = os.path.join(output_dir, deviant, filename)
                                    os.makedirs(os.path.dirname(filepath), exist_ok=True)  
                                    # Download and save the image
                                    # Download and save the image
                                    try:
                                        response = requests.get(image_url, stream=True)  # Assign response
                                        response.raise_for_status() # Check for errors
                            
                                        with open(filepath, 'wb') as image_file:
                                            for chunk in response.iter_content(chunk_size=8192):
                                                image_file.write(chunk)
                                            #print(f"Downloaded: {filename}")
        
                                    except requests.exceptions.RequestException as e:
                                        print(f"Error downloading {image_url}: {e}")
        
                                else:
                                    image_url = None
                                    print(f"Image URL not found for deviation ID: {deviation_id}")
                
                    # Check if there are more deviations to fetch
                    has_more = gallery['has_more']
                    if has_more:
                        gallery = self.da.get_gallery_folder(
                            deviant, offset=gallery['next_offset'])

            except Exception as e:
                print(f"Encountering the following {e} error")
                self.refresh_token()
                self.download_deviations(deviant)

In [ ]:
    def download_deviations(self,deviant):  
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()

        if self.access_token:
            try:
                
                # Create an output directory for downloaded images
                output_dir = f"/mnt/hdd/maittewa/deviantartGallery/downloaded_gallery_{deviant}"
                os.makedirs(output_dir, exist_ok=True)
                #os.makedirs(gallery_folder, exist_ok=True)
                # Download images
                has_more = True
                while has_more:
                    for deviation in gallery['results']:
                        if hasattr(deviation, 'content') and deviation.content:
                            if 'src' in deviation.content:
                                image_url = deviation.content['src']
                                if image_url:
                                    filename = f"{deviation.title}-{deviation.deviationid}.jpg"
                                    filepath = os.path.join(output_dir, deviant, filename)
                                    os.makedirs(os.path.dirname(filepath), exist_ok=True)  
                                    # Download and save the image
                                    # Download and save the image
                                    try:
                                        response = requests.get(image_url, stream=True)  # Assign response
                                        response.raise_for_status() # Check for errors
                            
                                        with open(filepath, 'wb') as image_file:
                                            for chunk in response.iter_content(chunk_size=8192):
                                                image_file.write(chunk)
                                            #print(f"Downloaded: {filename}")
        
                                    except requests.exceptions.RequestException as e:
                                        print(f"Error downloading {image_url}: {e}")
        
                                else:
                                    image_url = None
                                    print(f"Image URL not found for deviation ID: {deviation_id}")
                
                    # Check if there are more deviations to fetch
                    has_more = gallery['has_more']
                    if has_more:
                        gallery = self.da.get_gallery_folder(
                            deviant, offset=gallery['next_offset'])

            except Exception as e:
                print(f"Encountering the following {e} error")
                self.refresh_token()
                self.download_deviations(deviant)



In [ ]:
import deviantart
import os

# Replace with your client ID and client secret
client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"

# Create an API object
da = deviantart.Api(client_id, client_secret)

# Specify the username of the deviant whose gallery folder you want to download
username = "Ramonn90"  # Example username

# Create an output directory for downloaded images
output_dir = f"/mnt/hdd/maittewa/deviantartGallery/"
gallery_folder = f"downloaded_gallery_{username}"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(gallery_folder, exist_ok=True)

# Get the gallery folder
gallery = da.get_gallery_folder(username)
deviations_metadata = pd.DataFrame()
# Download images
has_more = True
while has_more:
    for deviation in gallery['results']:
        if hasattr(deviation, 'content') and deviation.content:
            if 'src' in deviation.content:
                image_url = deviation.content['src']
                if image_url:
                    filename = f"{deviation.title}-{deviation.deviationid}.jpg"
                    filepath = os.path.join(output_dir, gallery_folder, username, filename)
                    os.makedirs(os.path.dirname(filepath), exist_ok=True)  
                    # Download and save the image
                    try:
                        response = requests.get(image_url, stream=True)  # Assign response
                        response.raise_for_status() # Check for errors
                    
                        with open(filepath, 'wb') as image_file:
                            for chunk in response.iter_content(chunk_size=8192):
                                image_file.write(chunk)
                            print(f"Downloaded: {filename}")

                    except requests.exceptions.RequestException as e:
                         print(f"Error downloading {download_url}: {e}")

                else:
                    image_url = None
                    print(f"Image URL not found for deviation ID: {deviation_id}")

    # Check if there are more deviations to fetch
    has_more = gallery['has_more']
    if has_more:
        gallery = da.get_gallery_folder(
            username, offset=gallery['next_offset'])

In [ ]:
#Error handling
#print(f"Entering HTTPError handler...")
                print(f"HTTP Error: {e} for user: {username}")
                status_code_match = re.search(r"(\d{3})", str(e))  # Extract status code from error message
                if status_code_match and status_code_match.group(1) == "401":
                    print("401 Unauthorized error.")
                else:
                    # Handle other HTTP errors
                    print(f"Handling other HTTP errors...")

###Profile api response 
if profile_response is None:
            print("Status code is None")
        elif profile_response.status_code != 200:
            print(f"API request failed with status code {s.status_code}: {s.text}")
        else:
            # API request was successful
            d = json.loads(profile_response.text)
                